# 02. Layer-Wise GPT-2 Encoding Pipeline

This notebook extracts GPT-2 XL layer-wise representations, builds delayed TR-level design matrices, predicts raw and SRM-reconstructed fMRI responses, and summarizes layer-wise encoding performance.

**GitHub-ready notebook:** outputs have been cleared and private paths have been replaced with placeholders.

# Step 1: Prepare Word-Level Transcript

This cell only prepares the story transcript and word timing table for the SNL encoding pipeline. The analysis logic is unchanged from the previous encoding notebook


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ROI_NAME = "all_networks"
ROI_TARGET = "all_networks_fixed_srm50"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"

PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
STEP_DIR = PROJECT_DIR / "step1_prepare_transcript"
CSV_DIR = STEP_DIR / "csv"
JSON_DIR = STEP_DIR / "json"

for d in [PROJECT_ROOT, PROJECT_DIR, STEP_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"ROI name: {ROI_NAME}")
print(f"ROI target: {ROI_TARGET}")
print(f"Encoding run name: {ENCODING_RUN_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step directory: {STEP_DIR}")


# =========================
# 2. Input transcript path
# =========================

TRANSCRIPT_PATH = BASE_DIR / "align.csv"

if not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Transcript file not found: {TRANSCRIPT_PATH}\n"
        "Please update TRANSCRIPT_PATH to the real 21styear transcript/alignment file."
    )

print(f"Transcript path: {TRANSCRIPT_PATH}")


# =========================
# 3. Load transcript
# =========================
# Expected format:
# col 0 = original word
# col 1 = cleaned word
# col 2 = onset
# col 3 = offset
# No header row

df = pd.read_csv(TRANSCRIPT_PATH, header=None)

print("\nLoaded transcript preview:")
print(df.head())


# =========================
# 4. Specify transcript columns directly
# =========================

orig_word_col = 0
clean_word_col = 1
onset_col = 2
offset_col = 3

print("\nUsing fixed transcript columns:")
print(f"  original_word_col = {orig_word_col}")
print(f"  clean_word_col    = {clean_word_col}")
print(f"  onset_col         = {onset_col}")
print(f"  offset_col        = {offset_col}")


# =========================
# 5. Build working dataframe
# =========================

work_df = df[[orig_word_col, clean_word_col, onset_col, offset_col]].copy()
work_df.columns = ["word_original", "word_clean", "onset", "offset"]

# Convert timing columns to numeric
work_df["onset"] = pd.to_numeric(work_df["onset"], errors="coerce")
work_df["offset"] = pd.to_numeric(work_df["offset"], errors="coerce")

# Drop rows missing critical information
work_df = work_df.dropna(subset=["word_original", "word_clean", "onset", "offset"]).copy()

# Standardize text columns
work_df["word_original"] = work_df["word_original"].astype(str).str.strip()
work_df["word_clean"] = work_df["word_clean"].astype(str).str.strip()

# Drop rows with empty strings
work_df = work_df[
    (work_df["word_original"] != "") &
    (work_df["word_clean"] != "")
].copy()


# =========================
# 6. Normalize model-facing word forms
# =========================

def normalize_word(w: str) -> str:
    w = str(w).strip().lower()

    # Remove punctuation only at word edges
    w = re.sub(r"^[^\w<]+|[^\w>]+$", "", w)

    # Normalize unknowns
    if w in {"<unk>", "unk", "[unk]"}:
        return "<unk>"

    return w


work_df["word_normalized"] = work_df["word_clean"].apply(normalize_word)

# Remove rows that become empty after normalization
work_df = work_df[work_df["word_normalized"] != ""].copy()

# Keep only valid timing rows
work_df = work_df[work_df["offset"] >= work_df["onset"]].copy()


# =========================
# 7. Sort and add timing-derived columns
# =========================

# Minimal stability improvement:
# sort by onset, then offset, then reset index
work_df = work_df.sort_values(["onset", "offset"]).reset_index(drop=True)

work_df["word_index"] = np.arange(len(work_df))
work_df["duration"] = work_df["offset"] - work_df["onset"]
work_df["word_midpoint"] = (work_df["onset"] + work_df["offset"]) / 2.0

# Reorder columns
work_df = work_df[
    [
        "word_index",
        "word_original",
        "word_clean",
        "word_normalized",
        "onset",
        "offset",
        "duration",
        "word_midpoint",
    ]
].copy()

print("\nCleaned transcript preview:")
print(work_df.head(10))


# =========================
# 8. Basic checks
# =========================

if len(work_df) == 0:
    raise ValueError("No valid transcript rows remain after cleaning.")

onset_non_decreasing = bool(np.all(np.diff(work_df["onset"].values) >= 0))
duration_non_negative = bool(np.all(work_df["duration"].values >= 0))
unique_word_index = bool(work_df["word_index"].is_unique)
finite_onsets = bool(np.isfinite(work_df["onset"].values).all())
finite_offsets = bool(np.isfinite(work_df["offset"].values).all())

if not onset_non_decreasing:
    raise ValueError("Onset times are not nondecreasing after sorting.")
if not duration_non_negative:
    raise ValueError("Negative durations detected after cleaning.")
if not unique_word_index:
    raise ValueError("word_index is not unique.")
if not finite_onsets or not finite_offsets:
    raise ValueError("Non-finite onset/offset values detected.")

# Informative warnings only
if (work_df["duration"] == 0).any():
    print("[WARNING] Some words have zero duration.")

if work_df["word_normalized"].eq("<unk>").any():
    print(f"[INFO] Number of <unk> tokens: {(work_df['word_normalized'] == '<unk>').sum()}")


# =========================
# 9. Save outputs
# =========================

clean_csv_path = CSV_DIR / "21styear_word_timing_clean.csv"
work_df.to_csv(clean_csv_path, index=False, encoding="utf-8-sig")

# Optional duplicate-friendly export for quick inspection
preview_csv_path = CSV_DIR / "21styear_word_timing_clean_preview_first50.csv"
work_df.head(50).to_csv(preview_csv_path, index=False, encoding="utf-8-sig")


# =========================
# 10. Step-end validation summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "encoding_step1_prepare_transcript_cleaned",
    "input_transcript_path": str(TRANSCRIPT_PATH),

    "n_words_total": int(len(work_df)),
    "n_unique_normalized_words": int(work_df["word_normalized"].nunique()),

    "first_onset": float(work_df["onset"].min()),
    "last_offset": float(work_df["offset"].max()),
    "mean_duration": float(work_df["duration"].mean()),
    "median_duration": float(work_df["duration"].median()),
    "min_duration": float(work_df["duration"].min()),
    "max_duration": float(work_df["duration"].max()),

    "columns_used": {
        "word_original": orig_word_col,
        "word_clean": clean_word_col,
        "onset": onset_col,
        "offset": offset_col,
    },

    "checks": {
        "onset_non_decreasing": onset_non_decreasing,
        "duration_non_negative": duration_non_negative,
        "unique_word_index": unique_word_index,
        "finite_onsets": finite_onsets,
        "finite_offsets": finite_offsets,
    },

    "outputs": {
        "clean_csv": str(clean_csv_path),
        "preview_csv": str(preview_csv_path),
    },
}

summary_path = JSON_DIR / "step1_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved cleaned transcript to: {clean_csv_path}")
print(f"Saved preview CSV to: {preview_csv_path}")
print(f"Saved summary to: {summary_path}")


# =========================
# 11. Final step-end checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)
print(f"n_words_total: {len(work_df)}")
print(f"onset_non_decreasing: {onset_non_decreasing}")
print(f"duration_non_negative: {duration_non_negative}")
print(f"unique_word_index: {unique_word_index}")
print(f"finite_onsets: {finite_onsets}")
print(f"finite_offsets: {finite_offsets}")
print(f"first_onset: {work_df['onset'].min():.6f}")
print(f"last_offset: {work_df['offset'].max():.6f}")
print(f"mean_duration: {work_df['duration'].mean():.6f}")
print(f"median_duration: {work_df['duration'].median():.6f}")

print("\nStep 1 completed successfully.")


# Step 2: Extract GPT-2 XL Contextual Embeddings

This cell keeps the original embedding-extraction logic unchanged. Only the project path is moved to the SNL encoding folder so it reads the cleaned transcript generated by Step 1.

Step2_extract_gpt2xl_embeddings.

b2b-linguistic-coupling：
https://github.com/hassonlab/b2b-linguistic-coupling


add_causal_lm_embs(...)

In [ ]:
from pathlib import Path
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ROI_NAME = "all_networks"
ROI_TARGET = "all_networks_fixed_srm50"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"

PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
STEP1_DIR = PROJECT_DIR / "step1_prepare_transcript"
STEP2_DIR = PROJECT_DIR / "step2_extract_gpt2xl_alllayer_contextual_embeddings"

CSV_DIR = STEP2_DIR / "csv"
NPY_DIR = STEP2_DIR / "npy"
LAYER_DIR = NPY_DIR / "layers"
JSON_DIR = STEP2_DIR / "json"

for d in [PROJECT_ROOT, PROJECT_DIR, STEP2_DIR, CSV_DIR, NPY_DIR, LAYER_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"ROI name: {ROI_NAME}")
print(f"ROI target: {ROI_TARGET}")
print(f"Encoding run name: {ENCODING_RUN_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 1 directory: {STEP1_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")


# =========================
# 2. Input word table
# =========================

WORD_TABLE_PATH = STEP1_DIR / "csv" / "21styear_word_timing_clean.csv"
STEP1_SUMMARY_PATH = STEP1_DIR / "json" / "step1_summary.json"

if not WORD_TABLE_PATH.exists():
    raise FileNotFoundError(f"Word timing file not found: {WORD_TABLE_PATH}")
if not STEP1_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Step1 summary not found: {STEP1_SUMMARY_PATH}")

print(f"Word table path: {WORD_TABLE_PATH}")
print(f"Step1 summary path: {STEP1_SUMMARY_PATH}")

with open(STEP1_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step1_summary = json.load(f)


# =========================
# 3. Model setup
# =========================

MODEL_NAME = "gpt2-xl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model name: {MODEL_NAME}")
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_CONTEXT = int(tokenizer.model_max_length)
if MAX_CONTEXT is None or MAX_CONTEXT <= 0 or MAX_CONTEXT > 100000:
    MAX_CONTEXT = 1024

OVERLAP_TOKENS = 256
STEP_SIZE = MAX_CONTEXT - OVERLAP_TOKENS

print(f"MAX_CONTEXT = {MAX_CONTEXT}")
print(f"OVERLAP_TOKENS = {OVERLAP_TOKENS}")
print(f"STEP_SIZE = {STEP_SIZE}")

if STEP_SIZE <= 0:
    raise ValueError("STEP_SIZE must be > 0. Please re-check MAX_CONTEXT and OVERLAP_TOKENS.")


# =========================
# 4. Load cleaned transcript
# =========================

word_df = pd.read_csv(WORD_TABLE_PATH)

required_cols = [
    "word_index",
    "word_original",
    "word_clean",
    "word_normalized",
    "onset",
    "offset",
    "duration",
    "word_midpoint",
]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns in transcript file: {missing}")

word_df = word_df.copy().reset_index(drop=True)

print("\nLoaded cleaned transcript preview:")
print(word_df.head())
print(f"Number of words: {len(word_df)}")

if len(word_df) == 0:
    raise ValueError("The cleaned transcript is empty.")

if not word_df["word_index"].is_unique:
    raise ValueError("word_index is not unique in the transcript.")
if not np.all(np.diff(word_df["onset"].values) >= 0):
    raise ValueError("Transcript onset times are not nondecreasing.")
if not np.all(word_df["offset"].values >= word_df["onset"].values):
    raise ValueError("Found offset < onset in transcript.")


# =========================
# 5. Tokenize each word and build full token sequence
# =========================

all_token_ids = []
all_token_strs = []
token_to_word = []
word_token_start = []
word_token_end = []
word_n_subtokens = []

for word_idx, word in tqdm(
    enumerate(word_df["word_normalized"].astype(str).tolist()),
    total=len(word_df),
    desc="Tokenizing words"
):
    encoded = tokenizer(
        word,
        add_special_tokens=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    token_ids = encoded["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(token_ids)

    if len(token_ids) == 0:
        # fallback
        token_ids = [tokenizer.eos_token_id]
        token_strs = [tokenizer.eos_token]

    start = len(all_token_ids)
    end = start + len(token_ids)

    word_token_start.append(start)
    word_token_end.append(end)
    word_n_subtokens.append(len(token_ids))

    all_token_ids.extend(token_ids)
    all_token_strs.extend(token_strs)
    token_to_word.extend([word_idx] * len(token_ids))

all_token_ids = np.asarray(all_token_ids, dtype=np.int64)
token_to_word = np.asarray(token_to_word, dtype=np.int32)
word_token_start = np.asarray(word_token_start, dtype=np.int32)
word_token_end = np.asarray(word_token_end, dtype=np.int32)
word_n_subtokens = np.asarray(word_n_subtokens, dtype=np.int32)

print("\nTokenization summary:")
print(f"Total words: {len(word_df)}")
print(f"Total subtokens: {len(all_token_ids)}")
print(f"Mean subtokens per word: {word_n_subtokens.mean():.3f}")
print(f"Median subtokens per word: {np.median(word_n_subtokens):.3f}")
print(f"Max subtokens per word: {word_n_subtokens.max()}")

if len(all_token_ids) == 0:
    raise ValueError("Tokenization produced zero tokens.")


# =========================
# 6. Build overlapping windows across full token sequence
# =========================

def build_window_starts(n_tokens: int, max_context: int, step_size: int):
    if n_tokens <= max_context:
        return [0]

    starts = list(range(0, n_tokens - max_context + 1, step_size))
    last_start = n_tokens - max_context
    if starts[-1] != last_start:
        starts.append(last_start)
    return starts


window_starts = build_window_starts(
    n_tokens=len(all_token_ids),
    max_context=MAX_CONTEXT,
    step_size=STEP_SIZE
)

print(f"\nNumber of windows: {len(window_starts)}")

# Assign each token to the earliest window that contains it
# so each token gets the largest available left context
token_assigned_window = np.full(len(all_token_ids), fill_value=-1, dtype=np.int32)

for win_idx, start in enumerate(window_starts):
    end = min(start + MAX_CONTEXT, len(all_token_ids))
    token_slice = np.arange(start, end, dtype=np.int32)
    unassigned = token_assigned_window[token_slice] == -1
    token_assigned_window[token_slice[unassigned]] = win_idx

if np.any(token_assigned_window < 0):
    raise RuntimeError("Some tokens were not assigned to any window.")


# =========================
# 7. Prepare output memmap
# =========================

n_words = len(word_df)
hidden_size = int(model.config.hidden_size)
n_layers_total = int(model.config.n_layer) + 1   # embedding layer + transformer outputs

ALL_LAYER_MEMMAP_PATH = NPY_DIR / "gpt2xl_word_embeddings_all_layers_memmap.npy"
all_layer_memmap = np.lib.format.open_memmap(
    ALL_LAYER_MEMMAP_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(n_layers_total, n_words, hidden_size),
)

print("\nEmbedding storage prepared:")
print(f"n_layers_total = {n_layers_total}")
print(f"hidden_size = {hidden_size}")
print(f"Memmap path = {ALL_LAYER_MEMMAP_PATH}")


# =========================
# 8. Extract contextual all-layer activations
# =========================

start_time = time.time()

with torch.no_grad():
    for win_idx, start in enumerate(tqdm(window_starts, desc="Running contextual windows")):
        end = min(start + MAX_CONTEXT, len(all_token_ids))
        window_token_ids = all_token_ids[start:end]

        input_ids = torch.tensor(window_token_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
        attention_mask = torch.ones_like(input_ids, device=DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )

        hidden_states = outputs.hidden_states

        if len(hidden_states) != n_layers_total:
            raise ValueError(
                f"Expected {n_layers_total} hidden-state tensors, got {len(hidden_states)}."
            )

        local_positions = []
        global_token_indices = []

        for local_pos in range(end - start):
            global_idx = start + local_pos
            if token_assigned_window[global_idx] == win_idx:
                local_positions.append(local_pos)
                global_token_indices.append(global_idx)

        if len(local_positions) == 0:
            continue

        local_positions = np.asarray(local_positions, dtype=np.int64)
        global_token_indices = np.asarray(global_token_indices, dtype=np.int64)
        local_word_ids = token_to_word[global_token_indices]

        # mean-pool later, so now accumulate token vectors into each word
        for layer_idx in range(n_layers_total):
            layer_arr = hidden_states[layer_idx][0, local_positions, :].detach().cpu().numpy().astype(np.float32)
            np.add.at(all_layer_memmap[layer_idx], local_word_ids, layer_arr)

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

elapsed_minutes = (time.time() - start_time) / 60.0
print(f"\nContextual all-layer extraction completed.")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")


# =========================
# 9. Mean-pool subtokens within each word
# =========================

subtoken_denominator = np.maximum(word_n_subtokens.astype(np.float32), 1.0)

for layer_idx in tqdm(range(n_layers_total), desc="Averaging subtokens into word embeddings"):
    all_layer_memmap[layer_idx] /= subtoken_denominator[:, None]

all_layer_memmap.flush()


# =========================
# 10. Save each layer separately
# =========================

layer_rows = []

for layer_idx in range(n_layers_total):
    layer_arr = np.array(all_layer_memmap[layer_idx], dtype=np.float32)
    layer_path = LAYER_DIR / f"layer_{layer_idx:02d}.npy"
    np.save(layer_path, layer_arr)

    layer_rows.append({
        "layer_index": layer_idx,
        "path": str(layer_path),
        "shape": str(layer_arr.shape),
    })

layer_df = pd.DataFrame(layer_rows)
layer_df.to_csv(CSV_DIR / "layer_file_manifest.csv", index=False, encoding="utf-8-sig")

print(f"\nSaved per-layer embeddings to: {LAYER_DIR}")


# =========================
# 11. Save word-token alignment table
# =========================

token_df = pd.DataFrame({
    "global_token_index": np.arange(len(all_token_ids)),
    "token_id": all_token_ids,
    "token_str": all_token_strs,
    "word_index": token_to_word,
    "assigned_window": token_assigned_window,
})

token_df_path = CSV_DIR / "token_word_alignment.csv"
token_df.to_csv(token_df_path, index=False, encoding="utf-8-sig")

word_token_df = word_df.copy()
word_token_df["token_start"] = word_token_start
word_token_df["token_end"] = word_token_end
word_token_df["n_subtokens"] = word_n_subtokens

word_token_df_path = CSV_DIR / "word_timing_with_token_spans.csv"
word_token_df.to_csv(word_token_df_path, index=False, encoding="utf-8-sig")

print(f"Saved token-word alignment table to: {token_df_path}")
print(f"Saved word-token span table to: {word_token_df_path}")


# =========================
# 12. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "alllayer_contextual_windows_cleaned",
    "input_word_table_path": str(WORD_TABLE_PATH),
    "source_step1_summary_path": str(STEP1_SUMMARY_PATH),

    "model_name": MODEL_NAME,
    "device": DEVICE,
    "max_context": int(MAX_CONTEXT),
    "overlap_tokens": int(OVERLAP_TOKENS),
    "step_size": int(STEP_SIZE),

    "n_words": int(n_words),
    "n_subtokens_total": int(len(all_token_ids)),
    "mean_subtokens_per_word": float(word_n_subtokens.mean()),
    "median_subtokens_per_word": float(np.median(word_n_subtokens)),
    "max_subtokens_per_word": int(word_n_subtokens.max()),

    "n_windows": int(len(window_starts)),
    "hidden_size": int(hidden_size),
    "n_layers_total": int(n_layers_total),

    "all_layer_memmap_shape": [int(n_layers_total), int(n_words), int(hidden_size)],
    "elapsed_minutes": float(elapsed_minutes),

    "outputs": {
        "all_layer_memmap": str(ALL_LAYER_MEMMAP_PATH),
        "layer_manifest_csv": str(CSV_DIR / "layer_file_manifest.csv"),
        "token_alignment_csv": str(token_df_path),
        "word_token_span_csv": str(word_token_df_path),
        "layer_dir": str(LAYER_DIR),
    },
}

summary_path = JSON_DIR / "step2_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary to: {summary_path}")


# =========================
# 13. Final step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

all_layer_shape_ok = tuple(all_layer_memmap.shape) == (n_layers_total, n_words, hidden_size)
all_word_assignment_ok = len(word_token_start) == n_words and len(word_token_end) == n_words
all_token_assignment_ok = np.all(token_assigned_window >= 0)
all_finite_sample_ok = np.isfinite(np.array(all_layer_memmap[0, :10, :10])).all()

print(f"all_layer_shape_ok: {all_layer_shape_ok}")
print(f"all_word_assignment_ok: {all_word_assignment_ok}")
print(f"all_token_assignment_ok: {all_token_assignment_ok}")
print(f"all_finite_sample_ok: {all_finite_sample_ok}")
print(f"n_words: {n_words}")
print(f"n_subtokens_total: {len(all_token_ids)}")
print(f"n_windows: {len(window_starts)}")
print(f"n_layers_total: {n_layers_total}")
print(f"hidden_size: {hidden_size}")
print(f"elapsed_minutes: {elapsed_minutes:.2f}")

if not all_layer_shape_ok:
    raise ValueError("Final memmap shape is incorrect.")
if not all_word_assignment_ok:
    raise ValueError("Word-token span arrays do not match n_words.")
if not all_token_assignment_ok:
    raise ValueError("Some tokens were not assigned to any window.")
if not all_finite_sample_ok:
    raise ValueError("Non-finite values detected in sampled embedding output.")

print("\nStep 2 completed successfully.")


This step extracts a contextual GPT-2 XL representation for each word by feeding the current word together with its preceding context into the causal language model and taking the hidden state of the word’s last subtoken as its embedding.

“Brain-to-brain coupling in a shared linguistic embedding space during natural conversations” coding from many papers and this is just one of them~

# Step 3: Reduce GPT-2 XL All-Layer Embeddings To Layer-Wise Feature Sets

This step replaces the old final-layer PCA sweep. The goal is now layer-wise encoding: each GPT-2 XL layer is treated as a separate model feature space.

For each layer, PCA is used only as a fixed-dimensional compression tool. PCA is fit on training words only, then applied to all words, training words, and test words. This keeps the dimensionality identical across layers so later encoding results compare GPT-2 layer depth rather than different numbers of PCA components.



In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ROI_NAME = "all_networks"
ROI_TARGET = "all_networks_fixed_srm50"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"

PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
STEP2_DIR = PROJECT_DIR / "step2_extract_gpt2xl_alllayer_contextual_embeddings"
STEP3_DIR = PROJECT_DIR / "step3_reduce_embeddings_layerwise"
SUMMARY_CSV_DIR = STEP3_DIR / "csv"
SUMMARY_JSON_DIR = STEP3_DIR / "json"

for d in [PROJECT_ROOT, PROJECT_DIR, STEP3_DIR, SUMMARY_CSV_DIR, SUMMARY_JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"ROI name: {ROI_NAME}")
print(f"ROI target: {ROI_TARGET}")
print(f"Encoding run name: {ENCODING_RUN_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")
print(f"Step 3 directory: {STEP3_DIR}")


# =========================
# 2. Input settings
# =========================

WORD_TABLE_PATH = STEP2_DIR / "csv" / "word_timing_with_token_spans.csv"
STEP2_SUMMARY_PATH = STEP2_DIR / "json" / "step2_summary.json"
ALL_LAYER_MEMMAP_PATH = STEP2_DIR / "npy" / "gpt2xl_word_embeddings_all_layers_memmap.npy"

# Fixed-dimensional compression for every GPT-2 XL layer.
# PCA is only a compression step here; the scientific comparison is across layers.
LAYER_REDUCTION_COMPONENTS = 50

# Keep the same split logic as the previous encoding pipeline.
TR = 1.5
STORY_START_TR = 14
STORY_N_TR = 2226
TRAIN_TRS = STORY_N_TR // 2   # 1113
TRAIN_TEST_SPLIT_TIME = STORY_START_TR * TR + TRAIN_TRS * TR   # 1690.5 s

for p in [WORD_TABLE_PATH, STEP2_SUMMARY_PATH, ALL_LAYER_MEMMAP_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Word table path: {WORD_TABLE_PATH}")
print(f"All-layer memmap path: {ALL_LAYER_MEMMAP_PATH}")
print(f"Layer reduction components: {LAYER_REDUCTION_COMPONENTS}")
print(f"Train/test split time (seconds): {TRAIN_TEST_SPLIT_TIME:.5f}")


# =========================
# 3. Load data
# =========================

word_df = pd.read_csv(WORD_TABLE_PATH)

with open(STEP2_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step2_summary = json.load(f)

required_cols = ["word_index", "onset", "offset", "word_midpoint"]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns in word table: {missing}")

n_words = int(step2_summary["n_words"])
hidden_size = int(step2_summary["hidden_size"])
n_layers_total = int(step2_summary["n_layers_total"])

all_layer_memmap = np.lib.format.open_memmap(
    ALL_LAYER_MEMMAP_PATH,
    mode="r",
)

expected_shape = (n_layers_total, n_words, hidden_size)
if tuple(all_layer_memmap.shape) != expected_shape:
    raise ValueError(
        f"All-layer memmap shape mismatch: got {tuple(all_layer_memmap.shape)}, "
        f"expected {expected_shape}."
    )

if len(word_df) != n_words:
    raise ValueError(f"Word table rows ({len(word_df)}) do not match n_words ({n_words}).")

if LAYER_REDUCTION_COMPONENTS > min(n_words, hidden_size):
    raise ValueError(
        f"LAYER_REDUCTION_COMPONENTS={LAYER_REDUCTION_COMPONENTS} exceeds allowable rank "
        f"for all-layer embeddings shape {(n_words, hidden_size)}."
    )

print("\nLoaded Step 2 all-layer embeddings:")
print(f"word_df shape: {word_df.shape}")
print(f"all_layer_memmap shape: {tuple(all_layer_memmap.shape)}")
print(f"n_layers_total: {n_layers_total}")
print(f"hidden_size: {hidden_size}")


# =========================
# 4. Define train/test word split
# =========================

word_df = word_df.copy()
word_df["split"] = np.where(word_df["word_midpoint"] < TRAIN_TEST_SPLIT_TIME, "train", "test")

train_mask = word_df["split"].eq("train").to_numpy()
test_mask = word_df["split"].eq("test").to_numpy()

n_train_words = int(train_mask.sum())
n_test_words = int(test_mask.sum())

if n_train_words == 0 or n_test_words == 0:
    raise ValueError("Train/test split produced zero words in one partition.")

if LAYER_REDUCTION_COMPONENTS > min(n_train_words, hidden_size):
    raise ValueError(
        f"LAYER_REDUCTION_COMPONENTS={LAYER_REDUCTION_COMPONENTS} exceeds allowable PCA rank "
        f"for train embeddings shape {(n_train_words, hidden_size)}."
    )

word_split_csv = SUMMARY_CSV_DIR / "word_timing_with_split.csv"
word_df.to_csv(word_split_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nWord split summary:")
print(word_df["split"].value_counts().to_string())
print(f"n_train_words: {n_train_words}")
print(f"n_test_words : {n_test_words}")


# =========================
# 5. Fit one PCA compression per GPT-2 XL layer
# =========================

start_time = time.time()
layer_summary_rows = []

for layer_index in range(n_layers_total):
    layer_label = f"layer_{layer_index:02d}"
    layer_type = "embedding" if layer_index == 0 else "transformer"

    LAYER_DIR = STEP3_DIR / layer_label
    CSV_DIR = LAYER_DIR / "csv"
    NPY_DIR = LAYER_DIR / "npy"
    JSON_DIR = LAYER_DIR / "json"

    for d in [LAYER_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    layer_start = time.time()

    # Load one layer at a time so the cell does not hold all layers in RAM.
    embeddings = np.asarray(all_layer_memmap[layer_index], dtype=np.float32)

    if embeddings.shape != (n_words, hidden_size):
        raise ValueError(
            f"{layer_label}: embedding shape mismatch: {embeddings.shape} vs {(n_words, hidden_size)}."
        )

    if not np.isfinite(embeddings[:10, :10]).all():
        raise ValueError(f"{layer_label}: non-finite values detected in sampled embeddings.")

    train_embeddings = embeddings[train_mask]
    test_embeddings = embeddings[test_mask]

    pca = PCA(
        n_components=LAYER_REDUCTION_COMPONENTS,
        svd_solver="full",
        random_state=0,
    )
    pca.fit(train_embeddings)

    train_reduced = pca.transform(train_embeddings).astype(np.float32)
    test_reduced = pca.transform(test_embeddings).astype(np.float32)
    all_reduced = pca.transform(embeddings).astype(np.float32)

    np.save(NPY_DIR / f"gpt2xl_word_features_{layer_label}_all.npy", all_reduced)
    np.save(NPY_DIR / f"gpt2xl_word_features_{layer_label}_train.npy", train_reduced)
    np.save(NPY_DIR / f"gpt2xl_word_features_{layer_label}_test.npy", test_reduced)
    np.save(NPY_DIR / f"pca_components_{layer_label}.npy", pca.components_.astype(np.float32))
    np.save(NPY_DIR / f"pca_mean_{layer_label}.npy", pca.mean_.astype(np.float32))
    np.save(
        NPY_DIR / f"pca_explained_variance_ratio_{layer_label}.npy",
        pca.explained_variance_ratio_.astype(np.float32),
    )

    word_df.to_csv(CSV_DIR / "word_timing_with_split.csv", index=False, encoding="utf-8-sig", float_format="%.5f")

    explained_variance_df = pd.DataFrame({
        "component": np.arange(1, LAYER_REDUCTION_COMPONENTS + 1, dtype=int),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    })
    explained_variance_df.to_csv(
        CSV_DIR / "pca_explained_variance.csv",
        index=False,
        encoding="utf-8-sig",
        float_format="%.8f",
    )

    cumulative_ev = float(np.sum(pca.explained_variance_ratio_))
    layer_elapsed = (time.time() - layer_start) / 60.0

    layer_summary = {
        "dataset": "21styear",
        "roi_target": ROI_TARGET,
        "pipeline_type": "snl_gpt2xl_layerwise_fixed_dimensional_reduction",
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(LAYER_REDUCTION_COMPONENTS),
        "pca_fit_scope": "training_words_only",
        "n_words": int(n_words),
        "n_train_words": int(n_train_words),
        "n_test_words": int(n_test_words),
        "hidden_size": int(hidden_size),
        "input_embedding_shape": [int(x) for x in embeddings.shape],
        "train_reduced_shape": [int(x) for x in train_reduced.shape],
        "test_reduced_shape": [int(x) for x in test_reduced.shape],
        "all_reduced_shape": [int(x) for x in all_reduced.shape],
        "explained_variance_cumulative": cumulative_ev,
        "elapsed_minutes": float(layer_elapsed),
        "outputs": {
            "layer_dir": str(LAYER_DIR),
            "word_timing_with_split_csv": str(CSV_DIR / "word_timing_with_split.csv"),
            "features_all": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_all.npy"),
            "features_train": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_train.npy"),
            "features_test": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_test.npy"),
            "pca_components": str(NPY_DIR / f"pca_components_{layer_label}.npy"),
            "pca_mean": str(NPY_DIR / f"pca_mean_{layer_label}.npy"),
            "pca_explained_variance_ratio": str(NPY_DIR / f"pca_explained_variance_ratio_{layer_label}.npy"),
            "pca_explained_variance_csv": str(CSV_DIR / "pca_explained_variance.csv"),
        },
    }

    with open(JSON_DIR / "step3_layer_summary.json", "w", encoding="utf-8") as f:
        json.dump(layer_summary, f, indent=2)

    shape_ok = (
        all_reduced.shape == (n_words, LAYER_REDUCTION_COMPONENTS) and
        train_reduced.shape == (n_train_words, LAYER_REDUCTION_COMPONENTS) and
        test_reduced.shape == (n_test_words, LAYER_REDUCTION_COMPONENTS)
    )
    finite_ok = (
        np.isfinite(all_reduced).all() and
        np.isfinite(train_reduced).all() and
        np.isfinite(test_reduced).all()
    )

    if not shape_ok:
        raise ValueError(f"{layer_label}: reduced feature shapes are incorrect.")
    if not finite_ok:
        raise ValueError(f"{layer_label}: non-finite values detected in reduced features.")

    layer_summary_rows.append({
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(LAYER_REDUCTION_COMPONENTS),
        "n_words": int(n_words),
        "n_train_words": int(n_train_words),
        "n_test_words": int(n_test_words),
        "hidden_size": int(hidden_size),
        "explained_variance_cumulative": cumulative_ev,
        "elapsed_minutes": float(layer_elapsed),
        "layer_dir": str(LAYER_DIR),
        "features_all_path": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_all.npy"),
        "features_train_path": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_train.npy"),
        "features_test_path": str(NPY_DIR / f"gpt2xl_word_features_{layer_label}_test.npy"),
    })

    print(
        f"{layer_label}: reduced all={all_reduced.shape}, "
        f"train={train_reduced.shape}, test={test_reduced.shape}, "
        f"cumulative_EV={cumulative_ev:.6f}, elapsed={layer_elapsed:.2f} min"
    )


# =========================
# 6. Save overall Step 3 summary
# =========================

layer_summary_df = pd.DataFrame(layer_summary_rows)
layer_summary_csv = SUMMARY_CSV_DIR / "step3_layerwise_feature_summary.csv"
layer_summary_df.to_csv(
    layer_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.8f",
)

total_elapsed = (time.time() - start_time) / 60.0

overall_summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "snl_gpt2xl_layerwise_fixed_dimensional_reduction",
    "input_all_layer_memmap_path": str(ALL_LAYER_MEMMAP_PATH),
    "n_layers_total": int(n_layers_total),
    "layer_indices": [int(x) for x in layer_summary_df["layer_index"].tolist()],
    "layer_labels": layer_summary_df["layer_label"].tolist(),
    "reduction_method": "PCA",
    "reduction_components": int(LAYER_REDUCTION_COMPONENTS),
    "pca_fit_scope": "training_words_only_per_layer",
    "n_words": int(n_words),
    "n_train_words": int(n_train_words),
    "n_test_words": int(n_test_words),
    "hidden_size": int(hidden_size),
    "train_test_split_time_seconds": float(TRAIN_TEST_SPLIT_TIME),
    "total_elapsed_minutes": float(total_elapsed),
    "outputs": {
        "step3_dir": str(STEP3_DIR),
        "word_timing_with_split_csv": str(word_split_csv),
        "layerwise_feature_summary_csv": str(layer_summary_csv),
    },
}

overall_summary_path = SUMMARY_JSON_DIR / "step3_layerwise_summary.json"
with open(overall_summary_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# =========================
# 7. Step-end validation checks
# =========================

print("\n" + "=" * 70)
print("STEP 3 LAYER-WISE VALIDATION CHECKS")
print("=" * 70)

expected_layers_ok = len(layer_summary_df) == n_layers_total
unique_layer_labels_ok = layer_summary_df["layer_label"].is_unique
component_dim_ok = (layer_summary_df["reduction_components"] == LAYER_REDUCTION_COMPONENTS).all()
train_test_count_ok = (
    (layer_summary_df["n_train_words"] == n_train_words).all() and
    (layer_summary_df["n_test_words"] == n_test_words).all()
)
ev_finite_ok = np.isfinite(layer_summary_df["explained_variance_cumulative"].to_numpy()).all()

print(f"expected_layers_ok      : {expected_layers_ok} ({len(layer_summary_df)} layers)")
print(f"unique_layer_labels_ok  : {unique_layer_labels_ok}")
print(f"component_dim_ok        : {component_dim_ok}")
print(f"train_test_count_ok     : {train_test_count_ok}")
print(f"ev_finite_ok            : {ev_finite_ok}")
print(f"first layer label       : {layer_summary_df['layer_label'].iloc[0]}")
print(f"last layer label        : {layer_summary_df['layer_label'].iloc[-1]}")
print(f"mean cumulative EV      : {layer_summary_df['explained_variance_cumulative'].mean():.6f}")
print(f"Saved layer summary to  : {layer_summary_csv}")
print(f"Saved JSON summary to   : {overall_summary_path}")
print(f"Total elapsed           : {total_elapsed:.2f} min")

if not expected_layers_ok:
    raise ValueError("Unexpected number of layer-wise outputs.")
if not unique_layer_labels_ok:
    raise ValueError("Layer labels are not unique.")
if not component_dim_ok:
    raise ValueError("Layer-wise feature dimension does not match LAYER_REDUCTION_COMPONENTS.")
if not train_test_count_ok:
    raise ValueError("Train/test word counts are inconsistent across layers.")
if not ev_finite_ok:
    raise ValueError("Non-finite cumulative explained variance detected.")

print("\nStep 3 layer-wise fixed-dimensional reduction completed successfully.")


# Step 4: Downsample Layer-Wise Word Features To TR-Level Features — Lanczos Version


"Interpolation of the feature matrix
One challenge in fitting encoding models is that speech and BOLD data are sampled at very different frequencies. Approximately six words are spoken every two seconds, but only one brain image is recorded in that interval. To solve this problem, the stimulus matrix needs to be resampled to the same sampling frequency as the BOLD data. The procedure we provide for downsampling features to the fMRI acquisition rate can be thought of as comprising three steps. First, the discrete features for each word (or phoneme) are transformed into a continuous-time representation N(t) where t ∈ [0, T] and T indicates the length of the stimulus. This representation is zero at all timepoints except for the exact middle of each word (or phoneme), where it is equal to an infinitesimal-duration spike (Dirac δ-function) that is scaled by the feature value. Next, a low-pass antialiasing Lanczos filter is convolved with N(t) to get NLP(t). The cutoff frequency of this antialiasing filter is selected to match the Nyquist frequency of the fMRI data (half the acquisition rate, or 0.25 Hz). The cutoff frequency and filter roll-off (controlled by the number of lobes: more lobes yield a sharper roll-off, but at the cost of potentially increased noise) can be selected manually, although we recommend using the default values. Finally, NLP(t) is sampled at the fMRI acquisition times tr where r ∈ [1, 2….nTR] corresponds to the volume index in the fMRI acquisition. In practice, these three steps are accomplished simultaneously by way of a single matrix multiplication: the word- (or phoneme-) level stimulus matrix S (number of features by number of words/phonemes) is multiplied by a sparse “Lanczos” matrix L (number of words/phonemes by number of fMRI volumes). In essence, this assumes that the total brain response is the sum of responses to each word or phoneme. This approach has been widely used for language encoding models with natural stimuli10. An alternative to this approach would be to simply average the feature vectors for all the word or phonemes that appear within each 2-second period. However, that approach leads to discontinuities since words that fall infinitesimally before or after a boundary wind up in different time bins. The Lanczos method naturally accounts for this issue: if a word falls exactly at the boundary between two time bins, its features contribute equally to both (albeit scaled by 50%)."

# Step 4: Downsample Layer-Wise GPT-2 XL Features To TR-Level Features

This step follows the same Huth-style Lanczos interpolation logic as the previous encoding pipeline, but the input structure is now layer-wise rather than PCA-sweep based.

The layer-wise version loops over `layer_00` ... `layer_48`, reads each layer's fixed-dimensional word-level feature matrix from Step 3, and aligns it to fMRI TR midpoint times. Each layer gets its own TR-level train/test feature files so later delayed design matrices and encoding models can compare GPT-2 XL layers directly.

The train/test split remains defined in fMRI time: 1113 train TRs and 1113 test TRs. Word counts are therefore not forced to be exactly equal; they follow the word midpoint times relative to the same split time used throughout the encoding pipeline.


In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ROI_NAME = "all_networks"
ROI_TARGET = "all_networks_fixed_srm50"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"

PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
STEP3_DIR = PROJECT_DIR / "step3_reduce_embeddings_layerwise"
STEP4_DIR = PROJECT_DIR / "step4_downsample_to_tr_lanczos_layerwise"
SUMMARY_CSV_DIR = STEP4_DIR / "csv"
SUMMARY_NPY_DIR = STEP4_DIR / "npy"
SUMMARY_JSON_DIR = STEP4_DIR / "json"

for d in [PROJECT_ROOT, PROJECT_DIR, STEP4_DIR, SUMMARY_CSV_DIR, SUMMARY_NPY_DIR, SUMMARY_JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"ROI name: {ROI_NAME}")
print(f"ROI target: {ROI_TARGET}")
print(f"Encoding run name: {ENCODING_RUN_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 3 directory: {STEP3_DIR}")
print(f"Step 4 directory: {STEP4_DIR}")


# =========================
# 2. Input settings
# =========================

STEP3_LAYER_SUMMARY_PATH = STEP3_DIR / "csv" / "step3_layerwise_feature_summary.csv"
STEP3_OVERALL_JSON_PATH = STEP3_DIR / "json" / "step3_layerwise_summary.json"
MASTER_WORD_TABLE_PATH = STEP3_DIR / "csv" / "word_timing_with_split.csv"

TR = 1.5
STORY_START_TR = 14
STORY_N_TR = 2226
TRAIN_TRS = STORY_N_TR // 2  # 1113

# Huth-style timing logic:
# first define TR onset times, then explicitly shift by TR/2.0.
STORY_START_TIME = STORY_START_TR * TR
TR_ONSET_TIMES = STORY_START_TIME + np.arange(STORY_N_TR) * TR
TR_MIDPOINT_TIMES = TR_ONSET_TIMES + TR / 2.0

# Lanczos parameters
LANCZOS_WINDOW = 3
LANCZOS_CUTOFF_MULT = 1.0

for p in [STEP3_LAYER_SUMMARY_PATH, STEP3_OVERALL_JSON_PATH, MASTER_WORD_TABLE_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Step 3 layer summary path: {STEP3_LAYER_SUMMARY_PATH}")
print(f"Master word table path: {MASTER_WORD_TABLE_PATH}")
print(f"TR = {TR:.5f}")
print(f"STORY_START_TIME = {STORY_START_TIME:.5f}")
print(f"STORY_N_TR = {STORY_N_TR}")
print(f"LANCZOS_WINDOW = {LANCZOS_WINDOW}")
print(f"LANCZOS_CUTOFF_MULT = {LANCZOS_CUTOFF_MULT:.5f}")


# =========================
# 3. Huth-style Lanczos functions
# =========================

def lanczosfun(cutoff: float, t, window: int = 3) -> np.ndarray:
    """Compute the Lanczos kernel at time offset t."""
    t = np.asarray(t, dtype=np.float64) * cutoff
    val = np.zeros_like(t, dtype=np.float64)

    nonzero = t != 0
    val[nonzero] = (
        window
        * np.sin(np.pi * t[nonzero])
        * np.sin(np.pi * t[nonzero] / window)
        / (np.pi ** 2 * t[nonzero] ** 2)
    )
    val[~nonzero] = 1.0
    val[np.abs(t) > window] = 0.0
    return val


def make_lanczos_matrix(
    oldtime: np.ndarray,
    newtime: np.ndarray,
    window: int = 3,
    cutoff_mult: float = 1.0,
) -> np.ndarray:
    """Build the Lanczos interpolation matrix once for the shared word/TR timing grid."""
    cutoff = 1.0 / np.mean(np.diff(newtime)) * cutoff_mult

    print(f"\nBuilding Lanczos interpolation matrix with cutoff={cutoff:.6f} and {window} lobes.")

    lanczos_matrix = np.zeros((len(newtime), len(oldtime)), dtype=np.float64)
    for ndi in range(len(newtime)):
        lanczos_matrix[ndi, :] = lanczosfun(cutoff, newtime[ndi] - oldtime, window)

    return lanczos_matrix.astype(np.float32)


def apply_lanczos_matrix(data: np.ndarray, lanczos_matrix: np.ndarray) -> np.ndarray:
    """Apply a precomputed Lanczos matrix to columns of a word-level feature matrix."""
    newdata = np.dot(lanczos_matrix, data)
    return newdata.astype(np.float32)


# =========================
# 4. Load shared word timing and layer manifest
# =========================

word_df = pd.read_csv(MASTER_WORD_TABLE_PATH)
step3_layer_summary_df = pd.read_csv(STEP3_LAYER_SUMMARY_PATH)

with open(STEP3_OVERALL_JSON_PATH, "r", encoding="utf-8") as f:
    step3_overall_summary = json.load(f)

required_cols = ["word_index", "word_midpoint", "split"]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

required_summary_cols = [
    "layer_index",
    "layer_label",
    "reduction_components",
    "features_all_path",
]
missing_summary = [c for c in required_summary_cols if c not in step3_layer_summary_df.columns]
if missing_summary:
    raise ValueError(f"Missing required columns in Step 3 layer summary: {missing_summary}")

layer_labels = step3_layer_summary_df["layer_label"].tolist()
reduction_components_unique = sorted(step3_layer_summary_df["reduction_components"].unique().tolist())

print("\nLoaded shared inputs:")
print(f"word_df shape: {word_df.shape}")
print(f"step3_layer_summary_df shape: {step3_layer_summary_df.shape}")
print(f"n layer feature sets: {len(layer_labels)}")
print(f"first layer label: {layer_labels[0]}")
print(f"last layer label : {layer_labels[-1]}")
print(f"reduction_components_unique: {reduction_components_unique}")


# =========================
# 5. Prepare Huth-style timing variables
# =========================

word_midpoint_times = word_df["word_midpoint"].to_numpy(dtype=np.float64)
tr_midpoint_times = TR_MIDPOINT_TIMES.astype(np.float64)

print("\nTiming preview:")
print("First 5 word midpoint times:", word_midpoint_times[:5])
print("First 5 TR onset times:", TR_ONSET_TIMES[:5])
print("First 5 TR midpoint times:", tr_midpoint_times[:5])
print("Word split counts:")
print(word_df["split"].value_counts().to_string())

if not np.all(np.diff(word_midpoint_times) >= 0):
    raise ValueError("word_midpoint_times must be nondecreasing.")
if len(tr_midpoint_times) != STORY_N_TR:
    raise ValueError(f"TR midpoint count mismatch: {len(tr_midpoint_times)} vs expected {STORY_N_TR}")


# =========================
# 6. Build shared Lanczos matrix and TR table
# =========================

start_matrix_time = time.time()
lanczos_matrix = make_lanczos_matrix(
    oldtime=word_midpoint_times,
    newtime=tr_midpoint_times,
    window=LANCZOS_WINDOW,
    cutoff_mult=LANCZOS_CUTOFF_MULT,
)
matrix_elapsed_minutes = (time.time() - start_matrix_time) / 60.0

np.save(SUMMARY_NPY_DIR / "lanczos_interpolation_matrix.npy", lanczos_matrix)
np.save(SUMMARY_NPY_DIR / "tr_onset_times.npy", TR_ONSET_TIMES.astype(np.float32))
np.save(SUMMARY_NPY_DIR / "tr_midpoint_times.npy", TR_MIDPOINT_TIMES.astype(np.float32))

tr_time_table = pd.DataFrame({
    "tr_index": np.arange(STORY_N_TR),
    "tr_onset_time": TR_ONSET_TIMES,
    "tr_midpoint_time": TR_MIDPOINT_TIMES,
    "split": np.where(np.arange(STORY_N_TR) < TRAIN_TRS, "train", "test"),
})

tr_time_table_path = SUMMARY_CSV_DIR / "tr_time_table.csv"
tr_time_table.to_csv(tr_time_table_path, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nShared Lanczos matrix and TR timing table saved.")
print(f"lanczos_matrix shape: {lanczos_matrix.shape}")
print(f"Matrix elapsed minutes: {matrix_elapsed_minutes:.2f}")


# =========================
# 7. Downsample every GPT-2 XL layer feature set
# =========================

start_total_time = time.time()
step4_rows = []

for _, layer_row in step3_layer_summary_df.sort_values("layer_index").iterrows():
    layer_index = int(layer_row["layer_index"])
    layer_label = str(layer_row["layer_label"])
    layer_type = str(layer_row.get("layer_type", "unknown"))
    reduction_components = int(layer_row["reduction_components"])

    STEP3_LAYER_DIR = STEP3_DIR / layer_label
    STEP4_LAYER_DIR = STEP4_DIR / layer_label

    CSV_DIR = STEP4_LAYER_DIR / "csv"
    NPY_DIR = STEP4_LAYER_DIR / "npy"
    JSON_DIR = STEP4_LAYER_DIR / "json"

    for d in [STEP4_LAYER_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    WORD_TABLE_PATH = STEP3_LAYER_DIR / "csv" / "word_timing_with_split.csv"
    FEATURE_PATH = Path(str(layer_row["features_all_path"]))
    STEP3_SUMMARY_PATH = STEP3_LAYER_DIR / "json" / "step3_layer_summary.json"

    for p in [WORD_TABLE_PATH, FEATURE_PATH, STEP3_SUMMARY_PATH]:
        if not p.exists():
            raise FileNotFoundError(f"{layer_label}: missing required input file: {p}")

    layer_word_df = pd.read_csv(WORD_TABLE_PATH)
    word_features = np.load(FEATURE_PATH).astype(np.float32)

    with open(STEP3_SUMMARY_PATH, "r", encoding="utf-8") as f:
        step3_layer_summary = json.load(f)

    if len(layer_word_df) != len(word_df):
        raise ValueError(f"{layer_label}: word table row count does not match master word table.")
    if len(layer_word_df) != word_features.shape[0]:
        raise ValueError(
            f"{layer_label}: mismatch between word table rows ({len(layer_word_df)}) "
            f"and feature rows ({word_features.shape[0]})."
        )
    if word_features.shape[1] != reduction_components:
        raise ValueError(
            f"{layer_label}: feature dimension mismatch: "
            f"{word_features.shape[1]} vs {reduction_components}."
        )

    layer_start_time = time.time()
    tr_features = apply_lanczos_matrix(word_features, lanczos_matrix)
    elapsed_minutes = (time.time() - layer_start_time) / 60.0

    tr_features_train = tr_features[:TRAIN_TRS].astype(np.float32)
    tr_features_test = tr_features[TRAIN_TRS:].astype(np.float32)

    if tr_features.shape[0] != STORY_N_TR:
        raise ValueError(f"{layer_label}: TR feature rows mismatch: {tr_features.shape[0]} vs expected {STORY_N_TR}")
    if tr_features_train.shape[0] != TRAIN_TRS:
        raise ValueError(f"{layer_label}: train TR count mismatch: {tr_features_train.shape[0]} vs expected {TRAIN_TRS}")
    if tr_features_test.shape[0] != (STORY_N_TR - TRAIN_TRS):
        raise ValueError(
            f"{layer_label}: test TR count mismatch: {tr_features_test.shape[0]} vs expected {STORY_N_TR - TRAIN_TRS}"
        )

    np.save(NPY_DIR / f"tr_features_{layer_label}_lanczos_all.npy", tr_features)
    np.save(NPY_DIR / f"tr_features_{layer_label}_lanczos_train.npy", tr_features_train)
    np.save(NPY_DIR / f"tr_features_{layer_label}_lanczos_test.npy", tr_features_test)

    # Save lightweight per-layer copies for convenience.
    np.save(NPY_DIR / "tr_onset_times.npy", TR_ONSET_TIMES.astype(np.float32))
    np.save(NPY_DIR / "tr_midpoint_times.npy", TR_MIDPOINT_TIMES.astype(np.float32))
    tr_time_table.to_csv(CSV_DIR / "tr_time_table.csv", index=False, encoding="utf-8-sig", float_format="%.5f")

    summary_df = pd.DataFrame({
        "pipeline_type": ["snl_gpt2xl_layerwise_lanczos"],
        "method": ["Lanczos interpolation"],
        "layer_index": [layer_index],
        "layer_label": [layer_label],
        "layer_type": [layer_type],
        "reduction_method": ["PCA"],
        "reduction_components": [reduction_components],
        "n_input_words": [len(layer_word_df)],
        "input_feature_dim": [word_features.shape[1]],
        "output_trs_total": [tr_features.shape[0]],
        "train_trs": [tr_features_train.shape[0]],
        "test_trs": [tr_features_test.shape[0]],
        "output_feature_dim": [tr_features.shape[1]],
        "lanczos_window": [LANCZOS_WINDOW],
        "lanczos_cutoff_mult": [LANCZOS_CUTOFF_MULT],
        "elapsed_minutes": [elapsed_minutes],
    })

    summary_csv_path = CSV_DIR / "tr_feature_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig", float_format="%.8f")

    summary = {
        "dataset": "21styear",
        "roi_target": ROI_TARGET,
        "pipeline_type": "snl_gpt2xl_layerwise_lanczos",
        "method": "Lanczos interpolation",
        "implementation_style": "Huth-inspired DataSequence-style resampling",
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(reduction_components),
        "input_word_table_path": str(WORD_TABLE_PATH),
        "input_feature_path": str(FEATURE_PATH),
        "step3_layer_summary_path": str(STEP3_SUMMARY_PATH),
        "timing_logic": {
            "word_time_definition": "word midpoint times",
            "tr_time_definition": "TR onset times shifted by TR/2.0 to obtain TR midpoint times",
            "story_start_tr": int(STORY_START_TR),
            "story_n_tr": int(STORY_N_TR),
            "tr_seconds": float(TR),
            "story_start_time_seconds": float(STORY_START_TIME),
        },
        "lanczos_parameters": {
            "window": int(LANCZOS_WINDOW),
            "cutoff_mult": float(LANCZOS_CUTOFF_MULT),
        },
        "shapes": {
            "word_features_input": list(word_features.shape),
            "tr_features_all": list(tr_features.shape),
            "tr_features_train": list(tr_features_train.shape),
            "tr_features_test": list(tr_features_test.shape),
            "lanczos_matrix": list(lanczos_matrix.shape),
        },
        "train_test_split": {
            "train_trs": int(TRAIN_TRS),
            "test_trs": int(STORY_N_TR - TRAIN_TRS),
        },
        "outputs": {
            "tr_features_all": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_all.npy"),
            "tr_features_train": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_train.npy"),
            "tr_features_test": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_test.npy"),
            "shared_lanczos_matrix": str(SUMMARY_NPY_DIR / "lanczos_interpolation_matrix.npy"),
            "tr_onset_times": str(NPY_DIR / "tr_onset_times.npy"),
            "tr_midpoint_times": str(NPY_DIR / "tr_midpoint_times.npy"),
            "tr_time_table": str(CSV_DIR / "tr_time_table.csv"),
            "summary_csv": str(summary_csv_path),
        },
        "elapsed_minutes": float(elapsed_minutes),
    }

    summary_json_path = JSON_DIR / "step4_lanczos_summary.json"
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    shape_ok = (
        tr_features.shape == (STORY_N_TR, word_features.shape[1]) and
        tr_features_train.shape == (TRAIN_TRS, word_features.shape[1]) and
        tr_features_test.shape == (STORY_N_TR - TRAIN_TRS, word_features.shape[1])
    )
    finite_ok = (
        np.isfinite(tr_features).all() and
        np.isfinite(tr_features_train).all() and
        np.isfinite(tr_features_test).all()
    )
    split_ok = (
        len(tr_time_table) == STORY_N_TR and
        (tr_time_table["split"].value_counts().to_dict().get("train", 0) == TRAIN_TRS) and
        (tr_time_table["split"].value_counts().to_dict().get("test", 0) == STORY_N_TR - TRAIN_TRS)
    )

    if not shape_ok:
        raise ValueError(f"{layer_label}: output TR feature shapes are incorrect.")
    if not finite_ok:
        raise ValueError(f"{layer_label}: non-finite values detected in TR features.")
    if not split_ok:
        raise ValueError(f"{layer_label}: TR split table is inconsistent with expected train/test counts.")

    step4_rows.append({
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(reduction_components),
        "n_input_words": int(len(layer_word_df)),
        "input_feature_dim": int(word_features.shape[1]),
        "output_trs_total": int(tr_features.shape[0]),
        "train_trs": int(tr_features_train.shape[0]),
        "test_trs": int(tr_features_test.shape[0]),
        "output_feature_dim": int(tr_features.shape[1]),
        "elapsed_minutes": float(elapsed_minutes),
        "step4_layer_dir": str(STEP4_LAYER_DIR),
        "tr_features_all": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_all.npy"),
        "tr_features_train": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_train.npy"),
        "tr_features_test": str(NPY_DIR / f"tr_features_{layer_label}_lanczos_test.npy"),
        "summary_json": str(summary_json_path),
    })

    print(
        f"{layer_label}: downsampled all={tr_features.shape}, "
        f"train={tr_features_train.shape}, test={tr_features_test.shape}, "
        f"elapsed={elapsed_minutes:.2f} min"
    )


# =========================
# 8. Save layer-wise summaries
# =========================

total_elapsed_minutes = (time.time() - start_total_time) / 60.0
step4_summary_df = pd.DataFrame(step4_rows)

step4_summary_csv_path = SUMMARY_CSV_DIR / "step4_layerwise_feature_summary.csv"
step4_summary_df.to_csv(
    step4_summary_csv_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.8f",
)

overall_summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "snl_gpt2xl_layerwise_lanczos",
    "method": "Lanczos interpolation",
    "project_dir": str(PROJECT_DIR),
    "step3_dir": str(STEP3_DIR),
    "step4_dir": str(STEP4_DIR),
    "layer_labels": step4_summary_df["layer_label"].tolist(),
    "layer_indices": [int(x) for x in step4_summary_df["layer_index"].tolist()],
    "reduction_components": sorted([int(x) for x in step4_summary_df["reduction_components"].unique().tolist()]),
    "tr": float(TR),
    "story_start_tr": int(STORY_START_TR),
    "story_n_tr": int(STORY_N_TR),
    "train_trs": int(TRAIN_TRS),
    "test_trs": int(STORY_N_TR - TRAIN_TRS),
    "lanczos_window": int(LANCZOS_WINDOW),
    "lanczos_cutoff_mult": float(LANCZOS_CUTOFF_MULT),
    "lanczos_matrix_shape": list(lanczos_matrix.shape),
    "matrix_elapsed_minutes": float(matrix_elapsed_minutes),
    "total_elapsed_minutes": float(total_elapsed_minutes),
    "tr_time_table": str(tr_time_table_path),
    "shared_lanczos_matrix": str(SUMMARY_NPY_DIR / "lanczos_interpolation_matrix.npy"),
    "step4_layerwise_feature_summary_csv": str(step4_summary_csv_path),
    "source_step3_summary_csv": str(STEP3_LAYER_SUMMARY_PATH),
}

overall_summary_json_path = SUMMARY_JSON_DIR / "step4_lanczos_layerwise_summary.json"
with open(overall_summary_json_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# =========================
# 9. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP 4 LAYER-WISE VALIDATION CHECKS")
print("=" * 80)

all_layers_saved = len(step4_summary_df) == len(step3_layer_summary_df)
timing_ok = (
    np.all(np.diff(TR_ONSET_TIMES) > 0) and
    np.all(np.diff(TR_MIDPOINT_TIMES) > 0)
)
lanczos_finite_ok = np.isfinite(lanczos_matrix).all()
summary_shape_ok = (
    step4_summary_df["output_trs_total"].eq(STORY_N_TR).all() and
    step4_summary_df["train_trs"].eq(TRAIN_TRS).all() and
    step4_summary_df["test_trs"].eq(STORY_N_TR - TRAIN_TRS).all() and
    step4_summary_df["reduction_components"].equals(step4_summary_df["output_feature_dim"])
)
layer_label_match_ok = step4_summary_df["layer_label"].tolist() == step3_layer_summary_df.sort_values("layer_index")["layer_label"].tolist()

print(f"all_layers_saved      : {all_layers_saved}")
print(f"timing_ok             : {timing_ok}")
print(f"lanczos_finite_ok     : {lanczos_finite_ok}")
print(f"summary_shape_ok      : {summary_shape_ok}")
print(f"layer_label_match_ok  : {layer_label_match_ok}")
print(f"first layer label     : {step4_summary_df['layer_label'].iloc[0]}")
print(f"last layer label      : {step4_summary_df['layer_label'].iloc[-1]}")
print(f"lanczos_matrix shape  : {lanczos_matrix.shape}")
print(f"total_elapsed_minutes : {total_elapsed_minutes:.2f}")
print(f"Saved summary to      : {step4_summary_csv_path}")
print(f"Saved JSON summary to : {overall_summary_json_path}")

if not all_layers_saved:
    raise ValueError("Unexpected number of Step 4 layer-wise outputs.")
if not timing_ok:
    raise ValueError("TR timing arrays are not strictly increasing.")
if not lanczos_finite_ok:
    raise ValueError("Non-finite values detected in Lanczos matrix.")
if not summary_shape_ok:
    raise ValueError("Step 4 summary shapes are inconsistent.")
if not layer_label_match_ok:
    raise ValueError("Layer labels do not match Step 3 layer order.")

print("\nStep 4 layer-wise Lanczos downsampling completed successfully.")


# Step 5: Build Delayed Design Matrices For Each GPT-2 XL Layer

This step keeps the same Huth-style FIR delay logic as the previous encoding pipeline, but the feature sets are now GPT-2 XL layers rather than PCA dimensionalities.

For each `layer_00` ... `layer_48`, the Step 4 TR-level features are trimmed using the same `5 + TRIM : -TRIM` rule, z-scored column-wise, and expanded with positive delays `1..4`. Because each layer was reduced to the same fixed dimensionality in Step 3, every layer produces a delayed design matrix with the same number of columns.

These delayed matrices are still stimulus/model features only. They can be reused later by every network-specific raw and SRM-reconstructed encoding analysis without rerunning Steps 1-5.


In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ROI_NAME = "all_networks"
ROI_TARGET = "all_networks_fixed_srm50"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"

PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
STEP4_DIR = PROJECT_DIR / "step4_downsample_to_tr_lanczos_layerwise"
STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
SUMMARY_CSV_DIR = STEP5_DIR / "csv"
SUMMARY_JSON_DIR = STEP5_DIR / "json"

for d in [PROJECT_ROOT, PROJECT_DIR, STEP5_DIR, SUMMARY_CSV_DIR, SUMMARY_JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"ROI name: {ROI_NAME}")
print(f"ROI target: {ROI_TARGET}")
print(f"Encoding run name: {ENCODING_RUN_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 4 directory: {STEP4_DIR}")
print(f"Step 5 directory: {STEP5_DIR}")


# =========================
# 2. Input settings
# =========================

STEP4_LAYER_SUMMARY_PATH = STEP4_DIR / "csv" / "step4_layerwise_feature_summary.csv"
STEP4_OVERALL_JSON_PATH = STEP4_DIR / "json" / "step4_lanczos_layerwise_summary.json"

TRIM = 5
N_DELAYS = 4

for p in [STEP4_LAYER_SUMMARY_PATH, STEP4_OVERALL_JSON_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Step 4 layer summary path: {STEP4_LAYER_SUMMARY_PATH}")
print(f"TRIM = {TRIM}")
print(f"N_DELAYS = {N_DELAYS}")


# =========================
# 3. Huth-style helper functions
# =========================

def zs(v: np.ndarray) -> np.ndarray:
    """Huth-style column-wise z-score."""
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def make_delayed(stim: np.ndarray, delays) -> np.ndarray:
    """
    Positive delays shift the stimulus forward in time,
    then concatenate delayed copies column-wise.
    """
    stim = np.asarray(stim, dtype=np.float32)
    nt, ndim = stim.shape

    delayed_list = []
    for d in delays:
        dstim = np.zeros((nt, ndim), dtype=np.float32)
        if d < 0:
            dstim[:d, :] = stim[-d:, :]
        elif d > 0:
            dstim[d:, :] = stim[:-d, :]
        else:
            dstim[:, :] = stim
        delayed_list.append(dstim)

    return np.hstack(delayed_list).astype(np.float32)


# =========================
# 4. Load Step 4 layer-wise summary
# =========================

step4_summary_df = pd.read_csv(STEP4_LAYER_SUMMARY_PATH).sort_values("layer_index").reset_index(drop=True)
with open(STEP4_OVERALL_JSON_PATH, "r", encoding="utf-8") as f:
    step4_overall_summary = json.load(f)

required_cols = [
    "layer_index",
    "layer_label",
    "reduction_components",
    "train_trs",
    "test_trs",
    "output_feature_dim",
    "tr_features_all",
    "tr_features_train",
    "tr_features_test",
]
missing_cols = [c for c in required_cols if c not in step4_summary_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in Step 4 summary: {missing_cols}")

print("\nLoaded Step 4 layer-wise summary:")
print(step4_summary_df[["layer_label", "reduction_components", "train_trs", "test_trs", "output_feature_dim"]].head())
print("...")
print(step4_summary_df[["layer_label", "reduction_components", "train_trs", "test_trs", "output_feature_dim"]].tail())
print(f"n layer feature sets: {len(step4_summary_df)}")


# =========================
# 5. Build delayed matrices for every GPT-2 XL layer
# =========================

delays = range(1, N_DELAYS + 1)
step5_rows = []

for _, layer_row in step4_summary_df.iterrows():
    layer_index = int(layer_row["layer_index"])
    layer_label = str(layer_row["layer_label"])
    layer_type = str(layer_row.get("layer_type", "unknown"))
    reduction_components = int(layer_row["reduction_components"])

    STEP4_LAYER_DIR = STEP4_DIR / layer_label
    STEP5_LAYER_DIR = STEP5_DIR / layer_label

    CSV_DIR = STEP5_LAYER_DIR / "csv"
    NPY_DIR = STEP5_LAYER_DIR / "npy"
    JSON_DIR = STEP5_LAYER_DIR / "json"

    for d in [STEP5_LAYER_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    X_ALL_PATH = Path(str(layer_row["tr_features_all"]))
    X_TRAIN_PATH_IN = Path(str(layer_row["tr_features_train"]))
    X_TEST_PATH_IN = Path(str(layer_row["tr_features_test"]))
    STEP4_SUMMARY_PATH = STEP4_LAYER_DIR / "json" / "step4_lanczos_summary.json"

    for p in [X_ALL_PATH, X_TRAIN_PATH_IN, X_TEST_PATH_IN, STEP4_SUMMARY_PATH]:
        if not p.exists():
            raise FileNotFoundError(f"{layer_label}: missing required input file: {p}")

    X_all = np.load(X_ALL_PATH).astype(np.float32)
    X_train_story = np.load(X_TRAIN_PATH_IN).astype(np.float32)
    X_test_story = np.load(X_TEST_PATH_IN).astype(np.float32)

    with open(STEP4_SUMMARY_PATH, "r", encoding="utf-8") as f:
        step4_summary = json.load(f)

    if X_all.shape[0] != (X_train_story.shape[0] + X_test_story.shape[0]):
        raise ValueError(
            f"{layer_label}: mismatch: X_all rows={X_all.shape[0]} but "
            f"train+test={X_train_story.shape[0] + X_test_story.shape[0]}"
        )

    feature_dim = int(X_all.shape[1])
    if feature_dim != reduction_components:
        raise ValueError(
            f"{layer_label}: feature_dim={feature_dim}, expected {reduction_components}"
        )

    # Original Huth-inspired logic:
    #   stim = zscore(downsampled_feat[s][5+trim:-trim])
    X_train_trimmed = X_train_story[5 + TRIM : -TRIM].astype(np.float32)
    X_test_trimmed = X_test_story[5 + TRIM : -TRIM].astype(np.float32)

    if X_train_trimmed.shape[0] <= 0 or X_test_trimmed.shape[0] <= 0:
        raise ValueError(f"{layer_label}: trimmed train/test matrices have non-positive time dimension.")

    X_train_trimmed = zs(X_train_trimmed)
    X_test_trimmed = zs(X_test_trimmed)

    X_train_delayed = make_delayed(X_train_trimmed, delays)
    X_test_delayed = make_delayed(X_test_trimmed, delays)

    expected_dim = feature_dim * N_DELAYS
    if X_train_delayed.shape[1] != expected_dim:
        raise ValueError(
            f"{layer_label}: unexpected train delayed dim: {X_train_delayed.shape[1]} vs expected {expected_dim}"
        )
    if X_test_delayed.shape[1] != expected_dim:
        raise ValueError(
            f"{layer_label}: unexpected test delayed dim: {X_test_delayed.shape[1]} vs expected {expected_dim}"
        )

    X_TRAIN_OUT = NPY_DIR / "X_train_delayed_lanczos_huthstyle.npy"
    X_TEST_OUT = NPY_DIR / "X_test_delayed_lanczos_huthstyle.npy"

    np.save(X_TRAIN_OUT, X_train_delayed)
    np.save(X_TEST_OUT, X_test_delayed)

    column_rows = []
    for delay_idx, delay in enumerate(delays, start=1):
        for base_feat_idx in range(feature_dim):
            column_rows.append({
                "delayed_column_index": (delay_idx - 1) * feature_dim + base_feat_idx,
                "base_feature_index": base_feat_idx,
                "delay": int(delay),
                "layer_index": int(layer_index),
                "layer_label": layer_label,
            })

    columns_df = pd.DataFrame(column_rows)
    columns_csv_path = CSV_DIR / "delayed_feature_columns_huthstyle.csv"
    columns_df.to_csv(columns_csv_path, index=False, encoding="utf-8-sig")

    summary_df = pd.DataFrame({
        "pipeline_type": ["snl_gpt2xl_layerwise_huthstyle_fir"],
        "layer_index": [layer_index],
        "layer_label": [layer_label],
        "layer_type": [layer_type],
        "reduction_method": ["PCA"],
        "reduction_components": [reduction_components],
        "input_all_shape": [str(tuple(X_all.shape))],
        "input_train_shape_pretrim": [str(tuple(X_train_story.shape))],
        "input_test_shape_pretrim": [str(tuple(X_test_story.shape))],
        "input_train_shape_posttrim": [str(tuple(X_train_trimmed.shape))],
        "input_test_shape_posttrim": [str(tuple(X_test_trimmed.shape))],
        "output_train_shape": [str(tuple(X_train_delayed.shape))],
        "output_test_shape": [str(tuple(X_test_delayed.shape))],
        "trim": [TRIM],
        "n_delays": [N_DELAYS],
        "n_base_features": [feature_dim],
        "output_feature_dim": [expected_dim],
        "train_mean_abs": [float(np.mean(np.abs(X_train_delayed)))],
        "test_mean_abs": [float(np.mean(np.abs(X_test_delayed)))],
    })

    summary_csv_path = CSV_DIR / "delayed_design_summary_huthstyle.csv"
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig", float_format="%.8f")

    summary = {
        "dataset": "21styear",
        "roi_target": ROI_TARGET,
        "pipeline_type": "snl_gpt2xl_layerwise_huthstyle_fir",
        "input_source": "Huth-style Lanczos-downsampled TR features",
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(reduction_components),
        "trim": int(TRIM),
        "n_delays": int(N_DELAYS),
        "delay_definition": "range(1, ndelays+1)",
        "n_base_features": int(feature_dim),
        "output_feature_dim": int(expected_dim),
        "input_all_shape": list(X_all.shape),
        "input_train_shape_pretrim": list(X_train_story.shape),
        "input_test_shape_pretrim": list(X_test_story.shape),
        "input_train_shape_posttrim": list(X_train_trimmed.shape),
        "input_test_shape_posttrim": list(X_test_trimmed.shape),
        "output_train_shape": list(X_train_delayed.shape),
        "output_test_shape": list(X_test_delayed.shape),
        "original_huth_inspirations": {
            "encoding_utils_apply_zscore_and_hrf": [
                "stim = [zscore(downsampled_feat[s][5+trim:-trim]) for s in stories]",
                "delays = range(1, ndelays+1)",
                "delstim = make_delayed(stim, delays)"
            ],
            "ridge_utils_npp_zscore": "z-score each feature column",
            "ridge_utils_utils_make_delayed": "positive delays shift stimulus forward in time and hstack delayed copies",
        },
        "outputs": {
            "train_delayed_path": str(X_TRAIN_OUT),
            "test_delayed_path": str(X_TEST_OUT),
            "columns_csv": str(columns_csv_path),
            "summary_csv": str(summary_csv_path),
        },
        "source_step4_summary": str(STEP4_SUMMARY_PATH),
    }

    summary_json_path = JSON_DIR / "step5_summary_huthstyle.json"
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    shape_ok = (
        X_train_delayed.shape[1] == expected_dim and
        X_test_delayed.shape[1] == expected_dim
    )
    finite_ok = (
        np.isfinite(X_train_delayed).all() and
        np.isfinite(X_test_delayed).all()
    )
    trim_ok = (
        X_train_trimmed.shape[0] == X_train_story.shape[0] - (5 + 2 * TRIM) and
        X_test_trimmed.shape[0] == X_test_story.shape[0] - (5 + 2 * TRIM)
    )

    if not shape_ok:
        raise ValueError(f"{layer_label}: delayed design matrices have incorrect output dimensions.")
    if not finite_ok:
        raise ValueError(f"{layer_label}: non-finite values found in delayed design matrices.")
    if not trim_ok:
        raise ValueError(f"{layer_label}: unexpected post-trim time dimension.")

    step5_rows.append({
        "layer_index": int(layer_index),
        "layer_label": layer_label,
        "layer_type": layer_type,
        "reduction_method": "PCA",
        "reduction_components": int(reduction_components),
        "input_train_shape_pretrim": str(tuple(X_train_story.shape)),
        "input_test_shape_pretrim": str(tuple(X_test_story.shape)),
        "input_train_shape_posttrim": str(tuple(X_train_trimmed.shape)),
        "input_test_shape_posttrim": str(tuple(X_test_trimmed.shape)),
        "output_train_shape": str(tuple(X_train_delayed.shape)),
        "output_test_shape": str(tuple(X_test_delayed.shape)),
        "trim": int(TRIM),
        "n_delays": int(N_DELAYS),
        "n_base_features": int(feature_dim),
        "output_feature_dim": int(expected_dim),
        "train_mean_abs": float(np.mean(np.abs(X_train_delayed))),
        "test_mean_abs": float(np.mean(np.abs(X_test_delayed))),
        "step5_layer_dir": str(STEP5_LAYER_DIR),
        "train_delayed_path": str(X_TRAIN_OUT),
        "test_delayed_path": str(X_TEST_OUT),
        "summary_json": str(summary_json_path),
    })

    print(
        f"{layer_label}: delayed train={X_train_delayed.shape}, "
        f"test={X_test_delayed.shape}, output_dim={expected_dim}"
    )


# =========================
# 6. Save layer-wise summaries
# =========================

step5_summary_df = pd.DataFrame(step5_rows)
step5_summary_csv_path = SUMMARY_CSV_DIR / "step5_layerwise_design_summary.csv"
step5_summary_df.to_csv(
    step5_summary_csv_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.8f",
)

overall_summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "snl_gpt2xl_layerwise_huthstyle_fir",
    "project_dir": str(PROJECT_DIR),
    "step4_dir": str(STEP4_DIR),
    "step5_dir": str(STEP5_DIR),
    "layer_labels": step5_summary_df["layer_label"].tolist(),
    "layer_indices": [int(x) for x in step5_summary_df["layer_index"].tolist()],
    "reduction_components": sorted([int(x) for x in step5_summary_df["reduction_components"].unique().tolist()]),
    "trim": int(TRIM),
    "n_delays": int(N_DELAYS),
    "source_step4_summary_csv": str(STEP4_LAYER_SUMMARY_PATH),
    "step5_layerwise_design_summary_csv": str(step5_summary_csv_path),
}

overall_summary_json_path = SUMMARY_JSON_DIR / "step5_huthstyle_layerwise_summary.json"
with open(overall_summary_json_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# =========================
# 7. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP 5 LAYER-WISE VALIDATION CHECKS")
print("=" * 80)

all_layers_saved = len(step5_summary_df) == len(step4_summary_df)
dimension_relation_ok = (
    step5_summary_df["output_feature_dim"].to_numpy()
    == step5_summary_df["reduction_components"].to_numpy() * N_DELAYS
).all()
train_rows_ok = step5_summary_df["output_train_shape"].str.contains("1098").all()
test_rows_ok = step5_summary_df["output_test_shape"].str.contains("1098").all()
layer_label_match_ok = step5_summary_df["layer_label"].tolist() == step4_summary_df["layer_label"].tolist()

print(f"all_layers_saved      : {all_layers_saved}")
print(f"dimension_relation_ok : {dimension_relation_ok}")
print(f"train_rows_ok         : {train_rows_ok}")
print(f"test_rows_ok          : {test_rows_ok}")
print(f"layer_label_match_ok  : {layer_label_match_ok}")
print(f"first layer label     : {step5_summary_df['layer_label'].iloc[0]}")
print(f"last layer label      : {step5_summary_df['layer_label'].iloc[-1]}")
print(f"Saved summary to      : {step5_summary_csv_path}")
print(f"Saved JSON summary to : {overall_summary_json_path}")

if not all_layers_saved:
    raise ValueError("Unexpected number of Step 5 layer-wise outputs.")
if not dimension_relation_ok:
    raise ValueError("Delayed output feature dimension does not equal reduction components * N_DELAYS.")
if not train_rows_ok or not test_rows_ok:
    raise ValueError("Unexpected post-trim train/test row counts.")
if not layer_label_match_ok:
    raise ValueError("Layer labels do not match Step 4 layer order.")

print("\nStep 5 layer-wise delayed design matrix construction completed successfully.")


ALL previous stpes for model fitting now are finished

You do not have to adjust anything from the Step1-Step5 in this piepiline!!!!!!!!!!!!!!!!!!!!!!!

# Step 6: Prepare fMRI Responses For Each Network

Steps 1-5 define the shared layer-wise stimulus feature pipeline. Step 6 starts the network-specific neural-response side of the analysis.

For each Schaefer-17 network, this step reads the fixed-SRM50 Step 2 train/test fMRI response arrays, applies the same Huth-style time trimming used by the delayed GPT-2 design matrices, and converts responses from `voxels x time` into `time x voxels`. All network outputs are saved under one shared Step 6 folder, with one subfolder per network. It saves both subject-level response arrays and group-average response arrays, which lets later layer-wise encoding steps use exactly the same response timing for every GPT-2 layer.

The reference design matrix is taken from `layer_00` only for a time-axis alignment check. This does not mean later encoding uses only layer 00; all layers have the same post-trim 1098 TRs and 200 delayed features, so one layer is sufficient here for checking response alignment.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP6_ROOT_DIR = PROJECT_DIR / "step6_prepare_fmri_response"

SRM_PROJECT_DIR = BASE_DIR / "runs"
SRM_RUN_SUFFIX = "400_parcels_fixed_srm50"

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_REFERENCE_LABEL = "layer_00"
REFERENCE_STEP5_DIR = STEP5_DIR / LAYER_REFERENCE_LABEL

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Layer-wise Step 5 directory: {STEP5_DIR}")
print(f"Step 6 response root: {STEP6_ROOT_DIR}")
print(f"Reference Step 5 layer: {LAYER_REFERENCE_LABEL}")
print(f"SRM project directory: {SRM_PROJECT_DIR}")
print(f"Number of networks: {len(NETWORK_NAMES)}")


# =========================
# 2. Shared Step 5 inputs
# =========================

# Step 6 only needs the response trim and the post-trim time axis.
# All layer-wise design matrices use the same time axis, so layer_00 is enough
# as a reference alignment matrix here.
X_train_path = REFERENCE_STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_test_path = REFERENCE_STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"
step5_summary_path = STEP5_DIR / "json" / "step5_huthstyle_layerwise_summary.json"
step5_reference_summary_path = REFERENCE_STEP5_DIR / "json" / "step5_summary_huthstyle.json"

for p in [X_train_path, X_test_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing reference Step 5 design matrix: {p}")

if step5_summary_path.exists():
    with open(step5_summary_path, "r", encoding="utf-8") as f:
        step5_summary = json.load(f)
elif step5_reference_summary_path.exists():
    with open(step5_reference_summary_path, "r", encoding="utf-8") as f:
        step5_summary = json.load(f)
else:
    raise FileNotFoundError(
        f"Missing Step 5 summary file: {step5_summary_path} or {step5_reference_summary_path}"
    )

X_train = np.load(X_train_path, mmap_mode="r")
X_test = np.load(X_test_path, mmap_mode="r")
TRIM = int(step5_summary["trim"])

print(f"Reference Step 5 layer label: {LAYER_REFERENCE_LABEL}")
print(f"Reference X_train shape = {X_train.shape}")
print(f"Reference X_test shape  = {X_test.shape}")
print(f"Recovered TRIM from Step 5 summary: {TRIM}")


# =========================
# 3. Helper: Huth-style response trim
# =========================

# BrainIAK/SRM Step 2 outputs are [voxels x time], where time=1113 for train/test.
# Huth-style response trimming mirrors the stimulus side:
#   Y = Y[:, 5 + trim : -trim]
# because Step 5 used X_story[5 + trim : -trim].
def trim_response_huthstyle(Y_vox_time: np.ndarray, trim: int) -> np.ndarray:
    return Y_vox_time[:, 5 + trim : -trim].astype(np.float32)


# =========================
# 4. Prepare one network
# =========================

def prepare_one_network_response(roi_name: str) -> dict:
    roi_target = f"{roi_name}_only"
    run_name = f"{roi_name}_{SRM_RUN_SUFFIX}"

    step6_dir = STEP6_ROOT_DIR / roi_name

    csv_dir = step6_dir / "csv"
    npy_dir = step6_dir / "npy"
    json_dir = step6_dir / "json"

    for d in [STEP6_ROOT_DIR, step6_dir, csv_dir, npy_dir, json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    run_dir = SRM_PROJECT_DIR / run_name
    step2_brainiak_dir = run_dir / "step2_split_zscore"

    train_resp_path = step2_brainiak_dir / "npy" / "train_data.npy"
    test_resp_path = step2_brainiak_dir / "npy" / "test_data.npy"
    step2_brainiak_config_path = step2_brainiak_dir / "step2_config.json"

    for p in [train_resp_path, test_resp_path, step2_brainiak_config_path]:
        if not p.exists():
            raise FileNotFoundError(f"{roi_name}: missing SRM Step 2 input file: {p}")

    print("\n" + "=" * 80)
    print(f"Preparing Step 6 response for network: {roi_name}")
    print("=" * 80)
    print(f"Step 6 network directory: {step6_dir}")
    print(f"SRM train response path: {train_resp_path}")
    print(f"SRM test response path: {test_resp_path}")

    Y_train_list = np.load(train_resp_path, allow_pickle=True)
    Y_test_list = np.load(test_resp_path, allow_pickle=True)

    Y_train_list = [np.asarray(x, dtype=np.float32) for x in Y_train_list]   # voxels x time
    Y_test_list = [np.asarray(x, dtype=np.float32) for x in Y_test_list]     # voxels x time

    with open(step2_brainiak_config_path, "r", encoding="utf-8") as f:
        step2_brainiak_config = json.load(f)

    print("\nLoaded inputs:")
    print(f"Number of subjects = {len(Y_train_list)}")
    print(f"Y_train subject 1 shape (voxels x time) = {Y_train_list[0].shape}")
    print(f"Y_test subject 1 shape (voxels x time) = {Y_test_list[0].shape}")
    print(f"Reference X_train shape = {X_train.shape}")
    print(f"Reference X_test shape = {X_test.shape}")

    if len(Y_train_list) == 0 or len(Y_test_list) == 0:
        raise ValueError(f"{roi_name}: empty SRM response list.")
    if len(Y_train_list) != len(Y_test_list):
        raise ValueError(f"{roi_name}: mismatch in number of train/test response subjects.")
    if not all(y.shape == Y_train_list[0].shape for y in Y_train_list):
        raise ValueError(f"{roi_name}: inconsistent Y_train subject shapes.")
    if not all(y.shape == Y_test_list[0].shape for y in Y_test_list):
        raise ValueError(f"{roi_name}: inconsistent Y_test subject shapes.")
    if not all(np.isfinite(y).all() for y in Y_train_list):
        raise ValueError(f"{roi_name}: non-finite values found in Y_train_list.")
    if not all(np.isfinite(y).all() for y in Y_test_list):
        raise ValueError(f"{roi_name}: non-finite values found in Y_test_list.")

    # Apply Huth-style response trim to each subject.
    Y_train_trimmed_list = [trim_response_huthstyle(y, TRIM) for y in Y_train_list]
    Y_test_trimmed_list = [trim_response_huthstyle(y, TRIM) for y in Y_test_list]

    print("\nAfter Huth-style response trim:")
    print(f"Y_train_trimmed subject 1 shape = {Y_train_trimmed_list[0].shape}")
    print(f"Y_test_trimmed subject 1 shape = {Y_test_trimmed_list[0].shape}")

    if not all(y.shape == Y_train_trimmed_list[0].shape for y in Y_train_trimmed_list):
        raise ValueError(f"{roi_name}: inconsistent trimmed Y_train shapes across subjects.")
    if not all(y.shape == Y_test_trimmed_list[0].shape for y in Y_test_trimmed_list):
        raise ValueError(f"{roi_name}: inconsistent trimmed Y_test shapes across subjects.")

    # Original SRM response is [voxels x time].
    # Encoding expects [time x voxels].
    Y_train_time_vox_list = [y.T.astype(np.float32) for y in Y_train_trimmed_list]
    Y_test_time_vox_list = [y.T.astype(np.float32) for y in Y_test_trimmed_list]

    print("\nConverted to time x voxels:")
    print(f"Y_train_time_vox subject 1 shape = {Y_train_time_vox_list[0].shape}")
    print(f"Y_test_time_vox subject 1 shape = {Y_test_time_vox_list[0].shape}")

    Y_train_group = np.mean(np.stack(Y_train_time_vox_list, axis=0), axis=0).astype(np.float32)
    Y_test_group = np.mean(np.stack(Y_test_time_vox_list, axis=0), axis=0).astype(np.float32)

    print("\nGroup-average responses:")
    print(f"Y_train_group shape = {Y_train_group.shape}")
    print(f"Y_test_group shape = {Y_test_group.shape}")

    print("\nCritical alignment checks:")
    print(f"Reference X_train timepoints = {X_train.shape[0]}")
    print(f"Y_train_group timepoints = {Y_train_group.shape[0]}")
    print(f"Reference X_test timepoints = {X_test.shape[0]}")
    print(f"Y_test_group timepoints = {Y_test_group.shape[0]}")

    if X_train.shape[0] != Y_train_group.shape[0]:
        raise ValueError(
            f"{roi_name}: train time mismatch: X_train has {X_train.shape[0]} timepoints, "
            f"but Y_train_group has {Y_train_group.shape[0]}."
        )
    if X_test.shape[0] != Y_test_group.shape[0]:
        raise ValueError(
            f"{roi_name}: test time mismatch: X_test has {X_test.shape[0]} timepoints, "
            f"but Y_test_group has {Y_test_group.shape[0]}."
        )

    # Save outputs using stable filenames expected by later encoding steps.
    Y_train_group_path = npy_dir / "Y_train_group_huthstyle.npy"
    Y_test_group_path = npy_dir / "Y_test_group_huthstyle.npy"
    Y_train_list_path = npy_dir / "Y_train_list_huthstyle.npy"
    Y_test_list_path = npy_dir / "Y_test_list_huthstyle.npy"

    np.save(Y_train_group_path, Y_train_group)
    np.save(Y_test_group_path, Y_test_group)
    np.save(Y_train_list_path, np.array(Y_train_time_vox_list, dtype=object), allow_pickle=True)
    np.save(Y_test_list_path, np.array(Y_test_time_vox_list, dtype=object), allow_pickle=True)

    print(f"\nSaved Y_train_group to: {Y_train_group_path}")
    print(f"Saved Y_test_group to: {Y_test_group_path}")
    print(f"Saved Y_train_list to: {Y_train_list_path}")
    print(f"Saved Y_test_list to: {Y_test_list_path}")

    shape_rows = []
    for i, (yt, ys) in enumerate(zip(Y_train_time_vox_list, Y_test_time_vox_list)):
        shape_rows.append({
            "roi_name": roi_name,
            "subject_index": i,
            "train_shape": str(tuple(yt.shape)),
            "test_shape": str(tuple(ys.shape)),
            "n_timepoints_train": int(yt.shape[0]),
            "n_voxels": int(yt.shape[1]),
            "n_timepoints_test": int(ys.shape[0]),
        })

    shape_df = pd.DataFrame(shape_rows)
    shape_csv_path = csv_dir / "subject_response_shapes_huthstyle.csv"
    shape_df.to_csv(shape_csv_path, index=False, encoding="utf-8-sig")

    summary_df = pd.DataFrame({
        "roi_name": [roi_name],
        "n_subjects": [len(Y_train_time_vox_list)],
        "trim": [TRIM],
        "train_group_shape": [str(tuple(Y_train_group.shape))],
        "test_group_shape": [str(tuple(Y_test_group.shape))],
        "reference_layer": [LAYER_REFERENCE_LABEL],
        "reference_X_train_shape": [str(tuple(X_train.shape))],
        "reference_X_test_shape": [str(tuple(X_test.shape))],
        "alignment_train_ok": [bool(X_train.shape[0] == Y_train_group.shape[0])],
        "alignment_test_ok": [bool(X_test.shape[0] == Y_test_group.shape[0])],
    })

    summary_csv_path = csv_dir / "response_preparation_summary_huthstyle.csv"
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved shape table to: {shape_csv_path}")
    print(f"Saved summary table to: {summary_csv_path}")

    summary = {
        "dataset": "21styear",
        "roi_name": roi_name,
        "roi_target": roi_target,
        "pipeline_type": "prepare_fmri_response_huthstyle_fixed_srm50_layerwise",
        "source_srm_run": run_name,
        "source_srm_step2_config": str(step2_brainiak_config_path),
        "source_layerwise_step5_summary": str(step5_summary_path),
        "reference_step5_layer_label": LAYER_REFERENCE_LABEL,
        "reference_X_train_path": str(X_train_path),
        "reference_X_test_path": str(X_test_path),

        "trim": int(TRIM),
        "srm_input_layout": "voxels x time",
        "encoding_output_layout": "time x voxels",

        "n_subjects": int(len(Y_train_time_vox_list)),
        "train_subject0_shape_before_trim": list(Y_train_list[0].shape),
        "test_subject0_shape_before_trim": list(Y_test_list[0].shape),
        "train_subject0_shape_after_trim": list(Y_train_trimmed_list[0].shape),
        "test_subject0_shape_after_trim": list(Y_test_trimmed_list[0].shape),
        "train_subject0_shape_time_vox": list(Y_train_time_vox_list[0].shape),
        "test_subject0_shape_time_vox": list(Y_test_time_vox_list[0].shape),

        "Y_train_group_shape": list(Y_train_group.shape),
        "Y_test_group_shape": list(Y_test_group.shape),
        "reference_X_train_shape": list(X_train.shape),
        "reference_X_test_shape": list(X_test.shape),

        "alignment": {
            "train_ok": bool(X_train.shape[0] == Y_train_group.shape[0]),
            "test_ok": bool(X_test.shape[0] == Y_test_group.shape[0]),
        },

        "outputs": {
            "Y_train_group": str(Y_train_group_path),
            "Y_test_group": str(Y_test_group_path),
            "Y_train_list": str(Y_train_list_path),
            "Y_test_list": str(Y_test_list_path),
            "shape_csv": str(shape_csv_path),
            "summary_csv": str(summary_csv_path),
        },
    }

    summary_json_path = json_dir / "step6_summary_huthstyle.json"
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved summary to: {summary_json_path}")

    shape_ok = (
        all(y.shape == Y_train_time_vox_list[0].shape for y in Y_train_time_vox_list) and
        all(y.shape == Y_test_time_vox_list[0].shape for y in Y_test_time_vox_list)
    )
    finite_ok = (
        all(np.isfinite(y).all() for y in Y_train_time_vox_list) and
        all(np.isfinite(y).all() for y in Y_test_time_vox_list) and
        np.isfinite(Y_train_group).all() and
        np.isfinite(Y_test_group).all()
    )
    alignment_ok = (
        X_train.shape[0] == Y_train_group.shape[0] and
        X_test.shape[0] == Y_test_group.shape[0]
    )

    print("\nStep-end validation checks:")
    print(f"shape_ok: {shape_ok}")
    print(f"finite_ok: {finite_ok}")
    print(f"alignment_ok: {alignment_ok}")
    print(f"Y_train_group shape: {Y_train_group.shape}")
    print(f"Y_test_group shape: {Y_test_group.shape}")
    print(f"Reference X_train shape: {X_train.shape}")
    print(f"Reference X_test shape: {X_test.shape}")

    if not shape_ok:
        raise ValueError(f"{roi_name}: subject response shapes are inconsistent after trimming/transposing.")
    if not finite_ok:
        raise ValueError(f"{roi_name}: non-finite values detected in prepared fMRI responses.")
    if not alignment_ok:
        raise ValueError(f"{roi_name}: stimulus and response time dimensions do not align.")

    return {
        "roi_name": roi_name,
        "run_name": run_name,
        "step6_dir": str(step6_dir),
        "n_subjects": int(len(Y_train_time_vox_list)),
        "n_voxels": int(Y_train_group.shape[1]),
        "Y_train_group_shape": str(tuple(Y_train_group.shape)),
        "Y_test_group_shape": str(tuple(Y_test_group.shape)),
        "reference_layer": LAYER_REFERENCE_LABEL,
        "alignment_ok": bool(alignment_ok),
        "summary_json": str(summary_json_path),
    }


# =========================
# 5. Run all networks
# =========================

all_network_rows = []
for network_name in NETWORK_NAMES:
    all_network_rows.append(prepare_one_network_response(network_name))

all_network_summary_df = pd.DataFrame(all_network_rows)

summary_csv = STEP6_ROOT_DIR / "step6_all_network_response_summary.csv"
all_network_summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("ALL NETWORK STEP 6 SUMMARY")
print("=" * 80)
print(all_network_summary_df.to_string(index=False))
print(f"\nSaved all-network Step 6 summary to: {summary_csv}")

if not all(all_network_summary_df["alignment_ok"]):
    raise ValueError("At least one network failed Step 6 alignment checks.")

print("\nStep 6 completed successfully for all networks.")


step7_fit_encoding_model_huthstyle_multiROI

# Step 7A: Select Group-Level Alpha For Each Network And GPT-2 Layer

This step selects the ridge regularization parameter for the first layer-wise encoding analysis. The target is the raw anatomically aligned group-average fMRI response prepared in Step 6.

The code iterates over all `network x GPT-2 layer` pairs. For each pair, it reads the delayed layer-wise design matrix from Step 5 and the matching group-level response matrix from Step 6, then evaluates a log-spaced alpha grid using Huth-style bootstrap chunk validation. The selected alpha is saved for later group-level raw anatomical encoding.

This cell only selects alpha for raw group-level encoding. It does not fit the final encoding model, does not run subject-level models, and does not use SRM-reconstructed responses. Here `single_alpha=True` is used so each `network x layer` pair receives one shared alpha across voxels. This is the more stable setting for a group-level layer-wise comparison and can later be reused as a fixed hyperparameter for participant-level follow-up analyses, although such participant-level analyses should be described as using the group-selected alpha rather than subject-optimized alpha.


In [ ]:
from pathlib import Path
import json
import random
import time
import itertools as itools

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP6_DIR = PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = PROJECT_DIR / "step7a_select_alpha_layerwise"

FIG_DIR = STEP7A_DIR / "figures"
CSV_DIR = STEP7A_DIR / "csv"
JSON_DIR = STEP7A_DIR / "json"

for d in [STEP7A_DIR, FIG_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Step 5 layer-wise design directory: {STEP5_DIR}")
print(f"Step 6 response directory: {STEP6_DIR}")
print(f"Step 7A output directory: {STEP7A_DIR}")
print(f"Number of networks: {len(NETWORK_NAMES)}")
print(f"Number of GPT-2 XL layers including embedding layer: {len(LAYER_LABELS)}")


# =========================
# 2. Huth-style alpha selection settings
# =========================

ALPHAS = np.logspace(1, 5, 20).astype(np.float32)
NBOOTS = 15
CHUNKLEN = 40
NCHUNKS = 6
SINGCUTOFF = 1e-10
USE_CORR = True
SEED = 0
SINGLE_ALPHA = True

print("\nHuth-style alpha-selection parameters:")
print("ALPHAS =", [f"{x:.5f}" for x in ALPHAS])
print("NBOOTS =", NBOOTS)
print("CHUNKLEN =", CHUNKLEN)
print("NCHUNKS =", NCHUNKS)
print("SINGCUTOFF =", f"{SINGCUTOFF:.10f}")
print("USE_CORR =", USE_CORR)
print("SINGLE_ALPHA =", SINGLE_ALPHA)
print("SEED =", SEED)


# =========================
# 3. Helper functions
# =========================

def fmt_float(x: float, digits: int = 6) -> str:
    """Format values without scientific notation for printed summaries."""
    return f"{float(x):.{digits}f}"


def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def columnwise_corr(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise Pearson r using Huth-style z-scoring.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return np.nan_to_num((zs(y_true) * zs(y_pred)).mean(axis=0)).astype(np.float32)


def columnwise_r2(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise cross-validated R2.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - y_true.mean(axis=0, keepdims=True)) ** 2, axis=0)
    r2 = 1.0 - ss_res / np.maximum(ss_tot, 1e-12)
    return np.nan_to_num(r2).astype(np.float32)


def ridge_fit_predict_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    alpha,
    singcutoff: float = 1e-10,
):
    """
    Huth-style SVD ridge fit + predict.

    alpha can be either:
        - a scalar, meaning one alpha shared across all response columns
        - a vector with length n_targets, meaning one alpha per voxel/target

    This matters when SINGLE_ALPHA=False: the held-out sanity check and later
    final model should use the selected voxel-wise alphas, not the global
    average-best alpha.
    """
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))

    U = U[:, :ngoodS]
    S = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = np.dot(U.T, np.nan_to_num(Rresp))

    alpha_arr = np.asarray(alpha, dtype=np.float64)
    if alpha_arr.ndim == 0:
        wt = Vh.T.dot(np.diag(S / (S**2 + float(alpha_arr) ** 2))).dot(UR)
    else:
        alpha_arr = alpha_arr.ravel()
        if alpha_arr.shape[0] != Rresp.shape[1]:
            raise ValueError(
                f"Voxel-wise alpha length {alpha_arr.shape[0]} does not match "
                f"number of response columns {Rresp.shape[1]}."
            )
        shrink = S[:, None] / (S[:, None] ** 2 + alpha_arr[None, :] ** 2)
        wt = Vh.T.dot(shrink * UR)

    pred = np.dot(Pstim, wt).astype(np.float32)

    return wt.astype(np.float32), pred


def ridge_score_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    Presp: np.ndarray,
    alpha: float,
    use_corr: bool = True,
    singcutoff: float = 1e-10,
) -> np.ndarray:
    """
    Fit on Rstim/Rresp and score predictions on Pstim/Presp.
    """
    _, pred = ridge_fit_predict_huthstyle(
        Rstim=Rstim,
        Pstim=Pstim,
        Rresp=Rresp,
        alpha=alpha,
        singcutoff=singcutoff,
    )
    if use_corr:
        return columnwise_corr(Presp, pred)
    return columnwise_r2(Presp, pred)


def make_bootstrap_chunks(n_timepoints: int, chunklen: int, nchunks: int, rng: random.Random) -> np.ndarray:
    """
    Select validation chunks without overlap, following the same broad logic as Huth-style chunk CV.
    """
    chunk_starts = list(range(0, n_timepoints - chunklen + 1, chunklen))
    if len(chunk_starts) < nchunks:
        raise ValueError(
            f"Cannot draw {nchunks} chunks of length {chunklen} from {n_timepoints} timepoints."
        )
    chosen_starts = rng.sample(chunk_starts, nchunks)
    val_indices = []
    for start in chosen_starts:
        val_indices.extend(range(start, start + chunklen))
    return np.asarray(sorted(set(val_indices)), dtype=int)


def select_alpha_for_pair(
    X_train: np.ndarray,
    Y_train: np.ndarray,
    alphas: np.ndarray,
    nboots: int,
    chunklen: int,
    nchunks: int,
    use_corr: bool,
    single_alpha: bool,
    seed: int,
) -> dict:
    """
    Select ridge alpha for one network x layer pair.
    """
    rng = random.Random(seed)
    n_time = int(X_train.shape[0])
    n_voxels = int(Y_train.shape[1])
    n_alphas = int(len(alphas))

    bootstrap_scores = np.zeros((nboots, n_alphas, n_voxels), dtype=np.float32)

    for boot in range(nboots):
        val_idx = make_bootstrap_chunks(n_time, chunklen, nchunks, rng)
        train_mask = np.ones(n_time, dtype=bool)
        train_mask[val_idx] = False
        train_idx = np.where(train_mask)[0]

        X_boot_train = X_train[train_idx]
        Y_boot_train = Y_train[train_idx]
        X_boot_val = X_train[val_idx]
        Y_boot_val = Y_train[val_idx]

        for alpha_i, alpha in enumerate(alphas):
            score = ridge_score_huthstyle(
                Rstim=X_boot_train,
                Pstim=X_boot_val,
                Rresp=Y_boot_train,
                Presp=Y_boot_val,
                alpha=float(alpha),
                use_corr=use_corr,
                singcutoff=SINGCUTOFF,
            )
            bootstrap_scores[boot, alpha_i, :] = score

    mean_scores_by_alpha_voxel = np.nanmean(bootstrap_scores, axis=0)
    mean_scores_by_alpha = np.nanmean(mean_scores_by_alpha_voxel, axis=1)

    best_alpha_index = int(np.nanargmax(mean_scores_by_alpha))
    best_alpha = float(alphas[best_alpha_index])

    if single_alpha:
        voxel_alpha_indices = np.full(n_voxels, best_alpha_index, dtype=int)
        voxel_alphas = np.full(n_voxels, best_alpha, dtype=np.float32)
    else:
        voxel_alpha_indices = np.nanargmax(mean_scores_by_alpha_voxel, axis=0).astype(int)
        voxel_alphas = alphas[voxel_alpha_indices].astype(np.float32)

    return {
        "best_alpha": best_alpha,
        "best_alpha_index": best_alpha_index,
        "mean_scores_by_alpha": mean_scores_by_alpha.astype(np.float32),
        "mean_scores_by_alpha_voxel": mean_scores_by_alpha_voxel.astype(np.float32),
        "voxel_alpha_indices": voxel_alpha_indices,
        "voxel_alphas": voxel_alphas,
        "bootstrap_scores": bootstrap_scores,
    }


# =========================
# 4. Input validation
# =========================

step5_summary_csv = STEP5_DIR / "csv" / "step5_layerwise_design_summary.csv"
step6_summary_csv = STEP6_DIR / "step6_all_network_response_summary.csv"

for p in [step5_summary_csv, step6_summary_csv]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required summary file: {p}")

step5_summary_df = pd.read_csv(step5_summary_csv)
step6_summary_df = pd.read_csv(step6_summary_csv)

if len(step5_summary_df) != len(LAYER_LABELS):
    raise ValueError(
        f"Expected {len(LAYER_LABELS)} layer rows in Step 5 summary, found {len(step5_summary_df)}."
    )
if len(step6_summary_df) != len(NETWORK_NAMES):
    raise ValueError(
        f"Expected {len(NETWORK_NAMES)} network rows in Step 6 summary, found {len(step6_summary_df)}."
    )
if not step6_summary_df["alignment_ok"].all():
    raise ValueError("At least one Step 6 network failed alignment checks.")

print("\nInput summaries:")
print(f"Step 5 layer rows: {len(step5_summary_df)}")
print(f"Step 6 network rows: {len(step6_summary_df)}")
print("Step 6 alignment: all OK")


# =========================
# 5. Run alpha selection
# =========================

start_time = time.time()
summary_rows = []

pair_list = list(itools.product(NETWORK_NAMES, LAYER_LABELS))
iterator = pair_list
if tqdm is not None:
    iterator = tqdm(pair_list, desc="Step 7A alpha selection: network x layer")

for pair_index, (roi_name, layer_label) in enumerate(iterator, start=1):
    pair_seed = SEED + pair_index

    layer_step5_dir = STEP5_DIR / layer_label
    X_train_path = layer_step5_dir / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
    X_test_path = layer_step5_dir / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

    step6_network_dir = STEP6_DIR / roi_name
    Y_train_path = step6_network_dir / "npy" / "Y_train_group_huthstyle.npy"
    Y_test_path = step6_network_dir / "npy" / "Y_test_group_huthstyle.npy"

    for p in [X_train_path, X_test_path, Y_train_path, Y_test_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing input for {roi_name} {layer_label}: {p}")

    X_train = np.load(X_train_path).astype(np.float32)
    X_test = np.load(X_test_path).astype(np.float32)
    Y_train = np.load(Y_train_path).astype(np.float32)
    Y_test = np.load(Y_test_path).astype(np.float32)

    if X_train.shape[0] != Y_train.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: train time mismatch: X={X_train.shape}, Y={Y_train.shape}"
        )
    if X_test.shape[0] != Y_test.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: test time mismatch: X={X_test.shape}, Y={Y_test.shape}"
        )
    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in X.")
    if not np.isfinite(Y_train).all() or not np.isfinite(Y_test).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in Y.")

    result = select_alpha_for_pair(
        X_train=X_train,
        Y_train=Y_train,
        alphas=ALPHAS,
        nboots=NBOOTS,
        chunklen=CHUNKLEN,
        nchunks=NCHUNKS,
        use_corr=USE_CORR,
        single_alpha=SINGLE_ALPHA,
        seed=pair_seed,
    )

    # Held-out test sanity check using the selected shared alpha.
    _, test_pred = ridge_fit_predict_huthstyle(
        Rstim=X_train,
        Pstim=X_test,
        Rresp=Y_train,
        alpha=result["best_alpha"],
        singcutoff=SINGCUTOFF,
    )
    test_corrs = columnwise_corr(Y_test, test_pred)

    roi_layer_dir = STEP7A_DIR / roi_name / layer_label
    roi_layer_npy_dir = roi_layer_dir / "npy"
    roi_layer_json_dir = roi_layer_dir / "json"
    for d in [roi_layer_dir, roi_layer_npy_dir, roi_layer_json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    mean_scores_path = roi_layer_npy_dir / "mean_validation_scores_by_alpha.npy"
    voxel_scores_path = roi_layer_npy_dir / "mean_validation_scores_by_alpha_voxel.npy"
    voxel_alphas_path = roi_layer_npy_dir / "selected_voxel_alphas.npy"
    test_corrs_path = roi_layer_npy_dir / "heldout_test_corrs_raw_group.npy"

    np.save(mean_scores_path, result["mean_scores_by_alpha"])
    np.save(voxel_scores_path, result["mean_scores_by_alpha_voxel"])
    np.save(voxel_alphas_path, result["voxel_alphas"])
    np.save(test_corrs_path, test_corrs.astype(np.float32))

    best_validation_score = float(result["mean_scores_by_alpha"][result["best_alpha_index"]])
    row = {
        "roi_name": roi_name,
        "layer_label": layer_label,
        "layer_index": int(layer_label.split("_")[1]),
        "target_type": "raw_group",
        "alpha_selection_scope": "single_alpha_across_voxels",
        "heldout_test_alpha_mode": "single_best_alpha",
        "best_alpha": float(result["best_alpha"]),
        "best_alpha_index": int(result["best_alpha_index"]),
        "best_validation_score": best_validation_score,
        "heldout_test_mean_r": float(np.nanmean(test_corrs)),
        "heldout_test_median_r": float(np.nanmedian(test_corrs)),
        "heldout_test_max_r": float(np.nanmax(test_corrs)),
        "heldout_test_positive_fraction": float(np.mean(test_corrs > 0)),
        "n_train_timepoints": int(X_train.shape[0]),
        "n_test_timepoints": int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
        "n_voxels": int(Y_train.shape[1]),
        "nboots": int(NBOOTS),
        "chunklen": int(CHUNKLEN),
        "nchunks": int(NCHUNKS),
        "use_corr": bool(USE_CORR),
        "single_alpha": bool(SINGLE_ALPHA),
        "mean_scores_path": str(mean_scores_path),
        "voxel_scores_path": str(voxel_scores_path),
        "voxel_alphas_path": str(voxel_alphas_path),
        "test_corrs_path": str(test_corrs_path),
    }
    summary_rows.append(row)

    summary_json = {
        **row,
        "alphas": [float(a) for a in ALPHAS],
        "mean_validation_scores_by_alpha": [float(x) for x in result["mean_scores_by_alpha"]],
        "X_train_path": str(X_train_path),
        "X_test_path": str(X_test_path),
        "Y_train_path": str(Y_train_path),
        "Y_test_path": str(Y_test_path),
    }
    pair_json_path = roi_layer_json_dir / "step7a_alpha_selection_summary.json"
    with open(pair_json_path, "w", encoding="utf-8") as f:
        json.dump(summary_json, f, indent=2)

    if pair_index == 1 or pair_index % 25 == 0:
        elapsed_min = (time.time() - start_time) / 60.0
        print(
            f"[{pair_index:03d}/{len(pair_list):03d}] {roi_name} {layer_label} | "
            f"alpha={fmt_float(result['best_alpha'], 5)} | "
            f"val={fmt_float(best_validation_score, 6)} | "
            f"test mean r={fmt_float(np.nanmean(test_corrs), 6)} | "
            f"elapsed={fmt_float(elapsed_min, 2)} min"
        )


# =========================
# 6. Save combined summaries and overview figures
# =========================

alpha_summary_df = pd.DataFrame(summary_rows)
alpha_summary_csv = CSV_DIR / "step7a_layerwise_alpha_selection_summary.csv"
alpha_summary_df.to_csv(
    alpha_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

best_by_network_df = (
    alpha_summary_df.sort_values(["roi_name", "heldout_test_mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("heldout_test_mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step7a_best_layer_by_network_raw_group.csv"
best_by_network_df.to_csv(
    best_by_network_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

pivot_test = alpha_summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="heldout_test_mean_r",
).loc[NETWORK_NAMES]

plt.figure(figsize=(15, 6))
plt.imshow(pivot_test.values, aspect="auto", cmap="viridis")
plt.colorbar(label="Held-out mean encoding r")
plt.yticks(np.arange(len(pivot_test.index)), pivot_test.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Network")
plt.title("Layer-wise raw group encoding sanity check after alpha selection")
plt.tight_layout()
heatmap_path = FIG_DIR / "step7a_layerwise_heldout_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=240)
plt.close()

plt.figure(figsize=(15, 6))
alpha_pivot = alpha_summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="best_alpha",
).loc[NETWORK_NAMES]
plt.imshow(np.log10(alpha_pivot.values), aspect="auto", cmap="magma")
plt.colorbar(label="log10(selected alpha)")
plt.yticks(np.arange(len(alpha_pivot.index)), alpha_pivot.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Network")
plt.title("Selected alpha by network and GPT-2 XL layer")
plt.tight_layout()
alpha_heatmap_path = FIG_DIR / "step7a_layerwise_selected_alpha_heatmap.png"
plt.savefig(alpha_heatmap_path, dpi=240)
plt.close()

total_elapsed_min = (time.time() - start_time) / 60.0
summary = {
    "dataset": "21styear",
    "pipeline_type": "layerwise_group_raw_alpha_selection",
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_LABELS)),
    "target_type": "raw_group",
    "alphas": [float(a) for a in ALPHAS],
    "nboots": int(NBOOTS),
    "chunklen": int(CHUNKLEN),
    "nchunks": int(NCHUNKS),
    "use_corr": bool(USE_CORR),
    "single_alpha": bool(SINGLE_ALPHA),
    "singcutoff": float(SINGCUTOFF),
    "seed": int(SEED),
    "input_step5_dir": str(STEP5_DIR),
    "input_step6_dir": str(STEP6_DIR),
    "outputs": {
        "alpha_summary_csv": str(alpha_summary_csv),
        "best_by_network_csv": str(best_by_network_csv),
        "heldout_heatmap": str(heatmap_path),
        "alpha_heatmap": str(alpha_heatmap_path),
    },
    "total_elapsed_minutes": float(total_elapsed_min),
}

summary_json_path = JSON_DIR / "step7a_layerwise_alpha_selection_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("STEP 7A LAYER-WISE ALPHA SELECTION SUMMARY")
print("=" * 80)
print(f"Saved alpha summary: {alpha_summary_csv}")
print(f"Saved best layer by network: {best_by_network_csv}")
print(f"Saved held-out heatmap: {heatmap_path}")
print(f"Saved alpha heatmap: {alpha_heatmap_path}")
print(f"Saved JSON summary: {summary_json_path}")
print(f"Total elapsed: {fmt_float(total_elapsed_min, 2)} min")
print("\nBest layer by network:")
print(best_by_network_df[[
    "roi_name",
    "layer_label",
    "best_alpha",
    "best_validation_score",
    "heldout_test_mean_r",
    "heldout_test_median_r",
    "heldout_test_positive_fraction",
]].to_string(index=False))

print("\nStep 7A completed successfully.")


We prioritized cross-validated correlation as the main model-selection criterion, while monitoring the predicted-to-observed variance ratio to avoid overly shrunk solutions with implausibly flat predicted time courses.

# Step 7B: Group-Level Raw Encoding For Each Network And GPT-2 Layer

This step fits the final raw anatomical group-level encoding model for every `network x GPT-2 layer` pair.

For each pair, the code reads the delayed layer-wise GPT-2 design matrix from Step 5, the raw group-average fMRI response from Step 6, and the shared alpha selected in Step 7A. It then fits the ridge model on the training half and evaluates prediction accuracy on the held-out test half using voxel-wise Pearson correlation.

This is a group-level analysis because the response target is `Y_train_group_huthstyle.npy` / `Y_test_group_huthstyle.npy`, not individual participant response arrays. Subject-level follow-up should be handled separately and described as using either group-selected alpha or separately optimized subject-level alpha.


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP6_DIR = PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = PROJECT_DIR / "step7a_select_alpha_layerwise"
STEP7B_DIR = PROJECT_DIR / "step7b_raw_group_encoding_layerwise"

FIG_DIR = STEP7B_DIR / "figures"
CSV_DIR = STEP7B_DIR / "csv"
NPY_DIR = STEP7B_DIR / "npy"
JSON_DIR = STEP7B_DIR / "json"

for d in [STEP7B_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]
SINGCUTOFF = 1e-10

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Step 5 design directory: {STEP5_DIR}")
print(f"Step 6 response directory: {STEP6_DIR}")
print(f"Step 7A alpha directory: {STEP7A_DIR}")
print(f"Step 7B output directory: {STEP7B_DIR}")
print(f"Number of networks: {len(NETWORK_NAMES)}")
print(f"Number of GPT-2 XL layers including embedding layer: {len(LAYER_LABELS)}")


# =========================
# 2. Huth-style helper functions
# =========================

def fmt_float(x: float, digits: int = 6) -> str:
    """Format decimal values without scientific notation for printed summaries."""
    return f"{float(x):.{digits}f}"


def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def columnwise_corr(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise Pearson r using Huth-style z-scoring.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return np.nan_to_num((zs(y_true) * zs(y_pred)).mean(axis=0)).astype(np.float32)


def ridge_fit_predict_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    alpha: float,
    singcutoff: float = 1e-10,
):
    """
    Huth-style SVD ridge fit + predict for one shared alpha.
    """
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))

    U = U[:, :ngoodS]
    S = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = np.dot(U.T, np.nan_to_num(Rresp))
    wt = Vh.T.dot(np.diag(S / (S**2 + float(alpha) ** 2))).dot(UR)
    pred = np.dot(Pstim, wt).astype(np.float32)

    return wt.astype(np.float32), pred


# =========================
# 3. Load Step 7A alpha summary
# =========================

alpha_summary_csv = STEP7A_DIR / "csv" / "step7a_layerwise_alpha_selection_summary.csv"
step5_summary_csv = STEP5_DIR / "csv" / "step5_layerwise_design_summary.csv"
step6_summary_csv = STEP6_DIR / "step6_all_network_response_summary.csv"

for p in [alpha_summary_csv, step5_summary_csv, step6_summary_csv]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input summary: {p}")

alpha_summary_df = pd.read_csv(alpha_summary_csv)
step5_summary_df = pd.read_csv(step5_summary_csv)
step6_summary_df = pd.read_csv(step6_summary_csv)

expected_pairs = len(NETWORK_NAMES) * len(LAYER_LABELS)
if len(alpha_summary_df) != expected_pairs:
    raise ValueError(
        f"Expected {expected_pairs} alpha rows, found {len(alpha_summary_df)} in {alpha_summary_csv}."
    )
if alpha_summary_df[["roi_name", "layer_label", "best_alpha"]].isna().any().any():
    raise ValueError("Step 7A alpha summary contains missing roi/layer/alpha values.")
if not step6_summary_df["alignment_ok"].all():
    raise ValueError("At least one Step 6 network failed alignment checks.")

alpha_lookup = {
    (row.roi_name, row.layer_label): float(row.best_alpha)
    for row in alpha_summary_df.itertuples(index=False)
}

print("\nLoaded Step 7A alpha summary:")
print(f"Rows: {len(alpha_summary_df)}")
print(f"Alpha mode examples:")
print(alpha_summary_df[["roi_name", "layer_label", "best_alpha", "heldout_test_mean_r"]].head().to_string(index=False))


# =========================
# 4. Run final raw group-level encoding
# =========================

start_total = time.time()
summary_rows = []

pair_list = [(roi_name, layer_label) for roi_name in NETWORK_NAMES for layer_label in LAYER_LABELS]
iterator = pair_list
if tqdm is not None:
    iterator = tqdm(pair_list, desc="Step 7B raw group encoding: network x layer")

for pair_index, (roi_name, layer_label) in enumerate(iterator, start=1):
    layer_index = int(layer_label.split("_")[1])
    roi_target = f"{roi_name}_only"
    best_alpha = alpha_lookup[(roi_name, layer_label)]

    X_train_path = STEP5_DIR / layer_label / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
    X_test_path = STEP5_DIR / layer_label / "npy" / "X_test_delayed_lanczos_huthstyle.npy"
    Y_train_path = STEP6_DIR / roi_name / "npy" / "Y_train_group_huthstyle.npy"
    Y_test_path = STEP6_DIR / roi_name / "npy" / "Y_test_group_huthstyle.npy"

    for p in [X_train_path, X_test_path, Y_train_path, Y_test_path]:
        if not p.exists():
            raise FileNotFoundError(f"{roi_name} {layer_label}: missing required input: {p}")

    X_train = np.load(X_train_path).astype(np.float32)
    X_test = np.load(X_test_path).astype(np.float32)
    Y_train_group = np.load(Y_train_path).astype(np.float32)
    Y_test_group = np.load(Y_test_path).astype(np.float32)

    if X_train.shape[0] != Y_train_group.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: train time mismatch: X={X_train.shape}, Y={Y_train_group.shape}"
        )
    if X_test.shape[0] != Y_test_group.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: test time mismatch: X={X_test.shape}, Y={Y_test_group.shape}"
        )
    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in X.")
    if not np.isfinite(Y_train_group).all() or not np.isfinite(Y_test_group).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in Y.")

    weights, Y_pred_raw = ridge_fit_predict_huthstyle(
        Rstim=X_train,
        Pstim=X_test,
        Rresp=Y_train_group,
        alpha=best_alpha,
        singcutoff=SINGCUTOFF,
    )
    voxel_corrs_raw = columnwise_corr(Y_test_group, Y_pred_raw).astype(np.float32)

    layer_out_dir = STEP7B_DIR / roi_name / layer_label
    layer_npy_dir = layer_out_dir / "npy"
    layer_json_dir = layer_out_dir / "json"
    for d in [layer_out_dir, layer_npy_dir, layer_json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    pred_path = layer_npy_dir / "group_Y_pred_raw.npy"
    corr_path = layer_npy_dir / "group_voxel_corrs_raw.npy"
    weights_path = layer_npy_dir / "group_weights_raw.npy"

    np.save(pred_path, Y_pred_raw.astype(np.float32))
    np.save(corr_path, voxel_corrs_raw)
    np.save(weights_path, weights.astype(np.float32))

    mean_r = float(np.nanmean(voxel_corrs_raw))
    median_r = float(np.nanmedian(voxel_corrs_raw))
    max_r = float(np.nanmax(voxel_corrs_raw))
    positive_fraction = float(np.mean(voxel_corrs_raw > 0))

    row = {
        "roi_name": roi_name,
        "roi_target": roi_target,
        "layer_label": layer_label,
        "layer_index": layer_index,
        "target_type": "raw_group",
        "analysis_level": "group",
        "best_alpha": float(best_alpha),
        "mean_r": mean_r,
        "median_r": median_r,
        "max_r": max_r,
        "positive_voxel_fraction": positive_fraction,
        "n_train_timepoints": int(X_train.shape[0]),
        "n_test_timepoints": int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
        "n_voxels": int(Y_train_group.shape[1]),
        "X_train_path": str(X_train_path),
        "X_test_path": str(X_test_path),
        "Y_train_path": str(Y_train_path),
        "Y_test_path": str(Y_test_path),
        "prediction_path": str(pred_path),
        "voxel_corrs_path": str(corr_path),
        "weights_path": str(weights_path),
    }
    summary_rows.append(row)

    pair_json_path = layer_json_dir / "step7b_raw_group_encoding_summary.json"
    with open(pair_json_path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2)

    if pair_index == 1 or pair_index % 50 == 0:
        elapsed_min = (time.time() - start_total) / 60.0
        print(
            f"[{pair_index:03d}/{len(pair_list):03d}] {roi_name} {layer_label} | "
            f"alpha={fmt_float(best_alpha, 5)} | "
            f"mean r={fmt_float(mean_r, 6)} | "
            f"median r={fmt_float(median_r, 6)} | "
            f"elapsed={fmt_float(elapsed_min, 2)} min"
        )


# =========================
# 5. Save combined summaries and figures
# =========================

summary_df = pd.DataFrame(summary_rows)
summary_csv = CSV_DIR / "step7b_layerwise_raw_group_encoding_summary.csv"
summary_df.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

best_by_network_df = (
    summary_df.sort_values(["roi_name", "mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step7b_best_layer_by_network_raw_group.csv"
best_by_network_df.to_csv(
    best_by_network_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

pivot_mean = summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="mean_r",
).loc[NETWORK_NAMES]

plt.figure(figsize=(15, 6))
plt.imshow(pivot_mean.values, aspect="auto", cmap="viridis")
plt.colorbar(label="Mean voxel-wise prediction r")
plt.yticks(np.arange(len(pivot_mean.index)), pivot_mean.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Network")
plt.title("Raw group-level GPT-2 layer-wise encoding performance")
plt.tight_layout()
heatmap_path = FIG_DIR / "step7b_raw_group_layerwise_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=240)
plt.close()

plt.figure(figsize=(12, 7))
for roi_name in NETWORK_NAMES:
    sub = summary_df.loc[summary_df["roi_name"] == roi_name].sort_values("layer_index")
    plt.plot(sub["layer_index"], sub["mean_r"], linewidth=1.5, alpha=0.85, label=roi_name)
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Mean voxel-wise prediction r")
plt.title("Raw group-level layer profiles by network")
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
plt.tight_layout()
profile_path = FIG_DIR / "step7b_raw_group_layer_profiles.png"
plt.savefig(profile_path, dpi=240)
plt.close()

plt.figure(figsize=(8, 7))
plot_df = best_by_network_df.sort_values("mean_r", ascending=True)
plt.barh(plot_df["roi_name"], plot_df["mean_r"], color="#4C78A8", alpha=0.88)
for y, (_, row) in enumerate(plot_df.iterrows()):
    plt.text(
        row["mean_r"] + 0.002,
        y,
        row["layer_label"],
        va="center",
        fontsize=8,
    )
plt.xlabel("Best mean voxel-wise prediction r")
plt.ylabel("Network")
plt.title("Best raw group-level GPT-2 layer by network")
plt.tight_layout()
best_bar_path = FIG_DIR / "step7b_best_raw_group_layer_by_network.png"
plt.savefig(best_bar_path, dpi=240)
plt.close()

total_elapsed_min = (time.time() - start_total) / 60.0
summary_json = {
    "dataset": "21styear",
    "pipeline_type": "layerwise_raw_group_final_encoding",
    "analysis_level": "group",
    "target_type": "raw_group",
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_LABELS)),
    "singcutoff": float(SINGCUTOFF),
    "input_step5_dir": str(STEP5_DIR),
    "input_step6_dir": str(STEP6_DIR),
    "input_step7a_dir": str(STEP7A_DIR),
    "outputs": {
        "summary_csv": str(summary_csv),
        "best_by_network_csv": str(best_by_network_csv),
        "mean_r_heatmap": str(heatmap_path),
        "layer_profiles": str(profile_path),
        "best_layer_barplot": str(best_bar_path),
    },
    "total_elapsed_minutes": float(total_elapsed_min),
}

summary_json_path = JSON_DIR / "step7b_layerwise_raw_group_encoding_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("\n" + "=" * 80)
print("STEP 7B RAW GROUP-LEVEL LAYER-WISE ENCODING SUMMARY")
print("=" * 80)
print(f"Saved summary CSV: {summary_csv}")
print(f"Saved best-by-network CSV: {best_by_network_csv}")
print(f"Saved heatmap: {heatmap_path}")
print(f"Saved layer profiles: {profile_path}")
print(f"Saved best-layer barplot: {best_bar_path}")
print(f"Saved JSON summary: {summary_json_path}")
print(f"Total elapsed: {fmt_float(total_elapsed_min, 2)} min")
print("\nBest raw group-level layer by network:")
print(best_by_network_df[[
    "roi_name",
    "layer_label",
    "best_alpha",
    "mean_r",
    "median_r",
    "max_r",
    "positive_voxel_fraction",
]].to_string(index=False))

if summary_df[["mean_r", "median_r", "max_r"]].isna().any().any():
    raise ValueError("NaN detected in Step 7B encoding summary.")

print("\nStep 7B completed successfully.")


# Step 7C: Participant-Level Raw Encoding Follow-up For Each Network And GPT-2 Layer

This step runs a participant-level follow-up for raw anatomical encoding. For each `network x GPT-2 layer` pair, the same delayed GPT-2 design matrix is used to fit a separate encoding model for each participant's raw response.

The alpha is not re-selected per participant in this cell. Instead, it reuses the group-selected shared alpha from Step 7A for the corresponding `network x layer` pair. This keeps the participant-level follow-up aligned with the group-level analysis while avoiding an extremely expensive subject-specific hyperparameter search.

By default, this cell saves subject-level voxel-wise prediction correlations and subject summary tables, but it does not save every full prediction matrix or weight matrix. Saving all full predictions for `17 networks x 49 layers x 25 participants` would be very large and is not needed for the main participant-level summary.


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP6_DIR = PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = PROJECT_DIR / "step7a_select_alpha_layerwise"
STEP7C_DIR = PROJECT_DIR / "step7c_raw_subject_encoding_layerwise"

FIG_DIR = STEP7C_DIR / "figures"
CSV_DIR = STEP7C_DIR / "csv"
NPY_DIR = STEP7C_DIR / "npy"
JSON_DIR = STEP7C_DIR / "json"

for d in [STEP7C_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]
SINGCUTOFF = 1e-10

# Keep this False unless you specifically need the full prediction matrices.
# Full predictions across all network x layer x participant combinations are very large.
SAVE_FULL_SUBJECT_PREDICTIONS = False

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Step 5 design directory: {STEP5_DIR}")
print(f"Step 6 response directory: {STEP6_DIR}")
print(f"Step 7A alpha directory: {STEP7A_DIR}")
print(f"Step 7C output directory: {STEP7C_DIR}")
print(f"Number of networks: {len(NETWORK_NAMES)}")
print(f"Number of GPT-2 XL layers including embedding layer: {len(LAYER_LABELS)}")
print(f"Save full subject predictions: {SAVE_FULL_SUBJECT_PREDICTIONS}")


# =========================
# 2. Huth-style helper functions
# =========================

def fmt_float(x: float, digits: int = 6) -> str:
    """Format decimal values without scientific notation for printed summaries."""
    return f"{float(x):.{digits}f}"


def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def columnwise_corr(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise Pearson r using Huth-style z-scoring.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return np.nan_to_num((zs(y_true) * zs(y_pred)).mean(axis=0)).astype(np.float32)


def ridge_fit_predict_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    alpha: float,
    singcutoff: float = 1e-10,
):
    """
    Huth-style SVD ridge fit + predict for one shared alpha.
    """
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))

    U = U[:, :ngoodS]
    S = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = np.dot(U.T, np.nan_to_num(Rresp))
    wt = Vh.T.dot(np.diag(S / (S**2 + float(alpha) ** 2))).dot(UR)
    pred = np.dot(Pstim, wt).astype(np.float32)

    return wt.astype(np.float32), pred


def load_subject_response_list(path: Path) -> list:
    """
    Load subject-level response arrays saved by Step 6.

    Each subject response should be time x voxels.
    """
    arr = np.load(path, allow_pickle=True)
    if arr.dtype == object:
        subjects = [np.asarray(arr[i], dtype=np.float32) for i in range(arr.shape[0])]
    else:
        subjects = [np.asarray(arr[i], dtype=np.float32) for i in range(arr.shape[0])]
    return subjects


# =========================
# 3. Load Step 7A alpha summary
# =========================

alpha_summary_csv = STEP7A_DIR / "csv" / "step7a_layerwise_alpha_selection_summary.csv"
step6_summary_csv = STEP6_DIR / "step6_all_network_response_summary.csv"

for p in [alpha_summary_csv, step6_summary_csv]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input summary: {p}")

alpha_summary_df = pd.read_csv(alpha_summary_csv)
step6_summary_df = pd.read_csv(step6_summary_csv)

expected_pairs = len(NETWORK_NAMES) * len(LAYER_LABELS)
if len(alpha_summary_df) != expected_pairs:
    raise ValueError(
        f"Expected {expected_pairs} alpha rows, found {len(alpha_summary_df)} in {alpha_summary_csv}."
    )
if alpha_summary_df[["roi_name", "layer_label", "best_alpha"]].isna().any().any():
    raise ValueError("Step 7A alpha summary contains missing roi/layer/alpha values.")
if not step6_summary_df["alignment_ok"].all():
    raise ValueError("At least one Step 6 network failed alignment checks.")

alpha_lookup = {
    (row.roi_name, row.layer_label): float(row.best_alpha)
    for row in alpha_summary_df.itertuples(index=False)
}

print("\nLoaded Step 7A alpha summary:")
print(f"Rows: {len(alpha_summary_df)}")
print("Participant-level models will reuse these group-selected shared alphas.")


# =========================
# 4. Run participant-level raw encoding
# =========================

start_total = time.time()
subject_rows = []
pair_rows = []

pair_list = [(roi_name, layer_label) for roi_name in NETWORK_NAMES for layer_label in LAYER_LABELS]
iterator = pair_list
if tqdm is not None:
    iterator = tqdm(pair_list, desc="Step 7C raw participant encoding: network x layer")

for pair_index, (roi_name, layer_label) in enumerate(iterator, start=1):
    layer_index = int(layer_label.split("_")[1])
    roi_target = f"{roi_name}_only"
    best_alpha = alpha_lookup[(roi_name, layer_label)]

    X_train_path = STEP5_DIR / layer_label / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
    X_test_path = STEP5_DIR / layer_label / "npy" / "X_test_delayed_lanczos_huthstyle.npy"
    Y_train_list_path = STEP6_DIR / roi_name / "npy" / "Y_train_list_huthstyle.npy"
    Y_test_list_path = STEP6_DIR / roi_name / "npy" / "Y_test_list_huthstyle.npy"

    for p in [X_train_path, X_test_path, Y_train_list_path, Y_test_list_path]:
        if not p.exists():
            raise FileNotFoundError(f"{roi_name} {layer_label}: missing required input: {p}")

    X_train = np.load(X_train_path).astype(np.float32)
    X_test = np.load(X_test_path).astype(np.float32)
    Y_train_subjects = load_subject_response_list(Y_train_list_path)
    Y_test_subjects = load_subject_response_list(Y_test_list_path)

    if len(Y_train_subjects) != len(Y_test_subjects):
        raise ValueError(
            f"{roi_name} {layer_label}: train/test subject count mismatch "
            f"({len(Y_train_subjects)} vs {len(Y_test_subjects)})."
        )
    if len(Y_train_subjects) == 0:
        raise ValueError(f"{roi_name} {layer_label}: no subject responses loaded.")

    n_subjects = len(Y_train_subjects)
    n_voxels = int(Y_train_subjects[0].shape[1])

    subject_corrs = []
    subject_pred_paths = []

    layer_out_dir = STEP7C_DIR / roi_name / layer_label
    layer_npy_dir = layer_out_dir / "npy"
    layer_csv_dir = layer_out_dir / "csv"
    layer_json_dir = layer_out_dir / "json"
    for d in [layer_out_dir, layer_npy_dir, layer_csv_dir, layer_json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    for subject_index, (Y_train_subj, Y_test_subj) in enumerate(zip(Y_train_subjects, Y_test_subjects)):
        Y_train_subj = np.asarray(Y_train_subj, dtype=np.float32)
        Y_test_subj = np.asarray(Y_test_subj, dtype=np.float32)

        if X_train.shape[0] != Y_train_subj.shape[0]:
            raise ValueError(
                f"{roi_name} {layer_label} subject {subject_index}: train time mismatch "
                f"X={X_train.shape}, Y={Y_train_subj.shape}"
            )
        if X_test.shape[0] != Y_test_subj.shape[0]:
            raise ValueError(
                f"{roi_name} {layer_label} subject {subject_index}: test time mismatch "
                f"X={X_test.shape}, Y={Y_test_subj.shape}"
            )
        if Y_train_subj.shape[1] != n_voxels or Y_test_subj.shape[1] != n_voxels:
            raise ValueError(f"{roi_name} {layer_label}: inconsistent voxel count across subjects.")
        if not np.isfinite(Y_train_subj).all() or not np.isfinite(Y_test_subj).all():
            raise ValueError(f"{roi_name} {layer_label} subject {subject_index}: non-finite Y values.")

        _, Y_pred_subj = ridge_fit_predict_huthstyle(
            Rstim=X_train,
            Pstim=X_test,
            Rresp=Y_train_subj,
            alpha=best_alpha,
            singcutoff=SINGCUTOFF,
        )
        voxel_corrs_subj = columnwise_corr(Y_test_subj, Y_pred_subj).astype(np.float32)
        subject_corrs.append(voxel_corrs_subj)

        pred_path = ""
        if SAVE_FULL_SUBJECT_PREDICTIONS:
            pred_path_obj = layer_npy_dir / f"subject_{subject_index:02d}_Y_pred_raw.npy"
            np.save(pred_path_obj, Y_pred_subj.astype(np.float32))
            pred_path = str(pred_path_obj)
        subject_pred_paths.append(pred_path)

        subject_rows.append({
            "roi_name": roi_name,
            "roi_target": roi_target,
            "layer_label": layer_label,
            "layer_index": layer_index,
            "subject_index": int(subject_index),
            "target_type": "raw_subject",
            "analysis_level": "participant",
            "alpha_source": "group_selected_shared_alpha_from_step7a",
            "best_alpha": float(best_alpha),
            "mean_r": float(np.nanmean(voxel_corrs_subj)),
            "median_r": float(np.nanmedian(voxel_corrs_subj)),
            "max_r": float(np.nanmax(voxel_corrs_subj)),
            "positive_voxel_fraction": float(np.mean(voxel_corrs_subj > 0)),
            "n_train_timepoints": int(X_train.shape[0]),
            "n_test_timepoints": int(X_test.shape[0]),
            "n_features": int(X_train.shape[1]),
            "n_voxels": int(n_voxels),
            "prediction_path": pred_path,
        })

    subject_corrs = np.stack(subject_corrs, axis=0).astype(np.float32)
    mean_r_across_subjects_voxelwise = np.nanmean(subject_corrs, axis=0).astype(np.float32)

    subject_corrs_path = layer_npy_dir / "subject_voxel_corrs_raw.npy"
    mean_voxel_corrs_path = layer_npy_dir / "mean_r_across_subjects_raw.npy"
    np.save(subject_corrs_path, subject_corrs)
    np.save(mean_voxel_corrs_path, mean_r_across_subjects_voxelwise)

    pair_subject_df = pd.DataFrame([row for row in subject_rows if row["roi_name"] == roi_name and row["layer_label"] == layer_label])
    pair_subject_csv = layer_csv_dir / "subject_level_raw_encoding_summary.csv"
    pair_subject_df.to_csv(
        pair_subject_csv,
        index=False,
        encoding="utf-8-sig",
        float_format="%.6f",
    )

    pair_row = {
        "roi_name": roi_name,
        "roi_target": roi_target,
        "layer_label": layer_label,
        "layer_index": layer_index,
        "target_type": "raw_subject",
        "analysis_level": "participant",
        "alpha_source": "group_selected_shared_alpha_from_step7a",
        "best_alpha": float(best_alpha),
        "n_subjects": int(n_subjects),
        "mean_subject_mean_r": float(pair_subject_df["mean_r"].mean()),
        "median_subject_mean_r": float(pair_subject_df["mean_r"].median()),
        "min_subject_mean_r": float(pair_subject_df["mean_r"].min()),
        "max_subject_mean_r": float(pair_subject_df["mean_r"].max()),
        "positive_subject_fraction": float(np.mean(pair_subject_df["mean_r"].values > 0)),
        "mean_voxel_mean_r_across_subjects": float(np.nanmean(mean_r_across_subjects_voxelwise)),
        "median_voxel_mean_r_across_subjects": float(np.nanmedian(mean_r_across_subjects_voxelwise)),
        "n_train_timepoints": int(X_train.shape[0]),
        "n_test_timepoints": int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
        "n_voxels": int(n_voxels),
        "subject_voxel_corrs_path": str(subject_corrs_path),
        "mean_voxel_corrs_path": str(mean_voxel_corrs_path),
        "subject_summary_csv": str(pair_subject_csv),
    }
    pair_rows.append(pair_row)

    pair_json_path = layer_json_dir / "step7c_raw_subject_encoding_summary.json"
    with open(pair_json_path, "w", encoding="utf-8") as f:
        json.dump(pair_row, f, indent=2)

    if pair_index == 1 or pair_index % 25 == 0:
        elapsed_min = (time.time() - start_total) / 60.0
        print(
            f"[{pair_index:03d}/{len(pair_list):03d}] {roi_name} {layer_label} | "
            f"alpha={fmt_float(best_alpha, 5)} | "
            f"mean subject r={fmt_float(pair_row['mean_subject_mean_r'], 6)} | "
            f"positive subjects={fmt_float(pair_row['positive_subject_fraction'], 3)} | "
            f"elapsed={fmt_float(elapsed_min, 2)} min"
        )


# =========================
# 5. Save combined summaries and figures
# =========================

subject_summary_df = pd.DataFrame(subject_rows)
pair_summary_df = pd.DataFrame(pair_rows)

subject_summary_csv = CSV_DIR / "step7c_raw_subject_encoding_subject_summary.csv"
pair_summary_csv = CSV_DIR / "step7c_raw_subject_encoding_pair_summary.csv"

subject_summary_df.to_csv(
    subject_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)
pair_summary_df.to_csv(
    pair_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

best_by_network_df = (
    pair_summary_df.sort_values(["roi_name", "mean_subject_mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("mean_subject_mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step7c_best_layer_by_network_raw_subject.csv"
best_by_network_df.to_csv(
    best_by_network_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

pivot_subject = pair_summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="mean_subject_mean_r",
).loc[NETWORK_NAMES]

plt.figure(figsize=(15, 6))
plt.imshow(pivot_subject.values, aspect="auto", cmap="viridis")
plt.colorbar(label="Mean participant-level prediction r")
plt.yticks(np.arange(len(pivot_subject.index)), pivot_subject.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Network")
plt.title("Raw participant-level GPT-2 layer-wise encoding performance")
plt.tight_layout()
heatmap_path = FIG_DIR / "step7c_raw_subject_layerwise_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=240)
plt.close()

plt.figure(figsize=(12, 7))
for roi_name in NETWORK_NAMES:
    sub = pair_summary_df.loc[pair_summary_df["roi_name"] == roi_name].sort_values("layer_index")
    plt.plot(sub["layer_index"], sub["mean_subject_mean_r"], linewidth=1.5, alpha=0.85, label=roi_name)
plt.xlabel("GPT-2 XL layer index")
plt.ylabel("Mean participant-level prediction r")
plt.title("Raw participant-level layer profiles by network")
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
plt.tight_layout()
profile_path = FIG_DIR / "step7c_raw_subject_layer_profiles.png"
plt.savefig(profile_path, dpi=240)
plt.close()

plt.figure(figsize=(8, 7))
plot_df = best_by_network_df.sort_values("mean_subject_mean_r", ascending=True)
plt.barh(plot_df["roi_name"], plot_df["mean_subject_mean_r"], color="#59A14F", alpha=0.88)
for y, (_, row) in enumerate(plot_df.iterrows()):
    plt.text(
        row["mean_subject_mean_r"] + 0.001,
        y,
        row["layer_label"],
        va="center",
        fontsize=8,
    )
plt.xlabel("Best mean participant-level prediction r")
plt.ylabel("Network")
plt.title("Best raw participant-level GPT-2 layer by network")
plt.tight_layout()
best_bar_path = FIG_DIR / "step7c_best_raw_subject_layer_by_network.png"
plt.savefig(best_bar_path, dpi=240)
plt.close()

total_elapsed_min = (time.time() - start_total) / 60.0
summary_json = {
    "dataset": "21styear",
    "pipeline_type": "layerwise_raw_subject_followup_encoding",
    "analysis_level": "participant",
    "target_type": "raw_subject",
    "alpha_source": "group_selected_shared_alpha_from_step7a",
    "save_full_subject_predictions": bool(SAVE_FULL_SUBJECT_PREDICTIONS),
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_LABELS)),
    "n_pairs": int(len(pair_summary_df)),
    "n_subject_rows": int(len(subject_summary_df)),
    "singcutoff": float(SINGCUTOFF),
    "input_step5_dir": str(STEP5_DIR),
    "input_step6_dir": str(STEP6_DIR),
    "input_step7a_dir": str(STEP7A_DIR),
    "outputs": {
        "subject_summary_csv": str(subject_summary_csv),
        "pair_summary_csv": str(pair_summary_csv),
        "best_by_network_csv": str(best_by_network_csv),
        "mean_r_heatmap": str(heatmap_path),
        "layer_profiles": str(profile_path),
        "best_layer_barplot": str(best_bar_path),
    },
    "total_elapsed_minutes": float(total_elapsed_min),
}

summary_json_path = JSON_DIR / "step7c_raw_subject_encoding_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("\n" + "=" * 80)
print("STEP 7C RAW PARTICIPANT-LEVEL LAYER-WISE ENCODING SUMMARY")
print("=" * 80)
print(f"Saved subject summary CSV: {subject_summary_csv}")
print(f"Saved pair summary CSV: {pair_summary_csv}")
print(f"Saved best-by-network CSV: {best_by_network_csv}")
print(f"Saved heatmap: {heatmap_path}")
print(f"Saved layer profiles: {profile_path}")
print(f"Saved best-layer barplot: {best_bar_path}")
print(f"Saved JSON summary: {summary_json_path}")
print(f"Total elapsed: {fmt_float(total_elapsed_min, 2)} min")
print("\nBest raw participant-level layer by network:")
print(best_by_network_df[[
    "roi_name",
    "layer_label",
    "best_alpha",
    "mean_subject_mean_r",
    "median_subject_mean_r",
    "positive_subject_fraction",
]].to_string(index=False))

if subject_summary_df[["mean_r", "median_r", "max_r"]].isna().any().any():
    raise ValueError("NaN detected in Step 7C subject-level encoding summary.")
if pair_summary_df[["mean_subject_mean_r", "median_subject_mean_r"]].isna().any().any():
    raise ValueError("NaN detected in Step 7C pair-level encoding summary.")

print("\nStep 7C completed successfully.")


# Step 8A: Prepare SRM-Reconstructed fMRI Responses

This step prepares the SRM-reconstructed neural response targets for the layer-wise encoding analysis.

For each network, the raw participant-level response matrices from Step 6 are projected into the fixed SRM50 shared space and then projected back into voxel space using each participant's SRM mapping matrix: `Y_reconstructed = (Y @ W) @ W.T`. This creates SRM-reconstructed response matrices with the same `time x voxels` shape as the raw responses.

This reconstruction depends only on the network and participant responses, not on GPT-2 layer. Therefore each network is reconstructed once here, and later SRM+GPT-2 encoding steps can reuse these outputs for all layers.


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP6_DIR = PROJECT_DIR / "step6_prepare_fmri_response"
STEP8A_DIR = PROJECT_DIR / "step8a_prepare_srm_reconstructed_response"

SRM_PROJECT_DIR = BASE_DIR / "runs"
SRM_RUN_SUFFIX = "400_parcels_fixed_srm50"

CSV_DIR = STEP8A_DIR / "csv"
JSON_DIR = STEP8A_DIR / "json"

for d in [STEP8A_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_REFERENCE_LABEL = "layer_00"
REFERENCE_STEP5_DIR = STEP5_DIR / LAYER_REFERENCE_LABEL

# Keep this True if later participant-level SRM follow-up may be needed.
# These files are larger than group-level outputs, but they avoid recomputing SRM reconstruction.
SAVE_SUBJECT_RECONSTRUCTED_LISTS = True

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Step 5 design directory: {STEP5_DIR}")
print(f"Step 6 raw response directory: {STEP6_DIR}")
print(f"Step 8A output directory: {STEP8A_DIR}")
print(f"SRM project directory: {SRM_PROJECT_DIR}")
print(f"Reference Step 5 layer: {LAYER_REFERENCE_LABEL}")
print(f"Save subject reconstructed lists: {SAVE_SUBJECT_RECONSTRUCTED_LISTS}")
print(f"Number of networks: {len(NETWORK_NAMES)}")


# =========================
# 2. Shared reference X for time alignment only
# =========================

X_train_reference_path = REFERENCE_STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_test_reference_path = REFERENCE_STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

for p in [X_train_reference_path, X_test_reference_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing reference Step 5 design matrix: {p}")

X_train_reference = np.load(X_train_reference_path, mmap_mode="r")
X_test_reference = np.load(X_test_reference_path, mmap_mode="r")

N_TRAIN_REFERENCE = int(X_train_reference.shape[0])
N_TEST_REFERENCE = int(X_test_reference.shape[0])

print(f"Reference X_train rows: {N_TRAIN_REFERENCE}")
print(f"Reference X_test rows : {N_TEST_REFERENCE}")


# =========================
# 3. Helper functions
# =========================

def fmt_float(x: float, digits: int = 6) -> str:
    """Format decimal values without scientific notation for printed summaries."""
    return f"{float(x):.{digits}f}"


def load_subject_response_list(path: Path) -> list:
    """
    Load subject-level response arrays saved by Step 6.

    Each subject response should be time x voxels.
    """
    arr = np.load(path, allow_pickle=True)
    return [np.asarray(arr[i], dtype=np.float32) for i in range(arr.shape[0])]


def load_weight_matrices(path: Path) -> list:
    """
    Load SRM subject mapping matrices.

    Each W should be voxels x shared_features.
    """
    arr = np.load(path, allow_pickle=True)
    return [np.asarray(arr[i], dtype=np.float32) for i in range(arr.shape[0])]


def reconstruct_srm_voxel_response(Y_time_vox: np.ndarray, W_vox_k: np.ndarray) -> np.ndarray:
    """
    Reconstruct voxel-space response through the SRM shared subspace.

    Y_time_vox: time x voxels
    W_vox_k   : voxels x shared_features

    Equivalent to (Y @ W) @ W.T, avoiding explicit W @ W.T.
    """
    Y64 = np.asarray(Y_time_vox, dtype=np.float64)
    W64 = np.asarray(W_vox_k, dtype=np.float64)
    shared = Y64 @ W64
    recon = shared @ W64.T
    return recon.astype(np.float32)


# =========================
# 4. Prepare one network
# =========================

def prepare_srm_reconstructed_response_for_network(roi_name: str) -> dict:
    roi_target = f"{roi_name}_only"
    run_name = f"{roi_name}_{SRM_RUN_SUFFIX}"

    step6_network_dir = STEP6_DIR / roi_name
    step8a_network_dir = STEP8A_DIR / roi_name

    csv_dir = step8a_network_dir / "csv"
    npy_dir = step8a_network_dir / "npy"
    json_dir = step8a_network_dir / "json"

    for d in [step8a_network_dir, csv_dir, npy_dir, json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    srm_step3b_dir = SRM_PROJECT_DIR / run_name / "step3b_estimate_srm"

    Y_train_list_path = step6_network_dir / "npy" / "Y_train_list_huthstyle.npy"
    Y_test_list_path = step6_network_dir / "npy" / "Y_test_list_huthstyle.npy"
    Y_train_group_raw_path = step6_network_dir / "npy" / "Y_train_group_huthstyle.npy"
    Y_test_group_raw_path = step6_network_dir / "npy" / "Y_test_group_huthstyle.npy"
    W_list_path = srm_step3b_dir / "npy" / "weight_matrices.npy"
    srm_summary_path = srm_step3b_dir / "json" / "step3b_summary.json"

    for p in [Y_train_list_path, Y_test_list_path, Y_train_group_raw_path, Y_test_group_raw_path, W_list_path]:
        if not p.exists():
            raise FileNotFoundError(f"{roi_name}: missing required input: {p}")

    print("\n" + "=" * 100)
    print(f"Step 8A SRM reconstruction | network={roi_name}")
    print("=" * 100)
    print(f"Step 6 raw response directory: {step6_network_dir}")
    print(f"SRM Step 3B directory: {srm_step3b_dir}")
    print(f"Step 8A output directory: {step8a_network_dir}")

    start_time = time.time()

    Y_train_list = load_subject_response_list(Y_train_list_path)
    Y_test_list = load_subject_response_list(Y_test_list_path)
    Y_train_group_raw = np.load(Y_train_group_raw_path).astype(np.float32)
    Y_test_group_raw = np.load(Y_test_group_raw_path).astype(np.float32)
    W_list = load_weight_matrices(W_list_path)

    srm_summary = {}
    if srm_summary_path.exists():
        with open(srm_summary_path, "r", encoding="utf-8") as f:
            srm_summary = json.load(f)

    n_subjects = len(Y_train_list)
    if n_subjects == 0:
        raise ValueError(f"{roi_name}: empty subject response list.")
    if len(Y_train_list) != len(Y_test_list) or len(Y_train_list) != len(W_list):
        raise ValueError(f"{roi_name}: subject count mismatch across train/test/W.")

    n_train_timepoints = int(Y_train_list[0].shape[0])
    n_test_timepoints = int(Y_test_list[0].shape[0])
    n_voxels = int(Y_train_list[0].shape[1])
    n_srm_features = int(W_list[0].shape[1])

    print(f"n_subjects: {n_subjects}")
    print(f"n_voxels: {n_voxels}")
    print(f"SRM shared features K: {n_srm_features}")
    print(f"Y_train subject 0 shape: {Y_train_list[0].shape}")
    print(f"Y_test subject 0 shape: {Y_test_list[0].shape}")
    print(f"W subject 0 shape: {W_list[0].shape}")

    if n_train_timepoints != N_TRAIN_REFERENCE:
        raise ValueError(
            f"{roi_name}: train time mismatch: Y={n_train_timepoints}, reference X={N_TRAIN_REFERENCE}"
        )
    if n_test_timepoints != N_TEST_REFERENCE:
        raise ValueError(
            f"{roi_name}: test time mismatch: Y={n_test_timepoints}, reference X={N_TEST_REFERENCE}"
        )
    if Y_train_group_raw.shape != (n_train_timepoints, n_voxels):
        raise ValueError(f"{roi_name}: raw train group shape mismatch: {Y_train_group_raw.shape}")
    if Y_test_group_raw.shape != (n_test_timepoints, n_voxels):
        raise ValueError(f"{roi_name}: raw test group shape mismatch: {Y_test_group_raw.shape}")
    if not all(y.shape == Y_train_list[0].shape for y in Y_train_list):
        raise ValueError(f"{roi_name}: inconsistent Y_train subject shapes.")
    if not all(y.shape == Y_test_list[0].shape for y in Y_test_list):
        raise ValueError(f"{roi_name}: inconsistent Y_test subject shapes.")
    if not all(w.shape == W_list[0].shape for w in W_list):
        raise ValueError(f"{roi_name}: inconsistent SRM weight matrix shapes.")
    if not all(w.shape[0] == n_voxels for w in W_list):
        raise ValueError(f"{roi_name}: SRM W voxel dimension does not match response voxel count.")
    if not all(np.isfinite(y).all() for y in Y_train_list):
        raise ValueError(f"{roi_name}: non-finite values in Y_train_list.")
    if not all(np.isfinite(y).all() for y in Y_test_list):
        raise ValueError(f"{roi_name}: non-finite values in Y_test_list.")
    if not all(np.isfinite(w).all() for w in W_list):
        raise ValueError(f"{roi_name}: non-finite values in SRM weight matrices.")

    # Check SRM orthonormality for all subjects.
    ortho_rows = []
    for subj_idx, W in enumerate(W_list):
        WtW = W.T.astype(np.float64) @ W.astype(np.float64)
        max_abs_error = float(np.abs(WtW - np.eye(n_srm_features)).max())
        mean_abs_error = float(np.abs(WtW - np.eye(n_srm_features)).mean())
        ortho_rows.append({
            "roi_name": roi_name,
            "subject_index": int(subj_idx),
            "max_abs_orthonormality_error": max_abs_error,
            "mean_abs_orthonormality_error": mean_abs_error,
        })

    ortho_df = pd.DataFrame(ortho_rows)
    ortho_csv_path = csv_dir / "srm_weight_orthonormality_summary.csv"
    ortho_df.to_csv(
        ortho_csv_path,
        index=False,
        encoding="utf-8-sig",
        float_format="%.8f",
    )
    print("SRM W orthonormality preview:")
    print(ortho_df.head(3).to_string(index=False))

    Y_train_recon_list = []
    Y_test_recon_list = []
    recon_rows = []

    for subj_idx, W in enumerate(W_list):
        Y_train_recon = reconstruct_srm_voxel_response(Y_train_list[subj_idx], W)
        Y_test_recon = reconstruct_srm_voxel_response(Y_test_list[subj_idx], W)

        if Y_train_recon.shape != Y_train_list[subj_idx].shape:
            raise ValueError(f"{roi_name} subject {subj_idx}: train reconstruction shape mismatch.")
        if Y_test_recon.shape != Y_test_list[subj_idx].shape:
            raise ValueError(f"{roi_name} subject {subj_idx}: test reconstruction shape mismatch.")
        if not np.isfinite(Y_train_recon).all() or not np.isfinite(Y_test_recon).all():
            raise ValueError(f"{roi_name} subject {subj_idx}: non-finite reconstructed response.")

        Y_train_recon_list.append(Y_train_recon.astype(np.float32))
        Y_test_recon_list.append(Y_test_recon.astype(np.float32))

        train_delta = Y_train_recon - Y_train_list[subj_idx]
        test_delta = Y_test_recon - Y_test_list[subj_idx]
        recon_rows.append({
            "roi_name": roi_name,
            "subject_index": int(subj_idx),
            "train_reconstructed_shape": str(tuple(Y_train_recon.shape)),
            "test_reconstructed_shape": str(tuple(Y_test_recon.shape)),
            "train_raw_rms": float(np.sqrt(np.mean(Y_train_list[subj_idx] ** 2))),
            "train_reconstructed_rms": float(np.sqrt(np.mean(Y_train_recon ** 2))),
            "test_raw_rms": float(np.sqrt(np.mean(Y_test_list[subj_idx] ** 2))),
            "test_reconstructed_rms": float(np.sqrt(np.mean(Y_test_recon ** 2))),
            "train_reconstruction_delta_rms": float(np.sqrt(np.mean(train_delta ** 2))),
            "test_reconstruction_delta_rms": float(np.sqrt(np.mean(test_delta ** 2))),
        })

    Y_train_recon_stack = np.stack(Y_train_recon_list, axis=0).astype(np.float32)
    Y_test_recon_stack = np.stack(Y_test_recon_list, axis=0).astype(np.float32)
    Y_train_group_recon = np.mean(Y_train_recon_stack, axis=0).astype(np.float32)
    Y_test_group_recon = np.mean(Y_test_recon_stack, axis=0).astype(np.float32)

    if Y_train_group_recon.shape != Y_train_group_raw.shape:
        raise ValueError(f"{roi_name}: reconstructed train group shape mismatch.")
    if Y_test_group_recon.shape != Y_test_group_raw.shape:
        raise ValueError(f"{roi_name}: reconstructed test group shape mismatch.")

    Y_train_group_recon_path = npy_dir / "Y_train_group_srm_reconstructed_huthstyle.npy"
    Y_test_group_recon_path = npy_dir / "Y_test_group_srm_reconstructed_huthstyle.npy"
    np.save(Y_train_group_recon_path, Y_train_group_recon)
    np.save(Y_test_group_recon_path, Y_test_group_recon)

    Y_train_list_recon_path = ""
    Y_test_list_recon_path = ""
    if SAVE_SUBJECT_RECONSTRUCTED_LISTS:
        Y_train_list_recon_path_obj = npy_dir / "Y_train_list_srm_reconstructed_huthstyle.npy"
        Y_test_list_recon_path_obj = npy_dir / "Y_test_list_srm_reconstructed_huthstyle.npy"
        np.save(Y_train_list_recon_path_obj, Y_train_recon_stack)
        np.save(Y_test_list_recon_path_obj, Y_test_recon_stack)
        Y_train_list_recon_path = str(Y_train_list_recon_path_obj)
        Y_test_list_recon_path = str(Y_test_list_recon_path_obj)

    recon_df = pd.DataFrame(recon_rows)
    recon_csv_path = csv_dir / "srm_reconstruction_subject_summary.csv"
    recon_df.to_csv(
        recon_csv_path,
        index=False,
        encoding="utf-8-sig",
        float_format="%.6f",
    )

    elapsed_min = (time.time() - start_time) / 60.0
    summary = {
        "dataset": "21styear",
        "roi_name": roi_name,
        "roi_target": roi_target,
        "pipeline_type": "prepare_srm_reconstructed_response_layerwise",
        "source_srm_run": run_name,
        "source_srm_weight_matrices": str(W_list_path),
        "source_srm_summary": str(srm_summary_path),
        "source_step6_dir": str(step6_network_dir),
        "srm_summary": srm_summary,
        "reconstruction_formula": "Y_reconstructed = (Y @ W) @ W.T",
        "n_subjects": int(n_subjects),
        "n_train_timepoints": int(n_train_timepoints),
        "n_test_timepoints": int(n_test_timepoints),
        "n_voxels": int(n_voxels),
        "n_srm_features": int(n_srm_features),
        "Y_train_group_raw_shape": list(Y_train_group_raw.shape),
        "Y_test_group_raw_shape": list(Y_test_group_raw.shape),
        "Y_train_group_srm_reconstructed_shape": list(Y_train_group_recon.shape),
        "Y_test_group_srm_reconstructed_shape": list(Y_test_group_recon.shape),
        "Y_train_recon_stack_shape": list(Y_train_recon_stack.shape),
        "Y_test_recon_stack_shape": list(Y_test_recon_stack.shape),
        "reference_X_train_rows": int(N_TRAIN_REFERENCE),
        "reference_X_test_rows": int(N_TEST_REFERENCE),
        "alignment": {
            "train_ok": bool(Y_train_group_recon.shape[0] == N_TRAIN_REFERENCE),
            "test_ok": bool(Y_test_group_recon.shape[0] == N_TEST_REFERENCE),
        },
        "rms_summary": {
            "train_group_raw_rms": float(np.sqrt(np.mean(Y_train_group_raw ** 2))),
            "train_group_reconstructed_rms": float(np.sqrt(np.mean(Y_train_group_recon ** 2))),
            "test_group_raw_rms": float(np.sqrt(np.mean(Y_test_group_raw ** 2))),
            "test_group_reconstructed_rms": float(np.sqrt(np.mean(Y_test_group_recon ** 2))),
        },
        "outputs": {
            "Y_train_group_srm_reconstructed": str(Y_train_group_recon_path),
            "Y_test_group_srm_reconstructed": str(Y_test_group_recon_path),
            "Y_train_list_srm_reconstructed": Y_train_list_recon_path,
            "Y_test_list_srm_reconstructed": Y_test_list_recon_path,
            "reconstruction_subject_summary_csv": str(recon_csv_path),
            "orthonormality_csv": str(ortho_csv_path),
        },
        "elapsed_minutes": float(elapsed_min),
    }

    summary_json_path = json_dir / "step8a_srm_reconstructed_response_summary.json"
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved group SRM train response: {Y_train_group_recon_path}")
    print(f"Saved group SRM test response : {Y_test_group_recon_path}")
    if SAVE_SUBJECT_RECONSTRUCTED_LISTS:
        print(f"Saved subject SRM train stack: {Y_train_list_recon_path}")
        print(f"Saved subject SRM test stack : {Y_test_list_recon_path}")
    print(f"Saved subject reconstruction summary: {recon_csv_path}")
    print(f"Saved summary JSON: {summary_json_path}")
    print(f"Elapsed for {roi_name}: {fmt_float(elapsed_min, 2)} min")

    if not summary["alignment"]["train_ok"] or not summary["alignment"]["test_ok"]:
        raise ValueError(f"{roi_name}: SRM reconstructed responses do not align to Step 5 reference time axis.")

    return {
        "roi_name": roi_name,
        "roi_target": roi_target,
        "run_name": run_name,
        "step8a_dir": str(step8a_network_dir),
        "n_subjects": int(n_subjects),
        "n_voxels": int(n_voxels),
        "n_srm_features": int(n_srm_features),
        "Y_train_group_srm_reconstructed_shape": str(tuple(Y_train_group_recon.shape)),
        "Y_test_group_srm_reconstructed_shape": str(tuple(Y_test_group_recon.shape)),
        "train_alignment_ok": bool(summary["alignment"]["train_ok"]),
        "test_alignment_ok": bool(summary["alignment"]["test_ok"]),
        "train_group_raw_rms": float(summary["rms_summary"]["train_group_raw_rms"]),
        "train_group_reconstructed_rms": float(summary["rms_summary"]["train_group_reconstructed_rms"]),
        "test_group_raw_rms": float(summary["rms_summary"]["test_group_raw_rms"]),
        "test_group_reconstructed_rms": float(summary["rms_summary"]["test_group_reconstructed_rms"]),
        "summary_json": str(summary_json_path),
    }


# =========================
# 5. Run all networks
# =========================

start_total = time.time()
rows = []

iterator = NETWORK_NAMES
if tqdm is not None:
    iterator = tqdm(NETWORK_NAMES, desc="Step 8A SRM reconstruction by network")

for roi_name in iterator:
    rows.append(prepare_srm_reconstructed_response_for_network(roi_name))

all_network_summary_df = pd.DataFrame(rows)
all_network_summary_csv = CSV_DIR / "step8a_all_network_srm_reconstruction_summary.csv"
all_network_summary_df.to_csv(
    all_network_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

total_elapsed_min = (time.time() - start_total) / 60.0
overall_summary = {
    "dataset": "21styear",
    "pipeline_type": "prepare_srm_reconstructed_response_layerwise_all_networks",
    "n_networks": int(len(NETWORK_NAMES)),
    "save_subject_reconstructed_lists": bool(SAVE_SUBJECT_RECONSTRUCTED_LISTS),
    "input_step6_dir": str(STEP6_DIR),
    "input_srm_project_dir": str(SRM_PROJECT_DIR),
    "output_step8a_dir": str(STEP8A_DIR),
    "all_network_summary_csv": str(all_network_summary_csv),
    "total_elapsed_minutes": float(total_elapsed_min),
}

overall_summary_json = JSON_DIR / "step8a_all_network_srm_reconstruction_summary.json"
with open(overall_summary_json, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)

print("\n" + "=" * 80)
print("STEP 8A SRM-RECONSTRUCTED RESPONSE SUMMARY")
print("=" * 80)
print(all_network_summary_df.to_string(index=False))
print(f"\nSaved all-network summary CSV: {all_network_summary_csv}")
print(f"Saved all-network summary JSON: {overall_summary_json}")
print(f"Total elapsed: {fmt_float(total_elapsed_min, 2)} min")

if not all(all_network_summary_df["train_alignment_ok"]) or not all(all_network_summary_df["test_alignment_ok"]):
    raise ValueError("At least one network failed SRM reconstruction alignment checks.")

print("\nStep 8A completed successfully.")


# Step 8B: Select Group-Level Alpha For SRM-Reconstructed Responses

This step mirrors Step 7A, but the response target is now the SRM-reconstructed group-average fMRI response prepared in Step 8A.

For each `network x GPT-2 layer` pair, the code reads the delayed layer-wise GPT-2 design matrix from Step 5 and the SRM-reconstructed group response from Step 8A. It then evaluates the same log-spaced alpha grid using Huth-style bootstrap chunk validation and saves the selected shared alpha for later SRM+GPT-2 group-level encoding.

This re-selection is intentional. SRM reconstruction changes the target response variance/noise structure, so the most appropriate ridge regularization for `X_layer -> Y_srm_group` does not have to match the raw-response alpha selected in Step 7A.


In [ ]:
from pathlib import Path
import json
import random
import time
import itertools as itools

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"

ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP8A_DIR = PROJECT_DIR / "step8a_prepare_srm_reconstructed_response"
STEP8B_ALPHA_DIR = PROJECT_DIR / "step8b_select_alpha_srm_group_layerwise"

FIG_DIR = STEP8B_ALPHA_DIR / "figures"
CSV_DIR = STEP8B_ALPHA_DIR / "csv"
JSON_DIR = STEP8B_ALPHA_DIR / "json"

for d in [STEP8B_ALPHA_DIR, FIG_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]

print(f"Layer-wise encoding project directory: {PROJECT_DIR}")
print(f"Step 5 layer-wise design directory: {STEP5_DIR}")
print(f"Step 8A SRM response directory: {STEP8A_DIR}")
print(f"Step 8B alpha output directory: {STEP8B_ALPHA_DIR}")
print(f"Number of networks: {len(NETWORK_NAMES)}")
print(f"Number of GPT-2 XL representations: {len(LAYER_LABELS)}")


# =========================
# 2. Huth-style alpha selection settings
# =========================

ALPHAS = np.logspace(1, 5, 20).astype(np.float32)
NBOOTS = 15
CHUNKLEN = 40
NCHUNKS = 6
SINGCUTOFF = 1e-10
USE_CORR = True
SEED = 0
SINGLE_ALPHA = True

print("\nHuth-style alpha-selection parameters:")
print("ALPHAS =", [f"{x:.5f}" for x in ALPHAS])
print("NBOOTS =", NBOOTS)
print("CHUNKLEN =", CHUNKLEN)
print("NCHUNKS =", NCHUNKS)
print("SINGCUTOFF =", f"{SINGCUTOFF:.10f}")
print("USE_CORR =", USE_CORR)
print("SINGLE_ALPHA =", SINGLE_ALPHA)
print("SEED =", SEED)


# =========================
# 3. Helper functions
# =========================

def fmt_float(x: float, digits: int = 6) -> str:
    """Format values without scientific notation for printed summaries."""
    return f"{float(x):.{digits}f}"


def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def columnwise_corr(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise Pearson r using Huth-style z-scoring.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return np.nan_to_num((zs(y_true) * zs(y_pred)).mean(axis=0)).astype(np.float32)


def columnwise_r2(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise cross-validated R2.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - y_true.mean(axis=0, keepdims=True)) ** 2, axis=0)
    r2 = 1.0 - ss_res / np.maximum(ss_tot, 1e-12)
    return np.nan_to_num(r2).astype(np.float32)


def ridge_fit_predict_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    alpha: float,
    singcutoff: float = 1e-10,
):
    """
    Huth-style SVD ridge fit + predict for one shared alpha.
    """
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))

    U = U[:, :ngoodS]
    S = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = np.dot(U.T, np.nan_to_num(Rresp))
    wt = Vh.T.dot(np.diag(S / (S**2 + float(alpha) ** 2))).dot(UR)
    pred = np.dot(Pstim, wt).astype(np.float32)

    return wt.astype(np.float32), pred


def ridge_score_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    Presp: np.ndarray,
    alpha: float,
    use_corr: bool = True,
    singcutoff: float = 1e-10,
) -> np.ndarray:
    """
    Fit on Rstim/Rresp and score predictions on Pstim/Presp.
    """
    _, pred = ridge_fit_predict_huthstyle(
        Rstim=Rstim,
        Pstim=Pstim,
        Rresp=Rresp,
        alpha=float(alpha),
        singcutoff=singcutoff,
    )
    if use_corr:
        return columnwise_corr(Presp, pred)
    return columnwise_r2(Presp, pred)


def make_bootstrap_chunks(n_timepoints: int, chunklen: int, nchunks: int, rng: random.Random) -> np.ndarray:
    """
    Select validation chunks without overlap, following the same broad logic as Huth-style chunk CV.
    """
    chunk_starts = list(range(0, n_timepoints - chunklen + 1, chunklen))
    if len(chunk_starts) < nchunks:
        raise ValueError(
            f"Cannot draw {nchunks} chunks of length {chunklen} from {n_timepoints} timepoints."
        )
    chosen_starts = rng.sample(chunk_starts, nchunks)
    val_indices = []
    for start in chosen_starts:
        val_indices.extend(range(start, start + chunklen))
    return np.asarray(sorted(set(val_indices)), dtype=int)


def select_alpha_for_pair(
    X_train: np.ndarray,
    Y_train: np.ndarray,
    alphas: np.ndarray,
    nboots: int,
    chunklen: int,
    nchunks: int,
    use_corr: bool,
    seed: int,
) -> dict:
    """
    Select one shared ridge alpha for one network x layer pair.
    """
    rng = random.Random(seed)
    n_time = int(X_train.shape[0])
    n_voxels = int(Y_train.shape[1])
    n_alphas = int(len(alphas))

    bootstrap_scores = np.zeros((nboots, n_alphas, n_voxels), dtype=np.float32)

    for boot in range(nboots):
        val_idx = make_bootstrap_chunks(n_time, chunklen, nchunks, rng)
        train_mask = np.ones(n_time, dtype=bool)
        train_mask[val_idx] = False
        train_idx = np.where(train_mask)[0]

        X_boot_train = X_train[train_idx]
        Y_boot_train = Y_train[train_idx]
        X_boot_val = X_train[val_idx]
        Y_boot_val = Y_train[val_idx]

        for alpha_i, alpha in enumerate(alphas):
            score = ridge_score_huthstyle(
                Rstim=X_boot_train,
                Pstim=X_boot_val,
                Rresp=Y_boot_train,
                Presp=Y_boot_val,
                alpha=float(alpha),
                use_corr=use_corr,
                singcutoff=SINGCUTOFF,
            )
            bootstrap_scores[boot, alpha_i, :] = score

    mean_scores_by_alpha_voxel = np.nanmean(bootstrap_scores, axis=0)
    mean_scores_by_alpha = np.nanmean(mean_scores_by_alpha_voxel, axis=1)
    best_alpha_index = int(np.nanargmax(mean_scores_by_alpha))
    best_alpha = float(alphas[best_alpha_index])

    return {
        "best_alpha": best_alpha,
        "best_alpha_index": best_alpha_index,
        "mean_scores_by_alpha": mean_scores_by_alpha.astype(np.float32),
        "mean_scores_by_alpha_voxel": mean_scores_by_alpha_voxel.astype(np.float32),
    }


# =========================
# 4. Input validation
# =========================

step5_summary_csv = STEP5_DIR / "csv" / "step5_layerwise_design_summary.csv"
step8a_summary_csv = STEP8A_DIR / "csv" / "step8a_all_network_srm_reconstruction_summary.csv"

for p in [step5_summary_csv, step8a_summary_csv]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required summary file: {p}")

step5_summary_df = pd.read_csv(step5_summary_csv)
step8a_summary_df = pd.read_csv(step8a_summary_csv)

expected_pairs = len(NETWORK_NAMES) * len(LAYER_LABELS)
if len(step5_summary_df) != len(LAYER_LABELS):
    raise ValueError(
        f"Expected {len(LAYER_LABELS)} layer rows in Step 5 summary, found {len(step5_summary_df)}."
    )
if len(step8a_summary_df) != len(NETWORK_NAMES):
    raise ValueError(
        f"Expected {len(NETWORK_NAMES)} network rows in Step 8A summary, found {len(step8a_summary_df)}."
    )
if not step8a_summary_df["train_alignment_ok"].all() or not step8a_summary_df["test_alignment_ok"].all():
    raise ValueError("At least one Step 8A network failed alignment checks.")

print("\nInput summaries:")
print(f"Step 5 layer rows: {len(step5_summary_df)}")
print(f"Step 8A network rows: {len(step8a_summary_df)}")
print(f"Expected network x layer pairs: {expected_pairs}")
print("Step 8A alignment: all OK")


# =========================
# 5. Run alpha selection
# =========================

start_time = time.time()
summary_rows = []

pair_list = list(itools.product(NETWORK_NAMES, LAYER_LABELS))
iterator = pair_list
if tqdm is not None:
    iterator = tqdm(pair_list, desc="Step 8B SRM alpha selection: network x layer")

for pair_index, (roi_name, layer_label) in enumerate(iterator, start=1):
    pair_seed = SEED + pair_index

    layer_step5_dir = STEP5_DIR / layer_label
    X_train_path = layer_step5_dir / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
    X_test_path = layer_step5_dir / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

    step8a_network_dir = STEP8A_DIR / roi_name
    Y_train_path = step8a_network_dir / "npy" / "Y_train_group_srm_reconstructed_huthstyle.npy"
    Y_test_path = step8a_network_dir / "npy" / "Y_test_group_srm_reconstructed_huthstyle.npy"

    for p in [X_train_path, X_test_path, Y_train_path, Y_test_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing input for {roi_name} {layer_label}: {p}")

    X_train = np.load(X_train_path).astype(np.float32)
    X_test = np.load(X_test_path).astype(np.float32)
    Y_train = np.load(Y_train_path).astype(np.float32)
    Y_test = np.load(Y_test_path).astype(np.float32)

    if X_train.shape[0] != Y_train.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: train time mismatch: X={X_train.shape}, Y={Y_train.shape}"
        )
    if X_test.shape[0] != Y_test.shape[0]:
        raise ValueError(
            f"{roi_name} {layer_label}: test time mismatch: X={X_test.shape}, Y={Y_test.shape}"
        )
    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in X.")
    if not np.isfinite(Y_train).all() or not np.isfinite(Y_test).all():
        raise ValueError(f"{roi_name} {layer_label}: non-finite values in SRM Y.")

    result = select_alpha_for_pair(
        X_train=X_train,
        Y_train=Y_train,
        alphas=ALPHAS,
        nboots=NBOOTS,
        chunklen=CHUNKLEN,
        nchunks=NCHUNKS,
        use_corr=USE_CORR,
        seed=pair_seed,
    )

    # Held-out test sanity check using the selected shared alpha.
    _, test_pred = ridge_fit_predict_huthstyle(
        Rstim=X_train,
        Pstim=X_test,
        Rresp=Y_train,
        alpha=result["best_alpha"],
        singcutoff=SINGCUTOFF,
    )
    test_corrs = columnwise_corr(Y_test, test_pred)

    roi_layer_dir = STEP8B_ALPHA_DIR / roi_name / layer_label
    roi_layer_npy_dir = roi_layer_dir / "npy"
    roi_layer_json_dir = roi_layer_dir / "json"
    for d in [roi_layer_dir, roi_layer_npy_dir, roi_layer_json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    mean_scores_path = roi_layer_npy_dir / "mean_validation_scores_by_alpha.npy"
    voxel_scores_path = roi_layer_npy_dir / "mean_validation_scores_by_alpha_voxel.npy"
    test_corrs_path = roi_layer_npy_dir / "heldout_test_corrs_srm_group.npy"

    np.save(mean_scores_path, result["mean_scores_by_alpha"])
    np.save(voxel_scores_path, result["mean_scores_by_alpha_voxel"])
    np.save(test_corrs_path, test_corrs.astype(np.float32))

    best_validation_score = float(result["mean_scores_by_alpha"][result["best_alpha_index"]])
    row = {
        "roi_name": roi_name,
        "layer_label": layer_label,
        "layer_index": int(layer_label.split("_")[1]),
        "target_type": "srm_group",
        "alpha_selection_scope": "single_alpha_across_voxels",
        "heldout_test_alpha_mode": "single_best_alpha",
        "best_alpha": float(result["best_alpha"]),
        "best_alpha_index": int(result["best_alpha_index"]),
        "best_validation_score": best_validation_score,
        "heldout_test_mean_r": float(np.nanmean(test_corrs)),
        "heldout_test_median_r": float(np.nanmedian(test_corrs)),
        "heldout_test_max_r": float(np.nanmax(test_corrs)),
        "heldout_test_positive_fraction": float(np.mean(test_corrs > 0)),
        "n_train_timepoints": int(X_train.shape[0]),
        "n_test_timepoints": int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
        "n_voxels": int(Y_train.shape[1]),
        "nboots": int(NBOOTS),
        "chunklen": int(CHUNKLEN),
        "nchunks": int(NCHUNKS),
        "use_corr": bool(USE_CORR),
        "single_alpha": bool(SINGLE_ALPHA),
        "mean_scores_path": str(mean_scores_path),
        "voxel_scores_path": str(voxel_scores_path),
        "test_corrs_path": str(test_corrs_path),
    }
    summary_rows.append(row)

    summary_json = {
        **row,
        "alphas": [float(a) for a in ALPHAS],
        "mean_validation_scores_by_alpha": [float(x) for x in result["mean_scores_by_alpha"]],
        "X_train_path": str(X_train_path),
        "X_test_path": str(X_test_path),
        "Y_train_srm_group_path": str(Y_train_path),
        "Y_test_srm_group_path": str(Y_test_path),
    }
    pair_json_path = roi_layer_json_dir / "step8b_srm_group_alpha_selection_summary.json"
    with open(pair_json_path, "w", encoding="utf-8") as f:
        json.dump(summary_json, f, indent=2)

    if pair_index == 1 or pair_index % 25 == 0:
        elapsed_min = (time.time() - start_time) / 60.0
        print(
            f"[{pair_index:03d}/{len(pair_list):03d}] {roi_name} {layer_label} | "
            f"alpha={fmt_float(result['best_alpha'], 5)} | "
            f"val={fmt_float(best_validation_score, 6)} | "
            f"test mean r={fmt_float(np.nanmean(test_corrs), 6)} | "
            f"elapsed={fmt_float(elapsed_min, 2)} min"
        )


# =========================
# 6. Save combined summaries and overview figures
# =========================

alpha_summary_df = pd.DataFrame(summary_rows)
alpha_summary_csv = CSV_DIR / "step8b_layerwise_srm_group_alpha_selection_summary.csv"
alpha_summary_df.to_csv(
    alpha_summary_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

best_by_network_df = (
    alpha_summary_df.sort_values(["roi_name", "heldout_test_mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("heldout_test_mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step8b_best_layer_by_network_srm_group_alpha_sanity.csv"
best_by_network_df.to_csv(
    best_by_network_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",
)

pivot_test = alpha_summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="heldout_test_mean_r",
).loc[NETWORK_NAMES]

plt.figure(figsize=(15, 6))
plt.imshow(pivot_test.values, aspect="auto", cmap="viridis")
plt.colorbar(label="Held-out mean encoding r")
plt.yticks(np.arange(len(pivot_test.index)), pivot_test.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL representation index")
plt.ylabel("Network")
plt.title("SRM group-level GPT-2 layer-wise alpha sanity check")
plt.tight_layout()
heatmap_path = FIG_DIR / "step8b_srm_group_layerwise_heldout_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=240)
plt.close()

plt.figure(figsize=(15, 6))
alpha_pivot = alpha_summary_df.pivot(
    index="roi_name",
    columns="layer_index",
    values="best_alpha",
).loc[NETWORK_NAMES]
plt.imshow(np.log10(alpha_pivot.values), aspect="auto", cmap="magma")
plt.colorbar(label="log10(selected alpha)")
plt.yticks(np.arange(len(alpha_pivot.index)), alpha_pivot.index)
plt.xticks(np.arange(0, len(LAYER_LABELS), 4), np.arange(0, len(LAYER_LABELS), 4))
plt.xlabel("GPT-2 XL representation index")
plt.ylabel("Network")
plt.title("Selected alpha for SRM group responses")
plt.tight_layout()
alpha_heatmap_path = FIG_DIR / "step8b_srm_group_selected_alpha_heatmap.png"
plt.savefig(alpha_heatmap_path, dpi=240)
plt.close()

total_elapsed_min = (time.time() - start_time) / 60.0
summary = {
    "dataset": "21styear",
    "pipeline_type": "layerwise_srm_group_alpha_selection",
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_LABELS)),
    "target_type": "srm_group",
    "alphas": [float(a) for a in ALPHAS],
    "nboots": int(NBOOTS),
    "chunklen": int(CHUNKLEN),
    "nchunks": int(NCHUNKS),
    "use_corr": bool(USE_CORR),
    "single_alpha": bool(SINGLE_ALPHA),
    "singcutoff": float(SINGCUTOFF),
    "seed": int(SEED),
    "input_step5_dir": str(STEP5_DIR),
    "input_step8a_dir": str(STEP8A_DIR),
    "outputs": {
        "alpha_summary_csv": str(alpha_summary_csv),
        "best_by_network_csv": str(best_by_network_csv),
        "heldout_heatmap": str(heatmap_path),
        "alpha_heatmap": str(alpha_heatmap_path),
    },
    "total_elapsed_minutes": float(total_elapsed_min),
}

summary_json_path = JSON_DIR / "step8b_layerwise_srm_group_alpha_selection_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("STEP 8B SRM GROUP-LEVEL ALPHA SELECTION SUMMARY")
print("=" * 80)
print(f"Saved alpha summary: {alpha_summary_csv}")
print(f"Saved best layer by network: {best_by_network_csv}")
print(f"Saved held-out heatmap: {heatmap_path}")
print(f"Saved alpha heatmap: {alpha_heatmap_path}")
print(f"Saved JSON summary: {summary_json_path}")
print(f"Total elapsed: {fmt_float(total_elapsed_min, 2)} min")
print("\nBest SRM alpha-sanity layer by network:")
print(best_by_network_df[[
    "roi_name",
    "layer_label",
    "best_alpha",
    "best_validation_score",
    "heldout_test_mean_r",
    "heldout_test_median_r",
    "heldout_test_positive_fraction",
]].to_string(index=False))

print("\nStep 8B SRM alpha selection completed successfully.")


# Step 8C: Group-Level SRM-Reconstructed Encoding For Each Network And GPT-2 Layer

This step fits the final group-level SRM+GPT-2 encoding model for every functional network and every layer-wise GPT-2 representation. It uses the delayed layer-wise design matrices from Step 5, the SRM-reconstructed group responses from Step 8A, and the SRM-specific group alpha values selected in Step 8B.

The target here is group-level SRM-reconstructed fMRI response, not participant-level response. The later participant-level SRM follow-up should be updated separately before running.


In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm


# =========================
# Step 8C. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP8A_DIR = PROJECT_DIR / "step8a_prepare_srm_reconstructed_response"
STEP8B_ALPHA_DIR = PROJECT_DIR / "step8b_select_alpha_srm_group_layerwise"
STEP8C_DIR = PROJECT_DIR / "step8c_srm_group_encoding_layerwise"

FIG_DIR = STEP8C_DIR / "figures"
CSV_DIR = STEP8C_DIR / "csv"
NPY_DIR = STEP8C_DIR / "npy"
JSON_DIR = STEP8C_DIR / "json"

for d in [STEP8C_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
     "LimbicA", "LimbicB","TempPar",
]

N_LAYER_REPRESENTATIONS = 49
LAYER_LABELS = [f"layer_{i:02d}" for i in range(N_LAYER_REPRESENTATIONS)]
SINGCUTOFF = 1e-10

print("Step 8C: Group-level SRM-reconstructed encoding for layer-wise GPT-2 representations")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 5 directory : {STEP5_DIR}")
print(f"Step 8A directory: {STEP8A_DIR}")
print(f"Step 8B alpha dir: {STEP8B_ALPHA_DIR}")
print(f"Step 8C directory: {STEP8C_DIR}")
print(f"Networks: {len(NETWORK_NAMES)}")
print(f"Layer representations: {len(LAYER_LABELS)}")


# =========================
# Step 8C. Helper functions
# =========================

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"


def zs(x, axis=0, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mean = np.nanmean(x, axis=axis, keepdims=True)
    std = np.nanstd(x, axis=axis, keepdims=True)
    return (x - mean) / (std + eps)


def columnwise_corr(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch for correlation: {y_true.shape} vs {y_pred.shape}")

    yt = y_true - np.nanmean(y_true, axis=0, keepdims=True)
    yp = y_pred - np.nanmean(y_pred, axis=0, keepdims=True)
    denom = np.sqrt(np.nansum(yt ** 2, axis=0) * np.nansum(yp ** 2, axis=0)) + eps
    corr = np.nansum(yt * yp, axis=0) / denom
    corr = corr.astype(np.float32)
    corr[~np.isfinite(corr)] = 0.0
    return corr


def ridge_fit_predict_huthstyle(X_train, Y_train, X_test, alpha, singcutoff=SINGCUTOFF):
    X_train = np.asarray(X_train, dtype=np.float32)
    Y_train = np.asarray(Y_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    alpha = float(alpha)

    if X_train.shape[0] != Y_train.shape[0]:
        raise ValueError(f"X/Y train time mismatch: {X_train.shape} vs {Y_train.shape}")
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError(f"X train/test feature mismatch: {X_train.shape} vs {X_test.shape}")

    U, S, Vt = np.linalg.svd(X_train, full_matrices=False)
    valid = S > singcutoff
    if not np.any(valid):
        raise ValueError("No singular values survived singcutoff in ridge fit.")

    U = U[:, valid]
    S = S[valid]
    Vt = Vt[valid, :]

    shrink = S / (S ** 2 + alpha ** 2)
    weights = Vt.T @ (shrink[:, None] * (U.T @ Y_train))
    Y_pred = X_test @ weights
    return Y_pred.astype(np.float32), weights.astype(np.float32)


# =========================
# Step 8C. Load manifest tables
# =========================

step5_summary_path = STEP5_DIR / "csv" / "step5_layerwise_design_summary.csv"
step8a_summary_path = STEP8A_DIR / "csv" / "step8a_all_network_srm_reconstruction_summary.csv"
step8b_alpha_path = STEP8B_ALPHA_DIR / "csv" / "step8b_layerwise_srm_group_alpha_selection_summary.csv"

for p in [step5_summary_path, step8a_summary_path, step8b_alpha_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Step 8C input table: {p}")

step5_summary_df = pd.read_csv(step5_summary_path)
step8a_summary_df = pd.read_csv(step8a_summary_path)
alpha_df = pd.read_csv(step8b_alpha_path)

required_alpha_cols = {"roi_name", "layer_label", "best_alpha"}
missing_alpha_cols = required_alpha_cols - set(alpha_df.columns)
if missing_alpha_cols:
    raise ValueError(f"Step 8B alpha table is missing columns: {sorted(missing_alpha_cols)}")

alpha_df = alpha_df.copy()
alpha_df["best_alpha"] = pd.to_numeric(alpha_df["best_alpha"], errors="coerce")
if alpha_df["best_alpha"].isna().any():
    bad = alpha_df.loc[alpha_df["best_alpha"].isna(), ["roi_name", "layer_label", "best_alpha"]].head(10)
    raise ValueError(f"Non-numeric best_alpha values detected:\n{bad}")

expected_pairs = len(NETWORK_NAMES) * len(LAYER_LABELS)
if len(alpha_df) != expected_pairs:
    raise ValueError(f"Expected {expected_pairs} Step 8B alpha rows, found {len(alpha_df)}")

alpha_lookup = {
    (str(row.roi_name), str(row.layer_label)): float(row.best_alpha)
    for row in alpha_df.itertuples(index=False)
}

missing_pairs = [
    (roi_name, layer_label)
    for roi_name in NETWORK_NAMES
    for layer_label in LAYER_LABELS
    if (roi_name, layer_label) not in alpha_lookup
]
if missing_pairs:
    raise ValueError(f"Missing alpha values for first pairs: {missing_pairs[:10]}")

print("\nLoaded manifest tables:")
print(f"Step 5 design rows : {len(step5_summary_df)}")
print(f"Step 8A SRM rows   : {len(step8a_summary_df)}")
print(f"Step 8B alpha rows : {len(alpha_df)}")
print(f"Expected pairs     : {expected_pairs}")
print(f"Alpha range        : {fmt_float(alpha_df['best_alpha'].min())} to {fmt_float(alpha_df['best_alpha'].max())}")


# =========================
# Step 8C. Fit group-level SRM encoding models
# =========================

start_total = time.time()
summary_rows = []

iterator = list(range(expected_pairs))
pbar = tqdm(total=expected_pairs, desc="Step 8C SRM group encoding", unit="model")
model_counter = 0

for roi_name in NETWORK_NAMES:
    roi_target = f"{roi_name}_only"
    y_train_path = STEP8A_DIR / roi_name / "npy" / "Y_train_group_srm_reconstructed_huthstyle.npy"
    y_test_path = STEP8A_DIR / roi_name / "npy" / "Y_test_group_srm_reconstructed_huthstyle.npy"

    if not y_train_path.exists():
        raise FileNotFoundError(f"Missing SRM group train response for {roi_name}: {y_train_path}")
    if not y_test_path.exists():
        raise FileNotFoundError(f"Missing SRM group test response for {roi_name}: {y_test_path}")

    Y_train = np.load(y_train_path).astype(np.float32)
    Y_test = np.load(y_test_path).astype(np.float32)

    if Y_train.ndim != 2 or Y_test.ndim != 2:
        raise ValueError(f"Expected 2D group response arrays for {roi_name}, got {Y_train.shape} and {Y_test.shape}")
    if Y_train.shape[1] != Y_test.shape[1]:
        raise ValueError(f"Train/test voxel mismatch for {roi_name}: {Y_train.shape} vs {Y_test.shape}")

    for layer_label in LAYER_LABELS:
        layer_index = int(layer_label.split("_")[1])
        layer_dir = STEP5_DIR / layer_label / "npy"
        x_train_path = layer_dir / "X_train_delayed_lanczos_huthstyle.npy"
        x_test_path = layer_dir / "X_test_delayed_lanczos_huthstyle.npy"

        if not x_train_path.exists():
            raise FileNotFoundError(f"Missing delayed train design for {layer_label}: {x_train_path}")
        if not x_test_path.exists():
            raise FileNotFoundError(f"Missing delayed test design for {layer_label}: {x_test_path}")

        X_train = np.load(x_train_path).astype(np.float32)
        X_test = np.load(x_test_path).astype(np.float32)

        if X_train.shape[0] != Y_train.shape[0]:
            raise ValueError(f"Train time mismatch for {roi_name} {layer_label}: X {X_train.shape}, Y {Y_train.shape}")
        if X_test.shape[0] != Y_test.shape[0]:
            raise ValueError(f"Test time mismatch for {roi_name} {layer_label}: X {X_test.shape}, Y {Y_test.shape}")

        best_alpha = alpha_lookup[(roi_name, layer_label)]
        Y_pred, weights = ridge_fit_predict_huthstyle(X_train, Y_train, X_test, alpha=best_alpha)
        voxel_corrs = columnwise_corr(Y_test, Y_pred)

        pair_dir = STEP8C_DIR / roi_name / layer_label
        pair_npy_dir = pair_dir / "npy"
        pair_json_dir = pair_dir / "json"
        pair_npy_dir.mkdir(parents=True, exist_ok=True)
        pair_json_dir.mkdir(parents=True, exist_ok=True)

        pred_path = pair_npy_dir / "group_Y_pred_srm.npy"
        corrs_path = pair_npy_dir / "group_voxel_corrs_srm.npy"
        weights_path = pair_npy_dir / "group_weights_srm.npy"
        pair_summary_path = pair_json_dir / "step8c_srm_group_encoding_summary.json"

        np.save(pred_path, Y_pred.astype(np.float32))
        np.save(corrs_path, voxel_corrs.astype(np.float32))
        np.save(weights_path, weights.astype(np.float32))

        row = {
            "roi_name": roi_name,
            "roi_target": roi_target,
            "layer_label": layer_label,
            "layer_index": int(layer_index),
            "target_type": "srm_group",
            "analysis_level": "group",
            "best_alpha": float(best_alpha),
            "mean_r": float(np.nanmean(voxel_corrs)),
            "median_r": float(np.nanmedian(voxel_corrs)),
            "max_r": float(np.nanmax(voxel_corrs)),
            "positive_voxel_fraction": float(np.mean(voxel_corrs > 0)),
            "n_train_timepoints": int(X_train.shape[0]),
            "n_test_timepoints": int(X_test.shape[0]),
            "n_features": int(X_train.shape[1]),
            "n_voxels": int(Y_train.shape[1]),
            "x_train_path": str(x_train_path),
            "x_test_path": str(x_test_path),
            "y_train_path": str(y_train_path),
            "y_test_path": str(y_test_path),
            "prediction_path": str(pred_path),
            "voxel_corrs_path": str(corrs_path),
            "weights_path": str(weights_path),
        }

        with open(pair_summary_path, "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)

        summary_rows.append(row)
        model_counter += 1
        pbar.update(1)

        if model_counter % 50 == 0:
            elapsed_min = (time.time() - start_total) / 60.0
            print(
                f"Completed {model_counter}/{expected_pairs} models | "
                f"latest {roi_name} {layer_label}: mean r = {fmt_float(row['mean_r'])}, "
                f"alpha = {fmt_float(best_alpha)} | elapsed {fmt_float(elapsed_min, 2)} min"
            )

pbar.close()

summary_df = pd.DataFrame(summary_rows)
if len(summary_df) != expected_pairs:
    raise ValueError(f"Expected {expected_pairs} summary rows, found {len(summary_df)}")
if summary_df[["mean_r", "median_r", "max_r"]].isna().any().any():
    raise ValueError("NaN detected in Step 8C summary metrics.")

summary_csv = CSV_DIR / "step8c_layerwise_srm_group_encoding_summary.csv"
summary_json = JSON_DIR / "step8c_layerwise_srm_group_encoding_summary.json"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary_rows, f, indent=2)

best_by_network_df = (
    summary_df.sort_values(["roi_name", "mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step8c_best_layer_by_network_srm_group.csv"
best_by_network_df.to_csv(best_by_network_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved Step 8C group-level SRM encoding summaries:")
print(f"Summary CSV        : {summary_csv}")
print(f"Best-by-network CSV: {best_by_network_csv}")
print("\nBest SRM group layer by network:")
print(best_by_network_df[["roi_name", "layer_label", "best_alpha", "mean_r", "median_r", "positive_voxel_fraction"]].to_string(index=False, float_format=lambda x: f"{x:.5f}"))


# =========================
# Step 8C. Figures
# =========================

heatmap_df = summary_df.pivot(index="roi_name", columns="layer_label", values="mean_r").loc[NETWORK_NAMES, LAYER_LABELS]
plt.figure(figsize=(18, 7))
sns.heatmap(
    heatmap_df,
    cmap="magma",
    linewidths=0.15,
    linecolor="white",
    cbar_kws={"label": "Mean held-out encoding r"},
)
plt.title("SRM+GPT-2 Group-Level Encoding Across Networks And Layers")
plt.xlabel("GPT-2 representation")
plt.ylabel("Network")
plt.tight_layout()
heatmap_path = FIG_DIR / "step8c_srm_group_layerwise_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=300)
plt.close()

plt.figure(figsize=(13, 7))
for roi_name in NETWORK_NAMES:
    roi_df = summary_df[summary_df["roi_name"] == roi_name].sort_values("layer_index")
    plt.plot(roi_df["layer_index"], roi_df["mean_r"], linewidth=1.6, alpha=0.85, label=roi_name)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("GPT-2 representation index (0 = embedding; 1-48 = transformer layers)")
plt.ylabel("Mean held-out encoding r")
plt.title("SRM+GPT-2 Group-Level Encoding Profiles")
plt.legend(ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
profiles_path = FIG_DIR / "step8c_srm_group_layer_profiles.png"
plt.savefig(profiles_path, dpi=300)
plt.close()

plot_best_df = best_by_network_df.sort_values("mean_r", ascending=True)
plt.figure(figsize=(8, 8))
plt.barh(plot_best_df["roi_name"], plot_best_df["mean_r"], color="#C44E52", alpha=0.85)
for i, row in enumerate(plot_best_df.itertuples(index=False)):
    plt.text(
        float(row.mean_r) + 0.002,
        i,
        f"{row.layer_label} | r={float(row.mean_r):.5f}",
        va="center",
        fontsize=8,
    )
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Best mean held-out encoding r")
plt.ylabel("Network")
plt.title("Best SRM+GPT-2 Group-Level Layer By Network")
plt.tight_layout()
best_bar_path = FIG_DIR / "step8c_best_srm_group_layer_by_network.png"
plt.savefig(best_bar_path, dpi=300)
plt.close()

print("\nSaved Step 8C figures:")
print(f"Mean-r heatmap : {heatmap_path}")
print(f"Layer profiles : {profiles_path}")
print(f"Best-layer bar : {best_bar_path}")


# =========================
# Step 8C. Final validation
# =========================

total_elapsed = (time.time() - start_total) / 60.0
validation = {
    "expected_models": int(expected_pairs),
    "completed_models": int(len(summary_df)),
    "n_networks": int(summary_df["roi_name"].nunique()),
    "n_layer_representations": int(summary_df["layer_label"].nunique()),
    "target_type_values": sorted(summary_df["target_type"].unique().tolist()),
    "analysis_level_values": sorted(summary_df["analysis_level"].unique().tolist()),
    "mean_r_min": float(summary_df["mean_r"].min()),
    "mean_r_max": float(summary_df["mean_r"].max()),
    "total_elapsed_minutes": float(total_elapsed),
}
validation_path = JSON_DIR / "step8c_validation_summary.json"
with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2)

print("\nStep 8C validation:")
for k, v in validation.items():
    if isinstance(v, float):
        print(f"{k}: {fmt_float(v)}")
    else:
        print(f"{k}: {v}")

print(f"\nTotal elapsed: {total_elapsed:.2f} min")
print("Step 8C completed successfully.")


# Step 8D: Participant-Level SRM-Reconstructed Encoding For Each Network And GPT-2 Layer

This follow-up fits one encoding model per participant using SRM-reconstructed fMRI responses. It mirrors the raw participant-level follow-up, but the response targets come from Step 8A and the shared alpha values come from the SRM-specific group-level alpha selection in Step 8B.

This cell is participant-level because each model predicts one participant's SRM-reconstructed test response separately. Full prediction matrices are not saved by default because all network × layer × participant predictions are very large; the saved outputs focus on participant-wise and voxel-wise encoding correlations.


In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# =========================
# Step 8D. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix_layerwise"
STEP8A_DIR = PROJECT_DIR / "step8a_prepare_srm_reconstructed_response"
STEP8B_ALPHA_DIR = PROJECT_DIR / "step8b_select_alpha_srm_group_layerwise"
STEP8D_DIR = PROJECT_DIR / "step8d_srm_subject_encoding_layerwise"

FIG_DIR = STEP8D_DIR / "figures"
CSV_DIR = STEP8D_DIR / "csv"
NPY_DIR = STEP8D_DIR / "npy"
JSON_DIR = STEP8D_DIR / "json"

for d in [STEP8D_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
    "LimbicA", "LimbicB","TempPar",
]

N_LAYER_REPRESENTATIONS = 49
LAYER_LABELS = [f"layer_{i:02d}" for i in range(N_LAYER_REPRESENTATIONS)]
SINGCUTOFF = 1e-10
SAVE_FULL_SUBJECT_PREDICTIONS = False

print("Step 8D: Participant-level SRM-reconstructed encoding for layer-wise GPT-2 representations")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 5 directory : {STEP5_DIR}")
print(f"Step 8A directory: {STEP8A_DIR}")
print(f"Step 8B alpha dir: {STEP8B_ALPHA_DIR}")
print(f"Step 8D directory: {STEP8D_DIR}")
print(f"Networks: {len(NETWORK_NAMES)}")
print(f"Layer representations: {len(LAYER_LABELS)}")
print(f"Save full subject predictions: {SAVE_FULL_SUBJECT_PREDICTIONS}")


# =========================
# Step 8D. Helper functions
# =========================

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"


def columnwise_corr(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch for correlation: {y_true.shape} vs {y_pred.shape}")

    yt = y_true - np.nanmean(y_true, axis=0, keepdims=True)
    yp = y_pred - np.nanmean(y_pred, axis=0, keepdims=True)
    denom = np.sqrt(np.nansum(yt ** 2, axis=0) * np.nansum(yp ** 2, axis=0)) + eps
    corr = np.nansum(yt * yp, axis=0) / denom
    corr = corr.astype(np.float32)
    corr[~np.isfinite(corr)] = 0.0
    return corr


def ridge_fit_predict_huthstyle(Rstim, Pstim, Rresp, alpha, singcutoff=SINGCUTOFF):
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)
    alpha = float(alpha)

    if Rstim.shape[0] != Rresp.shape[0]:
        raise ValueError(f"Rstim/Rresp time mismatch: {Rstim.shape} vs {Rresp.shape}")
    if Rstim.shape[1] != Pstim.shape[1]:
        raise ValueError(f"Train/test feature mismatch: {Rstim.shape} vs {Pstim.shape}")

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    valid = S > singcutoff
    if not np.any(valid):
        raise ValueError("No singular values survived singcutoff in ridge fit.")

    U = U[:, valid]
    S = S[valid]
    Vh = Vh[valid, :]
    UR = U.T @ Rresp

    # Huth-style ridge convention: alphas are regularization scale values,
    # so the SVD shrinkage denominator uses alpha squared.
    shrink = S / (S ** 2 + alpha ** 2)
    wt = Vh.T @ (shrink[:, None] * UR)
    pred = Pstim @ wt
    return wt.astype(np.float32), pred.astype(np.float32)


# =========================
# Step 8D. Load alpha table and validate inputs
# =========================

step5_summary_path = STEP5_DIR / "csv" / "step5_layerwise_design_summary.csv"
step8a_summary_path = STEP8A_DIR / "csv" / "step8a_all_network_srm_reconstruction_summary.csv"
step8b_alpha_path = STEP8B_ALPHA_DIR / "csv" / "step8b_layerwise_srm_group_alpha_selection_summary.csv"

for p in [step5_summary_path, step8a_summary_path, step8b_alpha_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Step 8D input table: {p}")

step5_summary_df = pd.read_csv(step5_summary_path)
step8a_summary_df = pd.read_csv(step8a_summary_path)
alpha_df = pd.read_csv(step8b_alpha_path)

required_alpha_cols = {"roi_name", "layer_label", "best_alpha"}
missing_alpha_cols = required_alpha_cols - set(alpha_df.columns)
if missing_alpha_cols:
    raise ValueError(f"Step 8B alpha table is missing columns: {sorted(missing_alpha_cols)}")

alpha_df = alpha_df.copy()
alpha_df["best_alpha"] = pd.to_numeric(alpha_df["best_alpha"], errors="coerce")
if alpha_df["best_alpha"].isna().any():
    bad = alpha_df.loc[alpha_df["best_alpha"].isna(), ["roi_name", "layer_label", "best_alpha"]].head(10)
    raise ValueError(f"Non-numeric best_alpha values detected:\n{bad}")

expected_pairs = len(NETWORK_NAMES) * len(LAYER_LABELS)
if len(alpha_df) != expected_pairs:
    raise ValueError(f"Expected {expected_pairs} Step 8B alpha rows, found {len(alpha_df)}")

alpha_lookup = {
    (str(row.roi_name), str(row.layer_label)): float(row.best_alpha)
    for row in alpha_df.itertuples(index=False)
}

missing_pairs = [
    (roi_name, layer_label)
    for roi_name in NETWORK_NAMES
    for layer_label in LAYER_LABELS
    if (roi_name, layer_label) not in alpha_lookup
]
if missing_pairs:
    raise ValueError(f"Missing alpha values for first pairs: {missing_pairs[:10]}")

print("\nLoaded manifest tables:")
print(f"Step 5 design rows : {len(step5_summary_df)}")
print(f"Step 8A SRM rows   : {len(step8a_summary_df)}")
print(f"Step 8B alpha rows : {len(alpha_df)}")
print(f"Expected pairs     : {expected_pairs}")
print(f"Alpha range        : {fmt_float(alpha_df['best_alpha'].min())} to {fmt_float(alpha_df['best_alpha'].max())}")


# =========================
# Step 8D. Fit participant-level SRM encoding models
# =========================

start_total = time.time()
pair_summary_rows = []
subject_summary_rows = []

progress_total = expected_pairs
progress = tqdm(total=progress_total, desc="Step 8D SRM participant encoding", unit="pair") if tqdm is not None else None
pair_counter = 0

for roi_name in NETWORK_NAMES:
    roi_target = f"{roi_name}_only"
    y_train_list_path = STEP8A_DIR / roi_name / "npy" / "Y_train_list_srm_reconstructed_huthstyle.npy"
    y_test_list_path = STEP8A_DIR / roi_name / "npy" / "Y_test_list_srm_reconstructed_huthstyle.npy"

    if not y_train_list_path.exists():
        raise FileNotFoundError(f"Missing SRM subject train response for {roi_name}: {y_train_list_path}")
    if not y_test_list_path.exists():
        raise FileNotFoundError(f"Missing SRM subject test response for {roi_name}: {y_test_list_path}")

    Y_train_subjects = np.load(y_train_list_path).astype(np.float32)
    Y_test_subjects = np.load(y_test_list_path).astype(np.float32)

    if Y_train_subjects.ndim != 3 or Y_test_subjects.ndim != 3:
        raise ValueError(f"Expected 3D subject response arrays for {roi_name}, got {Y_train_subjects.shape} and {Y_test_subjects.shape}")
    if Y_train_subjects.shape[0] != Y_test_subjects.shape[0]:
        raise ValueError(f"Subject count mismatch for {roi_name}: {Y_train_subjects.shape} vs {Y_test_subjects.shape}")
    if Y_train_subjects.shape[2] != Y_test_subjects.shape[2]:
        raise ValueError(f"Voxel count mismatch for {roi_name}: {Y_train_subjects.shape} vs {Y_test_subjects.shape}")

    n_subjects = int(Y_train_subjects.shape[0])
    n_voxels = int(Y_train_subjects.shape[2])

    for layer_label in LAYER_LABELS:
        layer_index = int(layer_label.split("_")[1])
        x_train_path = STEP5_DIR / layer_label / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
        x_test_path = STEP5_DIR / layer_label / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

        if not x_train_path.exists():
            raise FileNotFoundError(f"Missing delayed train design for {layer_label}: {x_train_path}")
        if not x_test_path.exists():
            raise FileNotFoundError(f"Missing delayed test design for {layer_label}: {x_test_path}")

        X_train = np.load(x_train_path).astype(np.float32)
        X_test = np.load(x_test_path).astype(np.float32)

        if X_train.shape[0] != Y_train_subjects.shape[1]:
            raise ValueError(f"Train time mismatch for {roi_name} {layer_label}: X {X_train.shape}, Y {Y_train_subjects.shape}")
        if X_test.shape[0] != Y_test_subjects.shape[1]:
            raise ValueError(f"Test time mismatch for {roi_name} {layer_label}: X {X_test.shape}, Y {Y_test_subjects.shape}")

        best_alpha = alpha_lookup[(roi_name, layer_label)]

        pair_dir = STEP8D_DIR / roi_name / layer_label
        layer_npy_dir = pair_dir / "npy"
        layer_csv_dir = pair_dir / "csv"
        pred_dir = layer_npy_dir / "subject_predictions_srm"
        layer_npy_dir.mkdir(parents=True, exist_ok=True)
        layer_csv_dir.mkdir(parents=True, exist_ok=True)
        if SAVE_FULL_SUBJECT_PREDICTIONS:
            pred_dir.mkdir(parents=True, exist_ok=True)

        subject_corrs = np.zeros((n_subjects, n_voxels), dtype=np.float32)
        layer_subject_rows = []

        for subj_idx in range(n_subjects):
            Y_train_subj = Y_train_subjects[subj_idx]
            Y_test_subj = Y_test_subjects[subj_idx]

            if not np.isfinite(Y_train_subj).all() or not np.isfinite(Y_test_subj).all():
                raise ValueError(f"{roi_name} {layer_label} subject {subj_idx}: non-finite Y values.")

            _, Y_pred_subj = ridge_fit_predict_huthstyle(
                Rstim=X_train,
                Pstim=X_test,
                Rresp=Y_train_subj,
                alpha=best_alpha,
                singcutoff=SINGCUTOFF,
            )
            voxel_corrs = columnwise_corr(Y_test_subj, Y_pred_subj)
            subject_corrs[subj_idx, :] = voxel_corrs

            pred_path = ""
            if SAVE_FULL_SUBJECT_PREDICTIONS:
                pred_path_obj = pred_dir / f"subject_{subj_idx:02d}_Y_pred_srm.npy"
                np.save(pred_path_obj, Y_pred_subj.astype(np.float32))
                pred_path = str(pred_path_obj)

            subj_row = {
                "roi_name": roi_name,
                "roi_target": roi_target,
                "layer_label": layer_label,
                "layer_index": int(layer_index),
                "target_type": "srm_subject",
                "analysis_level": "participant",
                "alpha_source": "group_selected_shared_alpha_from_step8b_srm",
                "best_alpha": float(best_alpha),
                "subject_index": int(subj_idx),
                "mean_r": float(np.nanmean(voxel_corrs)),
                "median_r": float(np.nanmedian(voxel_corrs)),
                "max_r": float(np.nanmax(voxel_corrs)),
                "positive_voxel_fraction": float(np.mean(voxel_corrs > 0)),
                "n_train_timepoints": int(X_train.shape[0]),
                "n_test_timepoints": int(X_test.shape[0]),
                "n_features": int(X_train.shape[1]),
                "n_voxels": int(n_voxels),
                "prediction_path": pred_path,
            }
            layer_subject_rows.append(subj_row)
            subject_summary_rows.append(subj_row)

        mean_r_across_subjects_voxelwise = np.nanmean(subject_corrs, axis=0).astype(np.float32)
        subject_corrs_path = layer_npy_dir / "subject_voxel_corrs_srm.npy"
        mean_voxel_corrs_path = layer_npy_dir / "mean_r_across_subjects_srm.npy"
        np.save(subject_corrs_path, subject_corrs)
        np.save(mean_voxel_corrs_path, mean_r_across_subjects_voxelwise)

        layer_subject_df = pd.DataFrame(layer_subject_rows)
        subject_summary_csv = layer_csv_dir / "subject_level_srm_encoding_summary.csv"
        layer_subject_df.to_csv(subject_summary_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

        subject_mean_rs = layer_subject_df["mean_r"].to_numpy(dtype=float)
        pair_row = {
            "roi_name": roi_name,
            "roi_target": roi_target,
            "layer_label": layer_label,
            "layer_index": int(layer_index),
            "target_type": "srm_subject",
            "analysis_level": "participant",
            "alpha_source": "group_selected_shared_alpha_from_step8b_srm",
            "best_alpha": float(best_alpha),
            "n_subjects": int(n_subjects),
            "mean_subject_mean_r": float(np.nanmean(subject_mean_rs)),
            "median_subject_mean_r": float(np.nanmedian(subject_mean_rs)),
            "min_subject_mean_r": float(np.nanmin(subject_mean_rs)),
            "max_subject_mean_r": float(np.nanmax(subject_mean_rs)),
            "positive_subject_fraction": float(np.mean(subject_mean_rs > 0)),
            "mean_voxel_mean_r_across_subjects": float(np.nanmean(mean_r_across_subjects_voxelwise)),
            "median_voxel_mean_r_across_subjects": float(np.nanmedian(mean_r_across_subjects_voxelwise)),
            "n_train_timepoints": int(X_train.shape[0]),
            "n_test_timepoints": int(X_test.shape[0]),
            "n_features": int(X_train.shape[1]),
            "n_voxels": int(n_voxels),
            "subject_voxel_corrs_path": str(subject_corrs_path),
            "mean_voxel_corrs_path": str(mean_voxel_corrs_path),
            "subject_summary_csv": str(subject_summary_csv),
        }
        pair_summary_rows.append(pair_row)

        pair_counter += 1
        if progress is not None:
            progress.update(1)
        if pair_counter % 25 == 0:
            elapsed_min = (time.time() - start_total) / 60.0
            print(
                f"Completed {pair_counter}/{expected_pairs} network-layer pairs | "
                f"latest {roi_name} {layer_label}: mean participant r = {fmt_float(pair_row['mean_subject_mean_r'])}, "
                f"alpha = {fmt_float(best_alpha)} | elapsed {fmt_float(elapsed_min, 2)} min"
            )

if progress is not None:
    progress.close()

pair_summary_df = pd.DataFrame(pair_summary_rows)
subject_summary_df = pd.DataFrame(subject_summary_rows)

if len(pair_summary_df) != expected_pairs:
    raise ValueError(f"Expected {expected_pairs} pair summary rows, found {len(pair_summary_df)}")
if pair_summary_df[["mean_subject_mean_r", "median_subject_mean_r", "best_alpha"]].isna().any().any():
    raise ValueError("NaN detected in Step 8D pair summary metrics.")
if subject_summary_df[["mean_r", "median_r", "best_alpha"]].isna().any().any():
    raise ValueError("NaN detected in Step 8D subject summary metrics.")

pair_summary_csv = CSV_DIR / "step8d_srm_subject_encoding_pair_summary.csv"
subject_summary_csv = CSV_DIR / "step8d_srm_subject_encoding_subject_summary.csv"
pair_summary_df.to_csv(pair_summary_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
subject_summary_df.to_csv(subject_summary_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

best_by_network_df = (
    pair_summary_df.sort_values(["roi_name", "mean_subject_mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("mean_subject_mean_r", ascending=False)
    .reset_index(drop=True)
)
best_by_network_csv = CSV_DIR / "step8d_best_layer_by_network_srm_subject.csv"
best_by_network_df.to_csv(best_by_network_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved Step 8D participant-level SRM encoding summaries:")
print(f"Pair summary CSV   : {pair_summary_csv}")
print(f"Subject summary CSV: {subject_summary_csv}")
print(f"Best-by-network CSV: {best_by_network_csv}")
print("\nBest participant-level SRM layer by network:")
print(best_by_network_df[["roi_name", "layer_label", "best_alpha", "mean_subject_mean_r", "median_subject_mean_r", "positive_subject_fraction"]].to_string(index=False, float_format=lambda x: f"{x:.5f}"))


# =========================
# Step 8D. Figures
# =========================

heatmap_df = pair_summary_df.pivot(index="roi_name", columns="layer_label", values="mean_subject_mean_r").loc[NETWORK_NAMES, LAYER_LABELS]
plt.figure(figsize=(18, 7))
sns.heatmap(
    heatmap_df,
    cmap="magma",
    linewidths=0.15,
    linecolor="white",
    cbar_kws={"label": "Mean participant-level encoding r"},
)
plt.title("Participant-Level SRM+GPT-2 Encoding Across Networks And Layers")
plt.xlabel("GPT-2 representation")
plt.ylabel("Network")
plt.tight_layout()
heatmap_path = FIG_DIR / "step8d_srm_subject_layerwise_mean_r_heatmap.png"
plt.savefig(heatmap_path, dpi=300)
plt.close()

plt.figure(figsize=(13, 7))
for roi_name in NETWORK_NAMES:
    roi_df = pair_summary_df[pair_summary_df["roi_name"] == roi_name].sort_values("layer_index")
    plt.plot(roi_df["layer_index"], roi_df["mean_subject_mean_r"], linewidth=1.6, alpha=0.85, label=roi_name)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("GPT-2 representation index (0 = embedding; 1-48 = transformer layers)")
plt.ylabel("Mean participant-level encoding r")
plt.title("Participant-Level SRM+GPT-2 Encoding Profiles")
plt.legend(ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
profiles_path = FIG_DIR / "step8d_srm_subject_layer_profiles.png"
plt.savefig(profiles_path, dpi=300)
plt.close()

plot_best_df = best_by_network_df.sort_values("mean_subject_mean_r", ascending=True)
plt.figure(figsize=(8, 8))
plt.barh(plot_best_df["roi_name"], plot_best_df["mean_subject_mean_r"], color="#DD8452", alpha=0.85)
for i, row in enumerate(plot_best_df.itertuples(index=False)):
    plt.text(
        float(row.mean_subject_mean_r) + 0.001,
        i,
        f"{row.layer_label} | r={float(row.mean_subject_mean_r):.5f}",
        va="center",
        fontsize=8,
    )
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Best mean participant-level encoding r")
plt.ylabel("Network")
plt.title("Best Participant-Level SRM+GPT-2 Layer By Network")
plt.tight_layout()
best_bar_path = FIG_DIR / "step8d_best_srm_subject_layer_by_network.png"
plt.savefig(best_bar_path, dpi=300)
plt.close()

print("\nSaved Step 8D figures:")
print(f"Mean-r heatmap : {heatmap_path}")
print(f"Layer profiles : {profiles_path}")
print(f"Best-layer bar : {best_bar_path}")


# =========================
# Step 8D. Final validation
# =========================

total_elapsed = (time.time() - start_total) / 60.0
validation = {
    "expected_network_layer_pairs": int(expected_pairs),
    "completed_network_layer_pairs": int(len(pair_summary_df)),
    "completed_subject_models": int(len(subject_summary_df)),
    "n_networks": int(pair_summary_df["roi_name"].nunique()),
    "n_layer_representations": int(pair_summary_df["layer_label"].nunique()),
    "n_subjects_min": int(pair_summary_df["n_subjects"].min()),
    "n_subjects_max": int(pair_summary_df["n_subjects"].max()),
    "target_type_values": sorted(pair_summary_df["target_type"].unique().tolist()),
    "analysis_level_values": sorted(pair_summary_df["analysis_level"].unique().tolist()),
    "mean_subject_mean_r_min": float(pair_summary_df["mean_subject_mean_r"].min()),
    "mean_subject_mean_r_max": float(pair_summary_df["mean_subject_mean_r"].max()),
    "save_full_subject_predictions": bool(SAVE_FULL_SUBJECT_PREDICTIONS),
    "total_elapsed_minutes": float(total_elapsed),
}
validation_path = JSON_DIR / "step8d_validation_summary.json"
with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2)

print("\nStep 8D validation:")
for k, v in validation.items():
    if isinstance(v, float):
        print(f"{k}: {fmt_float(v)}")
    else:
        print(f"{k}: {v}")

print(f"\nTotal elapsed: {total_elapsed:.2f} min")
print("Step 8D completed successfully.")


# Step 9: Raw vs SRM Layer-Wise Encoding Summary And Figures

This step integrates the completed layer-wise Raw+LLM and SRM+LLM encoding outputs. It does not fit any new encoding models.

The main goals are to compare Raw+LLM versus SRM+LLM performance across networks and GPT-2 layers at both group level and participant level, summarize SRM-related encoding gains, identify the best-performing network-layer combinations, and generate paired figures with shared axes/color scales so Raw and SRM results can be compared directly.


In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel


# =========================
# Step 9. Project setup
# =========================

DATA_ROOT = Path(r"YOUR_SNL2026_ROOT")
BASE_DIR = DATA_ROOT
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME

STEP7B_DIR = PROJECT_DIR / "step7b_raw_group_encoding_layerwise"
STEP7C_DIR = PROJECT_DIR / "step7c_raw_subject_encoding_layerwise"
STEP8C_DIR = PROJECT_DIR / "step8c_srm_group_encoding_layerwise"
STEP8D_DIR = PROJECT_DIR / "step8d_srm_subject_encoding_layerwise"
STEP9_DIR = PROJECT_DIR / "step9_raw_vs_srm_layerwise_encoding_comparison"

FIG_DIR = STEP9_DIR / "figures"
CSV_DIR = STEP9_DIR / "csv"
JSON_DIR = STEP9_DIR / "json"
for d in [STEP9_DIR, FIG_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
    "TempPar",
]
LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]
LAYER_INDEXES = list(range(49))

print("Step 9: Raw vs SRM layer-wise encoding summary")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 9 directory : {STEP9_DIR}")
print(f"Networks: {len(NETWORK_NAMES)}")
print(f"Layer representations: {len(LAYER_LABELS)}")


# =========================
# Step 9. Helper functions
# =========================

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"


def fdr_bh(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    ok = np.isfinite(p)
    if not np.any(ok):
        return out
    p_ok = p[ok]
    order = np.argsort(p_ok)
    ranked = p_ok[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    out[ok] = restored
    return out


def sig_label(p):
    if not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required Step 9 input: {path}")
    return path


def annotate_panel(ax, df, value_col, label_prefix):
    mean_value = float(df[value_col].mean())
    best_idx = df[value_col].idxmax()
    best_row = df.loc[best_idx]
    text = (
        f"Mean r = {mean_value:.5f}\n"
        f"Best r = {float(best_row[value_col]):.5f}\n"
        f"Best = {best_row['roi_name']} {best_row['layer_label']}"
    )
    ax.text(
        0.02, 0.98, text,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#777777", alpha=0.86),
    )
    print(f"{label_prefix}: mean r = {mean_value:.5f}; best = {best_row['roi_name']} {best_row['layer_label']} r = {float(best_row[value_col]):.5f}")


def ordered_pivot(df, value_col):
    return df.pivot(index="roi_name", columns="layer_label", values=value_col).loc[NETWORK_NAMES, LAYER_LABELS]


# =========================
# Step 9. Load completed encoding summaries
# =========================

raw_group_path = require_file(STEP7B_DIR / "csv" / "step7b_layerwise_raw_group_encoding_summary.csv")
srm_group_path = require_file(STEP8C_DIR / "csv" / "step8c_layerwise_srm_group_encoding_summary.csv")
raw_subject_pair_path = require_file(STEP7C_DIR / "csv" / "step7c_raw_subject_encoding_pair_summary.csv")
srm_subject_pair_path = require_file(STEP8D_DIR / "csv" / "step8d_srm_subject_encoding_pair_summary.csv")
raw_subject_long_path = require_file(STEP7C_DIR / "csv" / "step7c_raw_subject_encoding_subject_summary.csv")
srm_subject_long_path = require_file(STEP8D_DIR / "csv" / "step8d_srm_subject_encoding_subject_summary.csv")

raw_group_df = pd.read_csv(raw_group_path)
srm_group_df = pd.read_csv(srm_group_path)
raw_subject_pair_df = pd.read_csv(raw_subject_pair_path)
srm_subject_pair_df = pd.read_csv(srm_subject_pair_path)
raw_subject_long_df = pd.read_csv(raw_subject_long_path)
srm_subject_long_df = pd.read_csv(srm_subject_long_path)

for name, df, expected_rows in [
    ("raw_group", raw_group_df, 833),
    ("srm_group", srm_group_df, 833),
    ("raw_subject_pair", raw_subject_pair_df, 833),
    ("srm_subject_pair", srm_subject_pair_df, 833),
]:
    if len(df) != expected_rows:
        raise ValueError(f"{name} expected {expected_rows} rows, found {len(df)}")
    if df["roi_name"].nunique() != len(NETWORK_NAMES):
        raise ValueError(f"{name} has unexpected network count: {df['roi_name'].nunique()}")
    if df["layer_label"].nunique() != len(LAYER_LABELS):
        raise ValueError(f"{name} has unexpected layer count: {df['layer_label'].nunique()}")

for name, df in [("raw_subject_long", raw_subject_long_df), ("srm_subject_long", srm_subject_long_df)]:
    expected_rows = 833 * 25
    if len(df) != expected_rows:
        raise ValueError(f"{name} expected {expected_rows} rows, found {len(df)}")

print("\nLoaded input tables:")
print(f"Raw group rows      : {len(raw_group_df)}")
print(f"SRM group rows      : {len(srm_group_df)}")
print(f"Raw participant rows: {len(raw_subject_pair_df)} pair rows; {len(raw_subject_long_df)} subject rows")
print(f"SRM participant rows: {len(srm_subject_pair_df)} pair rows; {len(srm_subject_long_df)} subject rows")


# =========================
# Step 9. Build merged comparison tables
# =========================

merge_keys = ["roi_name", "roi_target", "layer_label", "layer_index"]

group_compare_df = raw_group_df[merge_keys + [
    "best_alpha", "mean_r", "median_r", "max_r", "positive_voxel_fraction", "n_voxels", "voxel_corrs_path"
]].rename(columns={
    "best_alpha": "raw_best_alpha",
    "mean_r": "raw_group_mean_r",
    "median_r": "raw_group_median_r",
    "max_r": "raw_group_max_r",
    "positive_voxel_fraction": "raw_group_positive_voxel_fraction",
    "voxel_corrs_path": "raw_group_voxel_corrs_path",
}).merge(
    srm_group_df[merge_keys + ["best_alpha", "mean_r", "median_r", "max_r", "positive_voxel_fraction", "n_voxels", "voxel_corrs_path"]].rename(columns={
        "best_alpha": "srm_best_alpha",
        "mean_r": "srm_group_mean_r",
        "median_r": "srm_group_median_r",
        "max_r": "srm_group_max_r",
        "positive_voxel_fraction": "srm_group_positive_voxel_fraction",
        "voxel_corrs_path": "srm_group_voxel_corrs_path",
    }),
    on=merge_keys + ["n_voxels"],
    how="inner",
)
group_compare_df["delta_group_mean_r_srm_minus_raw"] = group_compare_df["srm_group_mean_r"] - group_compare_df["raw_group_mean_r"]
group_compare_df["delta_group_median_r_srm_minus_raw"] = group_compare_df["srm_group_median_r"] - group_compare_df["raw_group_median_r"]

group_long_df = pd.concat([
    raw_group_df.assign(condition="Raw + LLM", metric_value=raw_group_df["mean_r"]),
    srm_group_df.assign(condition="SRM + LLM", metric_value=srm_group_df["mean_r"]),
], ignore_index=True)

subject_pair_compare_df = raw_subject_pair_df[merge_keys + [
    "best_alpha", "n_subjects", "mean_subject_mean_r", "median_subject_mean_r", "min_subject_mean_r", "max_subject_mean_r", "positive_subject_fraction", "mean_voxel_corrs_path", "subject_summary_csv"
]].rename(columns={
    "best_alpha": "raw_best_alpha",
    "mean_subject_mean_r": "raw_subject_mean_r",
    "median_subject_mean_r": "raw_subject_median_r",
    "min_subject_mean_r": "raw_subject_min_r",
    "max_subject_mean_r": "raw_subject_max_r",
    "positive_subject_fraction": "raw_positive_subject_fraction",
    "mean_voxel_corrs_path": "raw_mean_voxel_corrs_path",
    "subject_summary_csv": "raw_subject_summary_csv",
}).merge(
    srm_subject_pair_df[merge_keys + ["best_alpha", "n_subjects", "mean_subject_mean_r", "median_subject_mean_r", "min_subject_mean_r", "max_subject_mean_r", "positive_subject_fraction", "mean_voxel_corrs_path", "subject_summary_csv"]].rename(columns={
        "best_alpha": "srm_best_alpha",
        "mean_subject_mean_r": "srm_subject_mean_r",
        "median_subject_mean_r": "srm_subject_median_r",
        "min_subject_mean_r": "srm_subject_min_r",
        "max_subject_mean_r": "srm_subject_max_r",
        "positive_subject_fraction": "srm_positive_subject_fraction",
        "mean_voxel_corrs_path": "srm_mean_voxel_corrs_path",
        "subject_summary_csv": "srm_subject_summary_csv",
    }),
    on=merge_keys + ["n_subjects"],
    how="inner",
)
subject_pair_compare_df["delta_subject_mean_r_srm_minus_raw"] = subject_pair_compare_df["srm_subject_mean_r"] - subject_pair_compare_df["raw_subject_mean_r"]
subject_pair_compare_df["delta_subject_median_r_srm_minus_raw"] = subject_pair_compare_df["srm_subject_median_r"] - subject_pair_compare_df["raw_subject_median_r"]

subject_pair_long_df = pd.concat([
    raw_subject_pair_df.assign(condition="Raw + LLM", metric_value=raw_subject_pair_df["mean_subject_mean_r"]),
    srm_subject_pair_df.assign(condition="SRM + LLM", metric_value=srm_subject_pair_df["mean_subject_mean_r"]),
], ignore_index=True)

subject_long_compare_df = raw_subject_long_df[merge_keys + ["subject_index", "mean_r", "median_r", "max_r", "positive_voxel_fraction"]].rename(columns={
    "mean_r": "raw_subject_mean_r",
    "median_r": "raw_subject_median_r",
    "max_r": "raw_subject_max_r",
    "positive_voxel_fraction": "raw_subject_positive_voxel_fraction",
}).merge(
    srm_subject_long_df[merge_keys + ["subject_index", "mean_r", "median_r", "max_r", "positive_voxel_fraction"]].rename(columns={
        "mean_r": "srm_subject_mean_r",
        "median_r": "srm_subject_median_r",
        "max_r": "srm_subject_max_r",
        "positive_voxel_fraction": "srm_subject_positive_voxel_fraction",
    }),
    on=merge_keys + ["subject_index"],
    how="inner",
)
subject_long_compare_df["delta_subject_mean_r_srm_minus_raw"] = subject_long_compare_df["srm_subject_mean_r"] - subject_long_compare_df["raw_subject_mean_r"]

if len(group_compare_df) != 833:
    raise ValueError(f"Group comparison table should have 833 rows, found {len(group_compare_df)}")
if len(subject_pair_compare_df) != 833:
    raise ValueError(f"Participant pair comparison table should have 833 rows, found {len(subject_pair_compare_df)}")
if len(subject_long_compare_df) != 833 * 25:
    raise ValueError(f"Participant long comparison table should have {833 * 25} rows, found {len(subject_long_compare_df)}")

# Per-network best rows for each level and condition.
def best_rows(df, value_col, label):
    out = (
        df.sort_values(["roi_name", value_col], ascending=[True, False])
        .groupby("roi_name", as_index=False)
        .head(1)
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )
    out.insert(4, "best_metric", label)
    return out

best_raw_group_df = best_rows(raw_group_df, "mean_r", "raw_group_mean_r")
best_srm_group_df = best_rows(srm_group_df, "mean_r", "srm_group_mean_r")
best_delta_group_df = best_rows(group_compare_df, "delta_group_mean_r_srm_minus_raw", "delta_group_mean_r_srm_minus_raw")
best_raw_subject_df = best_rows(raw_subject_pair_df, "mean_subject_mean_r", "raw_subject_mean_r")
best_srm_subject_df = best_rows(srm_subject_pair_df, "mean_subject_mean_r", "srm_subject_mean_r")
best_delta_subject_df = best_rows(subject_pair_compare_df, "delta_subject_mean_r_srm_minus_raw", "delta_subject_mean_r_srm_minus_raw")

# Subject-level paired tests for every network-layer pair.
test_rows = []
for (roi_name, layer_label), sub_df in subject_long_compare_df.groupby(["roi_name", "layer_label"], sort=False):
    raw_vals = sub_df.sort_values("subject_index")["raw_subject_mean_r"].to_numpy(dtype=float)
    srm_vals = sub_df.sort_values("subject_index")["srm_subject_mean_r"].to_numpy(dtype=float)
    t_stat, p_val = ttest_rel(srm_vals, raw_vals, nan_policy="omit")
    delta_vals = srm_vals - raw_vals
    test_rows.append({
        "roi_name": roi_name,
        "layer_label": layer_label,
        "layer_index": int(sub_df["layer_index"].iloc[0]),
        "n_subjects": int(np.sum(np.isfinite(delta_vals))),
        "raw_mean_r": float(np.nanmean(raw_vals)),
        "srm_mean_r": float(np.nanmean(srm_vals)),
        "delta_mean_r_srm_minus_raw": float(np.nanmean(delta_vals)),
        "delta_median_r_srm_minus_raw": float(np.nanmedian(delta_vals)),
        "positive_subject_fraction": float(np.mean(delta_vals > 0)),
        "t_statistic": float(t_stat),
        "p_value": float(p_val),
    })
subject_tests_df = pd.DataFrame(test_rows)
subject_tests_df["p_fdr"] = fdr_bh(subject_tests_df["p_value"].to_numpy(dtype=float))
subject_tests_df["significance_fdr"] = subject_tests_df["p_fdr"].apply(sig_label)

# Save CSV outputs.
group_compare_csv = CSV_DIR / "step9_group_raw_vs_srm_layerwise_comparison.csv"
subject_pair_compare_csv = CSV_DIR / "step9_participant_pair_raw_vs_srm_layerwise_comparison.csv"
subject_long_compare_csv = CSV_DIR / "step9_subject_level_raw_vs_srm_layerwise_comparison.csv"
subject_tests_csv = CSV_DIR / "step9_subject_level_paired_tests_by_network_layer.csv"
best_summary_csv = CSV_DIR / "step9_best_layer_summary_tables.csv"

group_compare_df.to_csv(group_compare_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
subject_pair_compare_df.to_csv(subject_pair_compare_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
subject_long_compare_df.to_csv(subject_long_compare_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
subject_tests_df.to_csv(subject_tests_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

best_summary_df = pd.concat([
    best_raw_group_df.assign(summary_type="best_raw_group"),
    best_srm_group_df.assign(summary_type="best_srm_group"),
    best_delta_group_df.assign(summary_type="best_delta_group"),
    best_raw_subject_df.assign(summary_type="best_raw_participant"),
    best_srm_subject_df.assign(summary_type="best_srm_participant"),
    best_delta_subject_df.assign(summary_type="best_delta_participant"),
], ignore_index=True, sort=False)
best_summary_df.to_csv(best_summary_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved Step 9 CSV tables:")
print(f"Group comparison       : {group_compare_csv}")
print(f"Participant pair table : {subject_pair_compare_csv}")
print(f"Subject-level table    : {subject_long_compare_csv}")
print(f"Paired tests           : {subject_tests_csv}")
print(f"Best-layer summary     : {best_summary_csv}")


# =========================
# Step 9. Paired line-profile figures with shared y-axis
# =========================

network_palette = dict(zip(NETWORK_NAMES, sns.color_palette("tab20", n_colors=len(NETWORK_NAMES))))

# Group-level Raw and SRM profile panels.
group_y_min = float(min(raw_group_df["mean_r"].min(), srm_group_df["mean_r"].min()))
group_y_max = float(max(raw_group_df["mean_r"].max(), srm_group_df["mean_r"].max()))
group_pad = max(0.005, (group_y_max - group_y_min) * 0.08)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
for ax, df, title, prefix in [
    (axes[0], raw_group_df, "Raw + LLM group-level encoding", "Group Raw+LLM"),
    (axes[1], srm_group_df, "SRM + LLM group-level encoding", "Group SRM+LLM"),
]:
    for roi_name in NETWORK_NAMES:
        roi_df = df[df["roi_name"] == roi_name].sort_values("layer_index")
        ax.plot(roi_df["layer_index"], roi_df["mean_r"], linewidth=1.6, alpha=0.88, color=network_palette[roi_name], label=roi_name)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation index (0 = embedding; 1-48 = transformer layers)")
    ax.set_ylim(group_y_min - group_pad, group_y_max + group_pad)
    annotate_panel(ax, df, "mean_r", prefix)
axes[0].set_ylabel("Mean held-out encoding r")
axes[1].legend(ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
fig.suptitle("Group-Level Layer-Wise Encoding Profiles", y=1.02)
fig.tight_layout()
group_profile_path = FIG_DIR / "step9_group_raw_vs_srm_layer_profiles_shared_y.png"
fig.savefig(group_profile_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# Participant-level Raw and SRM profile panels.
subject_y_min = float(min(raw_subject_pair_df["mean_subject_mean_r"].min(), srm_subject_pair_df["mean_subject_mean_r"].min()))
subject_y_max = float(max(raw_subject_pair_df["mean_subject_mean_r"].max(), srm_subject_pair_df["mean_subject_mean_r"].max()))
subject_pad = max(0.003, (subject_y_max - subject_y_min) * 0.08)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
for ax, df, title, prefix in [
    (axes[0], raw_subject_pair_df, "Raw + LLM participant-level encoding", "Participant Raw+LLM"),
    (axes[1], srm_subject_pair_df, "SRM + LLM participant-level encoding", "Participant SRM+LLM"),
]:
    for roi_name in NETWORK_NAMES:
        roi_df = df[df["roi_name"] == roi_name].sort_values("layer_index")
        ax.plot(roi_df["layer_index"], roi_df["mean_subject_mean_r"], linewidth=1.6, alpha=0.88, color=network_palette[roi_name], label=roi_name)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation index (0 = embedding; 1-48 = transformer layers)")
    ax.set_ylim(subject_y_min - subject_pad, subject_y_max + subject_pad)
    annotate_panel(ax, df, "mean_subject_mean_r", prefix)
axes[0].set_ylabel("Mean participant-level encoding r")
axes[1].legend(ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
fig.suptitle("Participant-Level Layer-Wise Encoding Profiles", y=1.02)
fig.tight_layout()
subject_profile_path = FIG_DIR / "step9_participant_raw_vs_srm_layer_profiles_shared_y.png"
fig.savefig(subject_profile_path, dpi=300, bbox_inches="tight")
plt.close(fig)


# =========================
# Step 9. Paired heatmaps with unified color scale
# =========================

raw_group_heat = ordered_pivot(raw_group_df, "mean_r")
srm_group_heat = ordered_pivot(srm_group_df, "mean_r")
heat_vmin = float(min(raw_group_heat.min().min(), srm_group_heat.min().min()))
heat_vmax = float(max(raw_group_heat.max().max(), srm_group_heat.max().max()))

fig, axes = plt.subplots(1, 2, figsize=(22, 7), sharey=True)
for ax, heat_df, title in [
    (axes[0], raw_group_heat, "Raw + LLM group-level mean r"),
    (axes[1], srm_group_heat, "SRM + LLM group-level mean r"),
]:
    sns.heatmap(
        heat_df,
        ax=ax,
        cmap="magma",
        vmin=heat_vmin,
        vmax=heat_vmax,
        linewidths=0.15,
        linecolor="white",
        cbar=ax is axes[1],
        cbar_kws={"label": "Mean held-out encoding r"},
    )
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation")
axes[0].set_ylabel("Network")
axes[1].set_ylabel("")
fig.suptitle("Group-Level Raw+LLM And SRM+LLM Encoding Heatmaps", y=1.02)
fig.tight_layout()
group_heatmap_path = FIG_DIR / "step9_group_raw_vs_srm_heatmaps_unified_color.png"
fig.savefig(group_heatmap_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# Participant-level heatmaps with unified color scale.
raw_subject_heat = ordered_pivot(raw_subject_pair_df, "mean_subject_mean_r")
srm_subject_heat = ordered_pivot(srm_subject_pair_df, "mean_subject_mean_r")
subject_heat_vmin = float(min(raw_subject_heat.min().min(), srm_subject_heat.min().min()))
subject_heat_vmax = float(max(raw_subject_heat.max().max(), srm_subject_heat.max().max()))

fig, axes = plt.subplots(1, 2, figsize=(22, 7), sharey=True)
for ax, heat_df, title in [
    (axes[0], raw_subject_heat, "Raw + LLM participant-level mean r"),
    (axes[1], srm_subject_heat, "SRM + LLM participant-level mean r"),
]:
    sns.heatmap(
        heat_df,
        ax=ax,
        cmap="magma",
        vmin=subject_heat_vmin,
        vmax=subject_heat_vmax,
        linewidths=0.15,
        linecolor="white",
        cbar=ax is axes[1],
        cbar_kws={"label": "Mean participant-level encoding r"},
    )
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation")
axes[0].set_ylabel("Network")
axes[1].set_ylabel("")
fig.suptitle("Participant-Level Raw+LLM And SRM+LLM Encoding Heatmaps", y=1.02)
fig.tight_layout()
subject_heatmap_path = FIG_DIR / "step9_participant_raw_vs_srm_heatmaps_unified_color.png"
fig.savefig(subject_heatmap_path, dpi=300, bbox_inches="tight")
plt.close(fig)


# =========================
# Step 9. Additional summary figures
# =========================

# Delta heatmaps: SRM minus Raw.
fig, axes = plt.subplots(1, 2, figsize=(22, 7), sharey=True)
group_delta_heat = ordered_pivot(group_compare_df, "delta_group_mean_r_srm_minus_raw")
subject_delta_heat = ordered_pivot(subject_pair_compare_df, "delta_subject_mean_r_srm_minus_raw")
delta_abs = float(max(np.nanmax(np.abs(group_delta_heat.to_numpy())), np.nanmax(np.abs(subject_delta_heat.to_numpy()))))
for ax, heat_df, title, cbar_label in [
    (axes[0], group_delta_heat, "Group-level SRM - Raw mean r", "Delta mean r"),
    (axes[1], subject_delta_heat, "Participant-level SRM - Raw mean r", "Delta mean r"),
]:
    sns.heatmap(
        heat_df,
        ax=ax,
        cmap="coolwarm",
        vmin=-delta_abs,
        vmax=delta_abs,
        center=0,
        linewidths=0.15,
        linecolor="white",
        cbar=ax is axes[1],
        cbar_kws={"label": cbar_label},
    )
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation")
axes[0].set_ylabel("Network")
axes[1].set_ylabel("")
fig.suptitle("SRM-Related Encoding Gain Across Networks And Layers", y=1.02)
fig.tight_layout()
delta_heatmap_path = FIG_DIR / "step9_group_and_participant_delta_heatmaps_shared_scale.png"
fig.savefig(delta_heatmap_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# Best-layer bars for group and participant levels.
network_best_compare_df = pd.DataFrame({"roi_name": NETWORK_NAMES}).merge(
    best_raw_group_df[["roi_name", "layer_label", "mean_r"]].rename(columns={"layer_label": "raw_group_best_layer", "mean_r": "raw_group_best_r"}),
    on="roi_name",
    how="left",
).merge(
    best_srm_group_df[["roi_name", "layer_label", "mean_r"]].rename(columns={"layer_label": "srm_group_best_layer", "mean_r": "srm_group_best_r"}),
    on="roi_name",
    how="left",
).merge(
    best_raw_subject_df[["roi_name", "layer_label", "mean_subject_mean_r"]].rename(columns={"layer_label": "raw_subject_best_layer", "mean_subject_mean_r": "raw_subject_best_r"}),
    on="roi_name",
    how="left",
).merge(
    best_srm_subject_df[["roi_name", "layer_label", "mean_subject_mean_r"]].rename(columns={"layer_label": "srm_subject_best_layer", "mean_subject_mean_r": "srm_subject_best_r"}),
    on="roi_name",
    how="left",
)
network_best_compare_df["group_best_delta_srm_minus_raw"] = network_best_compare_df["srm_group_best_r"] - network_best_compare_df["raw_group_best_r"]
network_best_compare_df["subject_best_delta_srm_minus_raw"] = network_best_compare_df["srm_subject_best_r"] - network_best_compare_df["raw_subject_best_r"]
network_best_compare_csv = CSV_DIR / "step9_network_best_layer_raw_vs_srm_comparison.csv"
network_best_compare_df.to_csv(network_best_compare_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

plot_group_best = network_best_compare_df.sort_values("srm_group_best_r", ascending=True)
y = np.arange(len(plot_group_best))
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
axes[0].barh(y - 0.18, plot_group_best["raw_group_best_r"], height=0.36, color="#8DA0CB", label="Raw + LLM")
axes[0].barh(y + 0.18, plot_group_best["srm_group_best_r"], height=0.36, color="#FC8D62", label="SRM + LLM")
axes[0].set_yticks(y)
axes[0].set_yticklabels(plot_group_best["roi_name"])
axes[0].set_title("Group-level best layer per network")
axes[0].set_xlabel("Best mean r")
axes[0].legend(frameon=False)

plot_subject_best = network_best_compare_df.set_index("roi_name").loc[plot_group_best["roi_name"]].reset_index()
axes[1].barh(y - 0.18, plot_subject_best["raw_subject_best_r"], height=0.36, color="#8DA0CB", label="Raw + LLM")
axes[1].barh(y + 0.18, plot_subject_best["srm_subject_best_r"], height=0.36, color="#FC8D62", label="SRM + LLM")
axes[1].set_title("Participant-level best layer per network")
axes[1].set_xlabel("Best mean r")
axes[1].legend(frameon=False)
fig.suptitle("Best Raw+LLM And SRM+LLM Layer-Wise Encoding By Network", y=1.02)
fig.tight_layout()
best_bar_path = FIG_DIR / "step9_best_layer_raw_vs_srm_group_and_participant_bars.png"
fig.savefig(best_bar_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# Scatter: raw vs SRM at matched network-layer combinations.
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, df, raw_col, srm_col, title in [
    (axes[0], group_compare_df, "raw_group_mean_r", "srm_group_mean_r", "Group-level network-layer pairs"),
    (axes[1], subject_pair_compare_df, "raw_subject_mean_r", "srm_subject_mean_r", "Participant-level network-layer pairs"),
]:
    ax.scatter(df[raw_col], df[srm_col], s=22, alpha=0.55, color="#4C72B0", edgecolor="none")
    xy_min = float(min(df[raw_col].min(), df[srm_col].min()))
    xy_max = float(max(df[raw_col].max(), df[srm_col].max()))
    pad = max(0.003, (xy_max - xy_min) * 0.08)
    ax.plot([xy_min - pad, xy_max + pad], [xy_min - pad, xy_max + pad], color="gray", linestyle="--", linewidth=1)
    ax.set_xlim(xy_min - pad, xy_max + pad)
    ax.set_ylim(xy_min - pad, xy_max + pad)
    ax.set_title(title)
    ax.set_xlabel("Raw + LLM mean r")
    ax.set_ylabel("SRM + LLM mean r")
fig.suptitle("Matched Raw+LLM vs SRM+LLM Encoding Across Network-Layer Pairs", y=1.02)
fig.tight_layout()
scatter_path = FIG_DIR / "step9_raw_vs_srm_matched_network_layer_scatter.png"
fig.savefig(scatter_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("\nSaved Step 9 figures:")
for p in [
    group_profile_path,
    subject_profile_path,
    group_heatmap_path,
    subject_heatmap_path,
    delta_heatmap_path,
    best_bar_path,
    scatter_path,
]:
    print(p)


# =========================
# Step 9. Final summary JSON
# =========================

summary = {
    "project_dir": str(PROJECT_DIR),
    "step9_dir": str(STEP9_DIR),
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layer_representations": int(len(LAYER_LABELS)),
    "group_rows": int(len(group_compare_df)),
    "participant_pair_rows": int(len(subject_pair_compare_df)),
    "subject_level_rows": int(len(subject_long_compare_df)),
    "group_raw_mean_r_all_pairs": float(raw_group_df["mean_r"].mean()),
    "group_srm_mean_r_all_pairs": float(srm_group_df["mean_r"].mean()),
    "group_delta_mean_r_all_pairs": float(group_compare_df["delta_group_mean_r_srm_minus_raw"].mean()),
    "participant_raw_mean_r_all_pairs": float(raw_subject_pair_df["mean_subject_mean_r"].mean()),
    "participant_srm_mean_r_all_pairs": float(srm_subject_pair_df["mean_subject_mean_r"].mean()),
    "participant_delta_mean_r_all_pairs": float(subject_pair_compare_df["delta_subject_mean_r_srm_minus_raw"].mean()),
    "best_group_raw": {
        "roi_name": str(raw_group_df.loc[raw_group_df["mean_r"].idxmax(), "roi_name"]),
        "layer_label": str(raw_group_df.loc[raw_group_df["mean_r"].idxmax(), "layer_label"]),
        "mean_r": float(raw_group_df["mean_r"].max()),
    },
    "best_group_srm": {
        "roi_name": str(srm_group_df.loc[srm_group_df["mean_r"].idxmax(), "roi_name"]),
        "layer_label": str(srm_group_df.loc[srm_group_df["mean_r"].idxmax(), "layer_label"]),
        "mean_r": float(srm_group_df["mean_r"].max()),
    },
    "best_participant_raw": {
        "roi_name": str(raw_subject_pair_df.loc[raw_subject_pair_df["mean_subject_mean_r"].idxmax(), "roi_name"]),
        "layer_label": str(raw_subject_pair_df.loc[raw_subject_pair_df["mean_subject_mean_r"].idxmax(), "layer_label"]),
        "mean_r": float(raw_subject_pair_df["mean_subject_mean_r"].max()),
    },
    "best_participant_srm": {
        "roi_name": str(srm_subject_pair_df.loc[srm_subject_pair_df["mean_subject_mean_r"].idxmax(), "roi_name"]),
        "layer_label": str(srm_subject_pair_df.loc[srm_subject_pair_df["mean_subject_mean_r"].idxmax(), "layer_label"]),
        "mean_r": float(srm_subject_pair_df["mean_subject_mean_r"].max()),
    },
    "outputs": {
        "csv": [
            str(group_compare_csv),
            str(subject_pair_compare_csv),
            str(subject_long_compare_csv),
            str(subject_tests_csv),
            str(best_summary_csv),
            str(network_best_compare_csv),
        ],
        "figures": [
            str(group_profile_path),
            str(subject_profile_path),
            str(group_heatmap_path),
            str(subject_heatmap_path),
            str(delta_heatmap_path),
            str(best_bar_path),
            str(scatter_path),
        ],
    },
}
summary_json = JSON_DIR / "step9_raw_vs_srm_layerwise_encoding_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nStep 9 final summary:")
print(json.dumps(summary, indent=2))
print("\nStep 9 completed successfully.")


# Step 10: Layer-Wise Brain Mapping And Layer-Depth Preference Maps

This step projects group-level layer-wise encoding results back into brain space. It reads the completed Raw+LLM and SRM+LLM voxel-wise encoding maps from Step 7B and Step 8C, together with the Step 9 layer-wise summary table.

In addition to standard Raw, SRM, and SRM-minus-Raw brain maps, this step creates layer-depth preference maps. For each voxel, the map identifies which GPT-2 transformer layer depth gives the strongest SRM+LLM encoding response, then visualizes that preferred layer depth as a percentage from early to late transformer layers.

The output includes per-network best-layer maps and summary montages of the strongest network-layer effects. This cell does not refit any encoding model.


In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

from nilearn import image
from nilearn.plotting import plot_glass_brain, plot_stat_map
from tqdm.auto import tqdm


# =========================
# Step 10. Project setup
# =========================

BASE_DIR = Path(r"YOUR_SNL2026_ROOT")
PROJECT_ROOT = BASE_DIR / "Layer_wise_snl_only"
ENCODING_RUN_NAME = "SNL_layerwise_encoding_fixed_srm50"
PROJECT_DIR = PROJECT_ROOT / ENCODING_RUN_NAME
SRM_RUNS_DIR = BASE_DIR / "runs"

STEP9_DIR = PROJECT_DIR / "step9_raw_vs_srm_layerwise_encoding_comparison"
STEP10_DIR = PROJECT_DIR / "step10_layerwise_brain_mapping"

CSV_DIR = STEP10_DIR / "csv"
JSON_DIR = STEP10_DIR / "json"
NII_DIR = STEP10_DIR / "nii"
FIG_BEST_DIR = STEP10_DIR / "figures_best_by_network"
FIG_DEPTH_DIR = STEP10_DIR / "figures_layer_depth_preference"
FIG_DEPTH_XYZ_DIR = STEP10_DIR / "figures_layer_depth_directional_slices"
FIG_COMBINED_DEPTH_DIR = STEP10_DIR / "figures_combined_top_network_depth_map"
FIG_MONTAGE_DIR = STEP10_DIR / "figures_summary_montage"
FIG_ALL_LAYER_GLASS_DIR = STEP10_DIR / "figures_all_layer_glass_optional"

for d in [
    STEP10_DIR,
    CSV_DIR,
    JSON_DIR,
    NII_DIR,
    FIG_BEST_DIR,
    FIG_DEPTH_DIR,
    FIG_DEPTH_XYZ_DIR,
    FIG_COMBINED_DEPTH_DIR,
    FIG_MONTAGE_DIR,
    FIG_ALL_LAYER_GLASS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
    "TempPar",
]
LAYER_LABELS = [f"layer_{i:02d}" for i in range(49)]
TRANSFORMER_LAYER_LABELS = [f"layer_{i:02d}" for i in range(1, 49)]

# Full all-layer figures are optional because 17 networks x 49 layers x 3 map types is large.
MAKE_ALL_LAYER_NIFTI = True
MAKE_ALL_LAYER_GLASS = False
MAKE_BEST_NETWORK_FIGURES = True
MAKE_LAYER_DEPTH_PREFERENCE_FIGURES = True
MAKE_DEPTH_DIRECTIONAL_SLICES = True
MAKE_COMBINED_TOP_NETWORK_DEPTH_MAP = True
MAKE_TOP_SUMMARY_MONTAGES = True
TOP_N_SUMMARY_MAPS = 8
TOP_N_COMBINED_DEPTH_NETWORKS = 8

EPS = 1e-6
PSEUDO_ZERO_THRESHOLD = 1e-5
DEPTH_VALID_PERCENTILE = 70

print("Step 10: Layer-wise brain mapping and layer-depth preference maps")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 9 directory : {STEP9_DIR}")
print(f"Step 10 directory: {STEP10_DIR}")
print(f"Networks: {len(NETWORK_NAMES)}")
print(f"Layer representations: {len(LAYER_LABELS)}")
print(f"MAKE_ALL_LAYER_NIFTI: {MAKE_ALL_LAYER_NIFTI}")
print(f"MAKE_ALL_LAYER_GLASS: {MAKE_ALL_LAYER_GLASS}")

start_total = time.time()


# =========================
# Step 10. Helper functions
# =========================

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"


def parse_layer_index(layer_label):
    return int(str(layer_label).split("_")[1])


def layer_depth_percent(layer_index):
    """
    Log-scaled transformer-layer depth.

    This keeps the ordering from early to late GPT-2 XL transformer layers, but
    expands early/middle-layer differences and compresses the deepest layers.
    layer_00 is the embedding representation and is not assigned a transformer
    depth percentage.
    """
    layer_index = int(layer_index)
    if layer_index <= 0:
        return np.nan
    return float(np.log1p(layer_index) / np.log1p(48) * 100.0)


def layer_depth_bin(layer_index):
    layer_index = int(layer_index)
    if layer_index == 0:
        return "embedding"
    pct = layer_depth_percent(layer_index)
    if pct <= 33.33333:
        return "early_layers_log_depth"
    if pct <= 66.66667:
        return "early_to_middle_layers_log_depth"
    if pct <= 85.0:
        return "middle_to_late_layers_log_depth"
    return "late_layers_log_depth"


def finite_nonzero_values(data, eps=PSEUDO_ZERO_THRESHOLD):
    data = np.asarray(data, dtype=float)
    return data[np.isfinite(data) & (np.abs(data) > eps)]


def robust_abs_percentile(values, percentile=99, fallback=0.00001):
    values = np.asarray(values, dtype=float).ravel()
    values = np.abs(values[np.isfinite(values)])
    values = values[values > PSEUDO_ZERO_THRESHOLD]
    if values.size == 0:
        return float(fallback)
    out = float(np.nanpercentile(values, percentile))
    if not np.isfinite(out) or out <= 0:
        return float(fallback)
    return out


def robust_positive_percentile(values, percentile=99, fallback=0.00001):
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]
    values = values[values > PSEUDO_ZERO_THRESHOLD]
    if values.size == 0:
        return float(fallback)
    out = float(np.nanpercentile(values, percentile))
    if not np.isfinite(out) or out <= 0:
        return float(fallback)
    return out


def robust_threshold(values, percentile=75, fallback=PSEUDO_ZERO_THRESHOLD):
    values = np.asarray(values, dtype=float).ravel()
    values = np.abs(values[np.isfinite(values)])
    values = values[values > PSEUDO_ZERO_THRESHOLD]
    if values.size == 0:
        return float(fallback)
    out = float(np.nanpercentile(values, percentile))
    if not np.isfinite(out) or out <= 0:
        return float(fallback)
    return max(out, float(fallback))


def vector_to_nifti(values, mask_bool, ref_nii):
    values = np.asarray(values, dtype=np.float32).ravel()
    n_mask = int(np.sum(mask_bool))
    if values.shape[0] != n_mask:
        raise ValueError(f"Vector length {values.shape[0]} does not match mask voxel count {n_mask}.")
    out = np.zeros(mask_bool.shape, dtype=np.float32)
    out[mask_bool] = values
    return nib.Nifti1Image(out, ref_nii.affine, ref_nii.header)


def format_colorbar_decimal(disp, digits=5):
    """Force ordinary decimal colorbar labels and hide any scientific offset text."""
    try:
        formatter = FormatStrFormatter(f"%.{digits}f")
        disp._cbar.ax.yaxis.set_major_formatter(formatter)
        disp._cbar.ax.xaxis.set_major_formatter(formatter)
        disp._cbar.ax.yaxis.offsetText.set_visible(False)
        disp._cbar.ax.xaxis.offsetText.set_visible(False)
        disp._cbar.update_ticks()
    except Exception:
        pass


def save_glass_map(img_path, out_path, title, cmap, vmin, vmax, threshold, symmetric_cbar=False, cbar_digits=5):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    disp = plot_glass_brain(
        str(img_path),
        display_mode="lyrz",
        colorbar=True,
        cmap=cmap,
        plot_abs=False,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        symmetric_cbar=symmetric_cbar,
        cbar_tick_format=f"%.{cbar_digits}f",
        title=title,
    )
    format_colorbar_decimal(disp, digits=cbar_digits)
    disp.savefig(out_path, dpi=220, bbox_inches="tight", pad_inches=0.05)
    plt.close()


def format_depth_colorbar(disp):
    """Use fixed 0-100 integer ticks and never show scientific notation."""
    try:
        formatter = FormatStrFormatter("%.0f")
        disp._cbar.set_ticks([0, 25, 50, 75, 100])
        disp._cbar.ax.yaxis.set_major_formatter(formatter)
        disp._cbar.ax.xaxis.set_major_formatter(formatter)
        disp._cbar.ax.yaxis.offsetText.set_visible(False)
        disp._cbar.ax.xaxis.offsetText.set_visible(False)
        disp._cbar.ax.set_ylabel("")
        disp._cbar.ax.set_xlabel("Best Encoding Layer (%)", labelpad=8)
        disp._cbar.update_ticks()
    except Exception:
        pass


def save_depth_glass_map(img_path, out_path, title, threshold=PSEUDO_ZERO_THRESHOLD, annotation_text=None):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    disp = plot_glass_brain(
        str(img_path),
        display_mode="lyrz",
        colorbar=True,
        cmap="inferno",
        plot_abs=False,
        threshold=threshold,
        vmin=0,
        vmax=100,
        black_bg=False,
        cbar_tick_format="%.0f",
        title=title,
    )
    format_depth_colorbar(disp)
    if annotation_text:
        fig = plt.gcf()
        fig.text(
            0.015, 0.015, annotation_text,
            ha="left", va="bottom", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#777777", alpha=0.88),
        )
    disp.savefig(out_path, dpi=220, bbox_inches="tight", pad_inches=0.05)
    plt.close()


def save_depth_directional_slice(img_path, out_path, title, display_mode, threshold=PSEUDO_ZERO_THRESHOLD):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    disp = plot_stat_map(
        str(img_path),
        display_mode=display_mode,
        cut_coords=7,
        colorbar=True,
        cmap="inferno",
        threshold=threshold,
        vmin=0,
        vmax=100,
        black_bg=False,
        symmetric_cbar=False,
        cbar_tick_format="%.0f",
        title=title,
    )
    format_depth_colorbar(disp)
    disp.savefig(out_path, dpi=240, bbox_inches="tight", pad_inches=0.05)
    plt.close()


def rebuild_roi_mask_and_ref(roi_name, expected_voxels):
    step1_dir = SRM_RUNS_DIR / f"{roi_name}_400_parcels_fixed_srm50" / "step1_setup_roi_cleaned"
    config_path = step1_dir / "step1_config.json"
    manifest_path = step1_dir / "csv" / "subject_file_manifest.csv"
    subjects_path = step1_dir / "csv" / "subjects_retained.csv"
    roi_mask_path = step1_dir / "selected_schaefer400_roi_mask.nii.gz"

    for p in [config_path, manifest_path, subjects_path, roi_mask_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing mask reconstruction input for {roi_name}: {p}")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    manifest_df = pd.read_csv(manifest_path)
    subjects = pd.read_csv(subjects_path)["subject"].tolist()
    if not subjects:
        raise ValueError(f"No retained subjects found for {roi_name}.")

    example_subject = subjects[0]
    example_bold_path = Path(manifest_df.loc[manifest_df["subject"] == example_subject, "bold_path"].iloc[0])
    if not example_bold_path.exists():
        raise FileNotFoundError(f"Example BOLD file not found for {roi_name}: {example_bold_path}")

    roi_mask_atlas_nii = nib.load(str(roi_mask_path))
    bold_nii = nib.load(str(example_bold_path))
    bold_data = bold_nii.get_fdata(dtype=np.float32)
    story_start_tr = int(config["story_start_tr"])
    story_end_tr = int(config["story_end_tr"])
    bold_crop = bold_data[..., story_start_tr:story_end_tr]
    bold_ref_nii = nib.Nifti1Image(bold_crop[..., 0], bold_nii.affine, bold_nii.header)

    roi_resampled_nii = image.resample_to_img(
        roi_mask_atlas_nii,
        bold_ref_nii,
        interpolation="nearest",
        force_resample=True,
        copy_header=True,
    )
    mask_bool = roi_resampled_nii.get_fdata() > 0
    mask_count = int(np.sum(mask_bool))
    if mask_count != int(expected_voxels):
        raise ValueError(f"{roi_name}: mask voxel count {mask_count} does not match expected {expected_voxels}.")

    return mask_bool, bold_ref_nii, {
        "roi_name": roi_name,
        "example_subject": example_subject,
        "example_bold_path": str(example_bold_path),
        "mask_voxel_count": mask_count,
        "story_start_tr": story_start_tr,
        "story_end_tr": story_end_tr,
    }


# =========================
# Step 10. Load Step 9 comparison table
# =========================

GROUP_COMPARISON_CSV = STEP9_DIR / "csv" / "step9_group_raw_vs_srm_layerwise_comparison.csv"
NETWORK_BEST_CSV = STEP9_DIR / "csv" / "step9_network_best_layer_raw_vs_srm_comparison.csv"

for p in [GROUP_COMPARISON_CSV, NETWORK_BEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Step 10 input: {p}")

group_df = pd.read_csv(GROUP_COMPARISON_CSV)
network_best_df = pd.read_csv(NETWORK_BEST_CSV)

required_cols = {
    "roi_name", "roi_target", "layer_label", "layer_index", "n_voxels",
    "raw_group_mean_r", "srm_group_mean_r", "delta_group_mean_r_srm_minus_raw",
    "raw_group_voxel_corrs_path", "srm_group_voxel_corrs_path",
}
missing_cols = required_cols.difference(group_df.columns)
if missing_cols:
    raise ValueError(f"Step 9 group comparison CSV is missing columns: {sorted(missing_cols)}")

if len(group_df) != len(NETWORK_NAMES) * len(LAYER_LABELS):
    raise ValueError(f"Expected {len(NETWORK_NAMES) * len(LAYER_LABELS)} group rows, found {len(group_df)}")

for col in ["raw_group_voxel_corrs_path", "srm_group_voxel_corrs_path"]:
    missing_paths = [p for p in group_df[col].tolist() if not Path(p).exists()]
    if missing_paths:
        raise FileNotFoundError(f"Missing voxel-correlation files in {col}. First missing: {missing_paths[0]}")

group_df = group_df.copy()
group_df["layer_depth_percent"] = group_df["layer_index"].apply(layer_depth_percent)
group_df["log_layer_depth_percent"] = group_df["layer_depth_percent"]
group_df["layer_depth_bin"] = group_df["layer_index"].apply(layer_depth_bin)

# Best network-layer rows for summary brain maps.
best_srm_rows = (
    group_df.sort_values(["roi_name", "srm_group_mean_r"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("srm_group_mean_r", ascending=False)
    .reset_index(drop=True)
)
best_delta_rows = (
    group_df.sort_values(["roi_name", "delta_group_mean_r_srm_minus_raw"], ascending=[True, False])
    .groupby("roi_name", as_index=False)
    .head(1)
    .sort_values("delta_group_mean_r_srm_minus_raw", ascending=False)
    .reset_index(drop=True)
)

top_srm_rows = group_df.sort_values("srm_group_mean_r", ascending=False).head(TOP_N_SUMMARY_MAPS).reset_index(drop=True)
top_delta_rows = group_df.sort_values("delta_group_mean_r_srm_minus_raw", ascending=False).head(TOP_N_SUMMARY_MAPS).reset_index(drop=True)

print("\nLoaded Step 9 group comparison:")
print(f"Rows: {len(group_df)}")
print(f"Raw mean r range  : {fmt_float(group_df['raw_group_mean_r'].min())} to {fmt_float(group_df['raw_group_mean_r'].max())}")
print(f"SRM mean r range  : {fmt_float(group_df['srm_group_mean_r'].min())} to {fmt_float(group_df['srm_group_mean_r'].max())}")
print(f"Delta mean r range: {fmt_float(group_df['delta_group_mean_r_srm_minus_raw'].min())} to {fmt_float(group_df['delta_group_mean_r_srm_minus_raw'].max())}")
print("\nTop SRM group rows:")
print(top_srm_rows[["roi_name", "layer_label", "layer_depth_percent", "srm_group_mean_r"]].to_string(index=False, float_format=lambda x: f"{x:.5f}"))
print("\nTop delta group rows:")
print(top_delta_rows[["roi_name", "layer_label", "layer_depth_percent", "delta_group_mean_r_srm_minus_raw"]].to_string(index=False, float_format=lambda x: f"{x:.5f}"))


# =========================
# Step 10. Global plotting ranges
# =========================

raw_sample_values = []
srm_sample_values = []
delta_sample_values = []

for _, row in tqdm(group_df.iterrows(), total=len(group_df), desc="Scan voxel ranges"):
    raw_vals = np.load(row["raw_group_voxel_corrs_path"]).astype(np.float32)
    srm_vals = np.load(row["srm_group_voxel_corrs_path"]).astype(np.float32)
    if raw_vals.shape != srm_vals.shape:
        raise ValueError(f"Voxel shape mismatch for {row['roi_name']} {row['layer_label']}: {raw_vals.shape} vs {srm_vals.shape}")
    raw_sample_values.append(raw_vals)
    srm_sample_values.append(srm_vals)
    delta_sample_values.append(srm_vals - raw_vals)

raw_all = np.concatenate(raw_sample_values)
srm_all = np.concatenate(srm_sample_values)
delta_all = np.concatenate(delta_sample_values)

encoding_vmax = robust_positive_percentile(np.concatenate([raw_all, srm_all]), percentile=99)
delta_absmax = robust_abs_percentile(delta_all, percentile=99)
encoding_threshold = robust_threshold(np.concatenate([raw_all, srm_all]), percentile=75)
delta_threshold = robust_threshold(delta_all, percentile=70)

range_summary = {
    "encoding_vmin": 0.0,
    "encoding_vmax": float(encoding_vmax),
    "delta_vmin": float(-delta_absmax),
    "delta_vmax": float(delta_absmax),
    "encoding_threshold": float(encoding_threshold),
    "delta_threshold": float(delta_threshold),
    "depth_vmin": 0.0,
    "depth_vmax": 100.0,
    "depth_scale": "log1p(layer_index) / log1p(48) * 100",
    "depth_valid_percentile": float(DEPTH_VALID_PERCENTILE),
}

range_csv = CSV_DIR / "step10_brain_map_color_ranges.csv"
pd.DataFrame([range_summary]).to_csv(range_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nPlotting ranges:")
for k, v in range_summary.items():
    if isinstance(v, (int, float, np.integer, np.floating)):
        print(f"{k}: {fmt_float(v)}")
    else:
        print(f"{k}: {v}")

# Free memory after global range scan.
del raw_sample_values, srm_sample_values, delta_sample_values, raw_all, srm_all, delta_all


# =========================
# Step 10. Build NIfTI maps and per-network figures
# =========================

mask_cache = {}
manifest_rows = []
depth_rows = []
mask_summary_rows = []

for roi_name in tqdm(NETWORK_NAMES, desc="Create brain maps by network"):
    roi_df = group_df[group_df["roi_name"] == roi_name].sort_values("layer_index").reset_index(drop=True)
    if roi_df.empty:
        raise ValueError(f"No Step 9 rows found for {roi_name}.")
    expected_voxels = int(roi_df["n_voxels"].iloc[0])

    mask_bool, ref_nii, mask_info = rebuild_roi_mask_and_ref(roi_name, expected_voxels)
    mask_cache[roi_name] = (mask_bool, ref_nii)
    mask_summary_rows.append(mask_info)

    roi_nii_dir = NII_DIR / roi_name
    roi_nii_dir.mkdir(parents=True, exist_ok=True)

    # Stack transformer-layer SRM voxel maps for layer-depth preference maps.
    transformer_srm_maps = []
    transformer_depths = []

    for _, row in roi_df.iterrows():
        layer_label = row["layer_label"]
        layer_index = int(row["layer_index"])
        raw_vals = np.load(row["raw_group_voxel_corrs_path"]).astype(np.float32)
        srm_vals = np.load(row["srm_group_voxel_corrs_path"]).astype(np.float32)
        delta_vals = (srm_vals - raw_vals).astype(np.float32)

        if raw_vals.shape[0] != expected_voxels or srm_vals.shape[0] != expected_voxels:
            raise ValueError(f"{roi_name} {layer_label}: voxel count mismatch.")

        raw_img_path = roi_nii_dir / f"{roi_name}_{layer_label}_raw_group_encoding_r.nii.gz"
        srm_img_path = roi_nii_dir / f"{roi_name}_{layer_label}_srm_group_encoding_r.nii.gz"
        delta_img_path = roi_nii_dir / f"{roi_name}_{layer_label}_delta_group_encoding_r_srm_minus_raw.nii.gz"

        if MAKE_ALL_LAYER_NIFTI:
            nib.save(vector_to_nifti(raw_vals, mask_bool, ref_nii), raw_img_path)
            nib.save(vector_to_nifti(srm_vals, mask_bool, ref_nii), srm_img_path)
            nib.save(vector_to_nifti(delta_vals, mask_bool, ref_nii), delta_img_path)

        if layer_index > 0:
            transformer_srm_maps.append(srm_vals)
            transformer_depths.append(layer_depth_percent(layer_index))

        if MAKE_ALL_LAYER_GLASS:
            fig_dir = FIG_ALL_LAYER_GLASS_DIR / roi_name / layer_label
            save_glass_map(
                srm_img_path,
                fig_dir / f"{roi_name}_{layer_label}_srm_group_encoding_r_glass.png",
                f"{roi_name} {layer_label}: SRM+LLM group encoding r",
                cmap="viridis",
                vmin=0,
                vmax=encoding_vmax,
                threshold=encoding_threshold,
            )
            save_glass_map(
                delta_img_path,
                fig_dir / f"{roi_name}_{layer_label}_delta_group_encoding_r_glass.png",
                f"{roi_name} {layer_label}: SRM+LLM - Raw+LLM encoding r",
                cmap="coolwarm",
                vmin=-delta_absmax,
                vmax=delta_absmax,
                threshold=delta_threshold,
                symmetric_cbar=False,
            )

        manifest_rows.append({
            "roi_name": roi_name,
            "roi_target": row["roi_target"],
            "layer_label": layer_label,
            "layer_index": int(layer_index),
            "layer_depth_percent": float(layer_depth_percent(layer_index)) if layer_index > 0 else np.nan,
            "layer_depth_bin": layer_depth_bin(layer_index),
            "raw_group_mean_r": float(row["raw_group_mean_r"]),
            "srm_group_mean_r": float(row["srm_group_mean_r"]),
            "delta_group_mean_r_srm_minus_raw": float(row["delta_group_mean_r_srm_minus_raw"]),
            "n_voxels": int(expected_voxels),
            "raw_nifti_path": str(raw_img_path),
            "srm_nifti_path": str(srm_img_path),
            "delta_nifti_path": str(delta_img_path),
        })

    # Voxel-wise layer-depth preference map across ALL transformer layers.
    # For each voxel, we scan layer_01...layer_48 and keep the layer whose
    # SRM+LLM encoding r is maximal for that voxel. Therefore this is not one
    # selected layer for the whole network; different voxels can prefer early,
    # middle, or late layers within the same network.
    srm_stack = np.vstack(transformer_srm_maps).astype(np.float32)
    depth_arr = np.asarray(transformer_depths, dtype=np.float32)
    best_idx = np.nanargmax(srm_stack, axis=0)
    best_depth_values = depth_arr[best_idx].astype(np.float32)
    best_layer_indices = best_idx + 1
    best_srm_values = np.nanmax(srm_stack, axis=0).astype(np.float32)

    valid_positive = best_srm_values[np.isfinite(best_srm_values) & (best_srm_values > PSEUDO_ZERO_THRESHOLD)]
    if valid_positive.size > 0:
        depth_valid_threshold = float(np.nanpercentile(valid_positive, DEPTH_VALID_PERCENTILE))
    else:
        depth_valid_threshold = PSEUDO_ZERO_THRESHOLD
    depth_map_values = np.zeros_like(best_depth_values, dtype=np.float32)
    depth_map_values[best_srm_values >= depth_valid_threshold] = best_depth_values[best_srm_values >= depth_valid_threshold]

    depth_img_path = roi_nii_dir / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_percent.nii.gz"
    best_srm_value_img_path = roi_nii_dir / f"{roi_name}_voxelwise_best_srm_encoding_r_across_transformer_layers.nii.gz"
    best_layer_index_img_path = roi_nii_dir / f"{roi_name}_voxelwise_preferred_srm_layer_index_across_transformer_layers.nii.gz"
    nib.save(vector_to_nifti(depth_map_values, mask_bool, ref_nii), depth_img_path)
    nib.save(vector_to_nifti(best_srm_values, mask_bool, ref_nii), best_srm_value_img_path)
    nib.save(vector_to_nifti(best_layer_indices.astype(np.float32), mask_bool, ref_nii), best_layer_index_img_path)

    depth_fig_path = FIG_DEPTH_DIR / roi_name / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_glass.png"
    depth_x_fig_path = FIG_DEPTH_XYZ_DIR / roi_name / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_sagittal_x.png"
    depth_y_fig_path = FIG_DEPTH_XYZ_DIR / roi_name / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_coronal_y.png"
    depth_z_fig_path = FIG_DEPTH_XYZ_DIR / roi_name / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_axial_z.png"
    if MAKE_LAYER_DEPTH_PREFERENCE_FIGURES:
        save_depth_glass_map(
            depth_img_path,
            depth_fig_path,
            f"{roi_name}: Voxel-wise Preferred Layer Depth",
            threshold=PSEUDO_ZERO_THRESHOLD,
        )
    if MAKE_DEPTH_DIRECTIONAL_SLICES:
        save_depth_directional_slice(
            depth_img_path,
            depth_x_fig_path,
            f"{roi_name}: Preferred Layer Depth (sagittal)",
            display_mode="x",
            threshold=PSEUDO_ZERO_THRESHOLD,
        )
        save_depth_directional_slice(
            depth_img_path,
            depth_y_fig_path,
            f"{roi_name}: Preferred Layer Depth (coronal)",
            display_mode="y",
            threshold=PSEUDO_ZERO_THRESHOLD,
        )
        save_depth_directional_slice(
            depth_img_path,
            depth_z_fig_path,
            f"{roi_name}: Preferred Layer Depth (axial)",
            display_mode="z",
            threshold=PSEUDO_ZERO_THRESHOLD,
        )

    depth_rows.append({
        "roi_name": roi_name,
        "n_voxels": int(expected_voxels),
        "depth_valid_threshold_best_srm_r": float(depth_valid_threshold),
        "visible_voxel_fraction": float(np.mean(depth_map_values > 0)),
        "depth_map_definition": "For each visible voxel, color is the log-scaled depth percent of the transformer layer with maximal SRM+LLM encoding r across layer_01...layer_48.",
        "mean_visible_depth_percent": float(np.nanmean(depth_map_values[depth_map_values > 0])) if np.any(depth_map_values > 0) else np.nan,
        "median_visible_depth_percent": float(np.nanmedian(depth_map_values[depth_map_values > 0])) if np.any(depth_map_values > 0) else np.nan,
        "depth_nifti_path": str(depth_img_path),
        "best_srm_value_nifti_path": str(best_srm_value_img_path),
        "best_layer_index_nifti_path": str(best_layer_index_img_path),
        "depth_figure_path": str(depth_fig_path),
        "depth_sagittal_x_figure_path": str(depth_x_fig_path),
        "depth_coronal_y_figure_path": str(depth_y_fig_path),
        "depth_axial_z_figure_path": str(depth_z_fig_path),
    })

    if MAKE_BEST_NETWORK_FIGURES:
        best_srm_row = best_srm_rows[best_srm_rows["roi_name"] == roi_name].iloc[0]
        best_delta_row = best_delta_rows[best_delta_rows["roi_name"] == roi_name].iloc[0]
        best_srm_layer = best_srm_row["layer_label"]
        best_delta_layer = best_delta_row["layer_label"]

        best_srm_img = roi_nii_dir / f"{roi_name}_{best_srm_layer}_srm_group_encoding_r.nii.gz"
        best_delta_img = roi_nii_dir / f"{roi_name}_{best_delta_layer}_delta_group_encoding_r_srm_minus_raw.nii.gz"

        save_glass_map(
            best_srm_img,
            FIG_BEST_DIR / roi_name / f"{roi_name}_best_srm_group_encoding_layer_glass.png",
            f"{roi_name}: best SRM+LLM layer {best_srm_layer} (r={best_srm_row['srm_group_mean_r']:.5f})",
            cmap="viridis",
            vmin=0,
            vmax=encoding_vmax,
            threshold=encoding_threshold,
        )
        save_glass_map(
            best_delta_img,
            FIG_BEST_DIR / roi_name / f"{roi_name}_best_delta_srm_minus_raw_layer_glass.png",
            f"{roi_name}: best SRM-minus-Raw layer {best_delta_layer} (delta={best_delta_row['delta_group_mean_r_srm_minus_raw']:.5f})",
            cmap="coolwarm",
            vmin=-delta_absmax,
            vmax=delta_absmax,
            threshold=delta_threshold,
            symmetric_cbar=False,
        )

manifest_df = pd.DataFrame(manifest_rows)
depth_df = pd.DataFrame(depth_rows)
mask_summary_df = pd.DataFrame(mask_summary_rows)

manifest_csv = CSV_DIR / "step10_layerwise_brain_map_manifest.csv"
depth_csv = CSV_DIR / "step10_log_layer_depth_preference_summary.csv"
mask_csv = CSV_DIR / "step10_mask_reconstruction_summary.csv"
manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
depth_df.to_csv(depth_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
mask_summary_df.to_csv(mask_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved Step 10 map tables:")
print(f"Manifest: {manifest_csv}")
print(f"Depth summary: {depth_csv}")
print(f"Mask summary: {mask_csv}")


# =========================
# Step 10. Combined top-network layer-depth map
# =========================

combined_depth_rows = []
combined_depth_nifti_path = ""
combined_depth_glass_path = ""
combined_depth_x_path = ""
combined_depth_y_path = ""
combined_depth_z_path = ""

if MAKE_COMBINED_TOP_NETWORK_DEPTH_MAP:
    # Select top networks by their average SRM+LLM performance across all layer-wise
    # models, not by one single selected layer. The map colors still come from each
    # voxel's preferred layer depth across layer_01...layer_48.
    top_network_rows = (
        group_df[group_df["layer_index"] > 0]
        .groupby("roi_name", as_index=False)
        .agg(
            mean_srm_group_r_across_layers=("srm_group_mean_r", "mean"),
            peak_srm_group_r_across_layers=("srm_group_mean_r", "max"),
            mean_delta_group_r_across_layers=("delta_group_mean_r_srm_minus_raw", "mean"),
        )
        .sort_values("mean_srm_group_r_across_layers", ascending=False)
        .head(TOP_N_COMBINED_DEPTH_NETWORKS)
        .reset_index(drop=True)
    )
    combined_ref_img = None
    combined_data = None

    for _, row in top_network_rows.iterrows():
        roi_name = row["roi_name"]
        depth_img_path = NII_DIR / roi_name / f"{roi_name}_voxelwise_preferred_srm_log_layer_depth_percent.nii.gz"
        if not depth_img_path.exists():
            raise FileNotFoundError(f"Missing depth map for combined top-network map: {depth_img_path}")
        img = nib.load(str(depth_img_path))
        data = np.asarray(img.get_fdata(), dtype=np.float32)
        if combined_ref_img is None:
            combined_ref_img = img
            combined_data = np.zeros(data.shape, dtype=np.float32)
        elif data.shape != combined_data.shape:
            raise ValueError(f"Combined depth map shape mismatch for {roi_name}: {data.shape} vs {combined_data.shape}")

        visible = np.isfinite(data) & (data > PSEUDO_ZERO_THRESHOLD)
        combined_data[visible] = data[visible]
        combined_depth_rows.append({
            "roi_name": roi_name,
            "selection_rule": "top networks by mean SRM+LLM group encoding r across transformer layers 1-48",
            "mean_srm_group_r_across_layers": float(row["mean_srm_group_r_across_layers"]),
            "peak_srm_group_r_across_layers": float(row["peak_srm_group_r_across_layers"]),
            "mean_delta_group_r_across_layers": float(row["mean_delta_group_r_across_layers"]),
            "included_visible_voxels": int(np.sum(visible)),
            "source_depth_nifti_path": str(depth_img_path),
        })

    if combined_ref_img is not None:
        combined_img = nib.Nifti1Image(combined_data.astype(np.float32), combined_ref_img.affine, combined_ref_img.header)
        combined_depth_nifti_path = NII_DIR / "combined_top_networks_voxelwise_preferred_srm_log_layer_depth_percent.nii.gz"
        nib.save(combined_img, combined_depth_nifti_path)

        combined_annotation = "Included regions selected by mean SRM+LLM r across all layer-wise models; colors show voxel-wise preferred layer depth across layers 1-48:\n" + "\n".join([
            f"{r['roi_name']}: mean r={r['mean_srm_group_r_across_layers']:.5f}, peak r={r['peak_srm_group_r_across_layers']:.5f}"
            for r in combined_depth_rows
        ])
        combined_depth_glass_path = FIG_COMBINED_DEPTH_DIR / "combined_top_networks_voxelwise_preferred_srm_log_layer_depth_glass.png"
        save_depth_glass_map(
            combined_depth_nifti_path,
            combined_depth_glass_path,
            "Top Networks: Voxel-wise Preferred Layer Depth",
            threshold=PSEUDO_ZERO_THRESHOLD,
            annotation_text=combined_annotation,
        )

        combined_depth_x_path = FIG_COMBINED_DEPTH_DIR / "combined_top_networks_voxelwise_preferred_srm_log_layer_depth_sagittal_x.png"
        combined_depth_y_path = FIG_COMBINED_DEPTH_DIR / "combined_top_networks_voxelwise_preferred_srm_log_layer_depth_coronal_y.png"
        combined_depth_z_path = FIG_COMBINED_DEPTH_DIR / "combined_top_networks_voxelwise_preferred_srm_log_layer_depth_axial_z.png"
        save_depth_directional_slice(combined_depth_nifti_path, combined_depth_x_path, "Top Networks: Preferred Layer Depth (sagittal)", "x")
        save_depth_directional_slice(combined_depth_nifti_path, combined_depth_y_path, "Top Networks: Preferred Layer Depth (coronal)", "y")
        save_depth_directional_slice(combined_depth_nifti_path, combined_depth_z_path, "Top Networks: Preferred Layer Depth (axial)", "z")

combined_depth_csv = CSV_DIR / "step10_combined_top_network_depth_map_regions.csv"
pd.DataFrame(combined_depth_rows).to_csv(combined_depth_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
print("\nCombined top-network depth map:")
print(combined_depth_nifti_path)
print(combined_depth_glass_path)
print(combined_depth_csv)


# =========================
# Step 10. Summary montages for strongest maps
# =========================

def render_montage_rows(selection_df, map_type, out_path, title, value_col):
    n = len(selection_df)
    if n == 0:
        return None
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 3.4 * n_rows))
    axes = np.asarray(axes).reshape(-1)

    for ax, (_, row) in zip(axes, selection_df.iterrows()):
        roi_name = row["roi_name"]
        layer_label = row["layer_label"]
        layer_index = int(row["layer_index"])
        depth_pct = layer_depth_percent(layer_index)
        roi_nii_dir = NII_DIR / roi_name
        if map_type == "srm":
            img_path = roi_nii_dir / f"{roi_name}_{layer_label}_srm_group_encoding_r.nii.gz"
            cmap = "viridis"
            vmin, vmax = 0, encoding_vmax
            threshold = encoding_threshold
        elif map_type == "delta":
            img_path = roi_nii_dir / f"{roi_name}_{layer_label}_delta_group_encoding_r_srm_minus_raw.nii.gz"
            cmap = "coolwarm"
            vmin, vmax = -delta_absmax, delta_absmax
            threshold = delta_threshold
        else:
            raise ValueError(f"Unknown montage map_type: {map_type}")

        display = plot_glass_brain(
            str(img_path),
            display_mode="z",
            colorbar=False,
            cmap=cmap,
            plot_abs=False,
            threshold=threshold,
            vmin=vmin,
            vmax=vmax,
            black_bg=False,
            axes=ax,
        )
        ax.set_title(
            f"{roi_name} {layer_label} | depth {depth_pct:.1f}% | {value_col}={row[value_col]:.5f}",
            fontsize=10,
        )

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title, y=1.01, fontsize=14, fontweight="bold")
    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=260, bbox_inches="tight")
    plt.close(fig)
    return out_path

montage_paths = []
if MAKE_TOP_SUMMARY_MONTAGES:
    top_srm_montage_path = render_montage_rows(
        top_srm_rows,
        map_type="srm",
        out_path=FIG_MONTAGE_DIR / "step10_top_srm_group_encoding_network_layer_maps.png",
        title=f"Top {TOP_N_SUMMARY_MAPS} Single-Layer SRM+LLM Group-Level Brain Maps",
        value_col="srm_group_mean_r",
    )
    top_delta_montage_path = render_montage_rows(
        top_delta_rows,
        map_type="delta",
        out_path=FIG_MONTAGE_DIR / "step10_top_delta_srm_minus_raw_network_layer_maps.png",
        title=f"Top {TOP_N_SUMMARY_MAPS} Single-Layer SRM-minus-Raw Group-Level Brain Maps",
        value_col="delta_group_mean_r_srm_minus_raw",
    )
    montage_paths.extend([str(p) for p in [top_srm_montage_path, top_delta_montage_path] if p is not None])

selection_srm_csv = CSV_DIR / "step10_top_srm_group_encoding_maps_selected_for_montage.csv"
selection_delta_csv = CSV_DIR / "step10_top_delta_maps_selected_for_montage.csv"
top_srm_rows.to_csv(selection_srm_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
top_delta_rows.to_csv(selection_delta_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nMontage selections saved:")
print(selection_srm_csv)
print(selection_delta_csv)
for p in montage_paths:
    print(f"Montage: {p}")


# =========================
# Step 10. Final validation and summary
# =========================

elapsed_min = (time.time() - start_total) / 60.0
summary = {
    "project_dir": str(PROJECT_DIR),
    "step10_dir": str(STEP10_DIR),
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layer_representations": int(len(LAYER_LABELS)),
    "n_manifest_rows": int(len(manifest_df)),
    "make_all_layer_nifti": bool(MAKE_ALL_LAYER_NIFTI),
    "make_all_layer_glass": bool(MAKE_ALL_LAYER_GLASS),
    "color_ranges": range_summary,
    "top_n_summary_maps": int(TOP_N_SUMMARY_MAPS),
    "total_elapsed_minutes": float(elapsed_min),
    "outputs": {
        "manifest_csv": str(manifest_csv),
        "depth_summary_csv": str(depth_csv),
        "mask_summary_csv": str(mask_csv),
        "range_csv": str(range_csv),
        "top_srm_selection_csv": str(selection_srm_csv),
        "top_delta_selection_csv": str(selection_delta_csv),
        "combined_top_network_depth_regions_csv": str(combined_depth_csv),
        "combined_top_network_depth_nifti": str(combined_depth_nifti_path),
        "combined_top_network_depth_glass": str(combined_depth_glass_path),
        "combined_top_network_depth_directional_figures": [str(combined_depth_x_path), str(combined_depth_y_path), str(combined_depth_z_path)],
        "montage_figures": montage_paths,
    },
}
summary_json = JSON_DIR / "step10_layerwise_brain_mapping_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nStep 10 validation:")
print(f"Manifest rows: {len(manifest_df)} expected {len(NETWORK_NAMES) * len(LAYER_LABELS)}")
print(f"Depth rows: {len(depth_df)} expected {len(NETWORK_NAMES)}")
print(f"Total elapsed: {elapsed_min:.2f} min")
print(f"Summary JSON: {summary_json}")

if len(manifest_df) != len(NETWORK_NAMES) * len(LAYER_LABELS):
    raise ValueError("Unexpected number of Step 10 manifest rows.")
if len(depth_df) != len(NETWORK_NAMES):
    raise ValueError("Unexpected number of layer-depth preference rows.")

print("\nStep 10 completed successfully.")
